# Preparación del dataset para modelado inmobiliario

Este notebook continúa el EDA. El dataset original se conserva sin modificaciones y todas las decisiones de preparación se aplican sobre una copia de trabajo.

## 1. Crear la copia de trabajo y eliminar columnas iniciales

Partimos de `secondary_market_date_transformado.csv`, que fue utilizado como fuente del EDA. Antes de eliminar `date_posted`, reconstruimos `periodo` con la misma regla usada durante el EDA para conservar la dimensión temporal. En esta primera etapa eliminamos únicamente las columnas seleccionadas: identificador, fecha original, precio total derivado del objetivo y variables que no continuaremos usando en esta versión del dataset.

La eliminación se realiza sobre `df_modelado`, una copia independiente. El archivo fuente no se sobrescribe.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RUTA_ORIGEN = Path("data_sets_transformados/secondary_market_date_transformado.csv")
RUTA_SALIDA = Path("data_sets_transformados/secondary_market_modelado_base.csv")

columnas_eliminar = [
    "id",
    "date_posted",
    "price_rub",
    "building_year",
    "okrug",
    "metro_line"
]

df_original = pd.read_csv(RUTA_ORIGEN, parse_dates=["date_posted"])
df_modelado = df_original.copy()
df_modelado["periodo"] = df_modelado["date_posted"].dt.to_period("M").astype(str)

columnas_faltantes = sorted(set(columnas_eliminar) - set(df_modelado.columns))
if columnas_faltantes:
    raise KeyError(f"No se encontraron las columnas esperadas: {columnas_faltantes}")

df_modelado = df_modelado.drop(columns=columnas_eliminar)
df_modelado.to_csv(RUTA_SALIDA, index=False)

print(f"Dataset original: {df_original.shape[0]:,} filas x {df_original.shape[1]} columnas")
print(f"Dataset de modelado: {df_modelado.shape[0]:,} filas x {df_modelado.shape[1]} columnas")
print(f"Columnas eliminadas: {columnas_eliminar}")
print(f"Copia guardada en: {RUTA_SALIDA}")
display(df_modelado.head())

Dataset original: 50,000 filas x 29 columnas
Dataset de modelado: 50,000 filas x 24 columnas
Columnas eliminadas: ['id', 'date_posted', 'price_rub', 'building_year', 'okrug', 'metro_line']
Copia guardada en: data_sets_transformados\secondary_market_modelado_base.csv


,district,lat,lon,total_area,living_area,kitchen_area,rooms,floor,total_floors,building_type,...,metro_distance_min,metro_distance_type,to_center_km,price_per_sqm,mortgage_rate_at_listing,seller_type,date_posted_year,date_posted_month,building_age,periodo
0,Begovoy,55.79533,37.55519,71.0,46.5,6.1,2,19,25,khrushchev,...,18,transport,6.17,221700,23.5,agency,2024,11,55,2024-11
1,Sokolniki,55.79695,37.68875,79.6,52.8,10.8,3,2,25,modern_panel,...,7,walk,6.40,247300,19.5,owner,2025,8,9,2025-08
2,Sosenskoye,55.54901,37.43118,74.0,42.2,8.9,2,6,9,monolith,...,102,transport,25.70,116400,12.0,owner,2022,5,6,2022-05
3,Sviblovo,55.86462,37.61471,52.3,34.0,6.0,1,5,22,monolith,...,9,walk,12.32,282400,23.5,agency,2025,4,13,2025-04
4,Begovoy,55.79429,37.54631,62.2,38.3,13.1,2,12,12,brick,...,10,walk,6.47,300300,15.5,agency,2023,7,21,2023-07


### Interpretación

La cantidad de filas debe mantenerse igual: esta etapa solo elimina columnas de la copia de trabajo. `price_per_sqm` continúa siendo la variable objetivo y `periodo` se conserva para decidir más adelante cómo realizar la evaluación temporal.

## 2. One-hot encoding sin colinealidad entre dummies

Convertimos las variables categóricas en indicadores numéricos. Usamos `drop_first=True` para conservar una categoría de referencia por variable y evitar la trampa de las variables ficticias cuando el modelo incluya intercepto.

`periodo` no se convierte a dummies: se conserva como clave temporal para ordenar y separar entrenamiento y prueba cronológicamente.

In [2]:
RUTA_ENTRADA_ONE_HOT = Path("data_sets_transformados/secondary_market_modelado_base.csv")
RUTA_SALIDA_ONE_HOT = Path("data_sets_transformados/secondary_market_modelado_one_hot.csv")

df_modelado = pd.read_csv(RUTA_ENTRADA_ONE_HOT)
columnas_categoricas = [
    columna
    for columna in df_modelado.select_dtypes(include=["object", "string", "category", "bool"]).columns
    if columna != "periodo"
]

categorias_por_variable = df_modelado[columnas_categoricas].nunique(dropna=False)
df_modelado_one_hot = pd.get_dummies(
    df_modelado,
    columns=columnas_categoricas,
    drop_first=True,
    dtype=int
)

# Verificamos que cada variable categórica haya generado una categoría menos.
dummies_por_variable = {
    columna: [nombre for nombre in df_modelado_one_hot.columns if nombre.startswith(f"{columna}_")]
    for columna in columnas_categoricas
}
validacion_dummies = pd.DataFrame({
    "variable": columnas_categoricas,
    "categorias_originales": [categorias_por_variable[columna] for columna in columnas_categoricas],
    "dummies_generadas": [len(dummies_por_variable[columna]) for columna in columnas_categoricas],
})
validacion_dummies["cumple_drop_first"] = (
    validacion_dummies["dummies_generadas"] == validacion_dummies["categorias_originales"] - 1
)

if not validacion_dummies["cumple_drop_first"].all():
    raise AssertionError("Alguna variable categórica no respetó drop_first=True")

df_modelado_one_hot.to_csv(RUTA_SALIDA_ONE_HOT, index=False)

print(f"Variables categóricas codificadas: {columnas_categoricas}")
print(f"Dataset antes del one-hot: {df_modelado.shape[0]:,} filas x {df_modelado.shape[1]} columnas")
print(f"Dataset después del one-hot: {df_modelado_one_hot.shape[0]:,} filas x {df_modelado_one_hot.shape[1]} columnas")
print("Periodo conservado sin one-hot:", "periodo" in df_modelado_one_hot.columns)
print(f"Archivo guardado en: {RUTA_SALIDA_ONE_HOT}")
display(validacion_dummies)
display(df_modelado_one_hot.head())

Variables categóricas codificadas: ['district', 'building_type', 'has_balcony', 'renovation', 'metro_station', 'metro_distance_type', 'seller_type']
Dataset antes del one-hot: 50,000 filas x 24 columnas
Dataset después del one-hot: 50,000 filas x 259 columnas
Periodo conservado sin one-hot: True
Archivo guardado en: data_sets_transformados\secondary_market_modelado_one_hot.csv


,variable,categorias_originales,dummies_generadas,cumple_drop_first
0,district,129,128,True
1,building_type,6,5,True
2,has_balcony,2,1,True
3,renovation,4,3,True
4,metro_station,104,103,True
5,metro_distance_type,2,1,True
6,seller_type,2,1,True


,lat,lon,total_area,living_area,kitchen_area,rooms,floor,total_floors,ceiling_height,metro_distance_min,...,metro_station_Ulitsa Podbelskogo,metro_station_Universitet,metro_station_VDNKh,metro_station_Voykovskaya,metro_station_Vystavochnaya,metro_station_Yasenevo,metro_station_Yugo-Zapadnaya,metro_station_Yuzhnaya,metro_distance_type_walk,seller_type_owner
0,55.79533,37.55519,71.0,46.5,6.1,2,19,25,2.5,18,...,0,0,0,0,0,0,0,0,0,0
1,55.79695,37.68875,79.6,52.8,10.8,3,2,25,2.5,7,...,0,0,0,0,0,0,0,0,1,1
2,55.54901,37.43118,74.0,42.2,8.9,2,6,9,3.0,102,...,0,0,0,0,0,0,0,0,0,1
3,55.86462,37.61471,52.3,34.0,6.0,1,5,22,2.5,9,...,0,0,0,0,0,0,0,0,1,0
4,55.79429,37.54631,62.2,38.3,13.1,2,12,12,2.5,10,...,0,0,0,0,0,0,0,0,1,0


### Interpretación

Cada variable categórica queda representada por una categoría menos que las originales. La categoría omitida funciona como referencia y evita dependencia lineal perfecta entre las dummies y el intercepto. Esto evita la colinealidad propia del one-hot completo; no garantiza que no existan otras relaciones fuertes entre variables numéricas o derivadas, que deberán revisarse en una etapa posterior.

## 3. Regresión lineal OLS y registro en MLflow

Ajustamos una regresión lineal con `statsmodels.OLS` sobre el dataset one-hot. La separación es temporal: se entrena con períodos anteriores a `2025-01` y se evalúa desde `2025-01` en adelante. `periodo` se usa para separar los conjuntos, pero no entra como predictor.

La corrida registra métricas de entrenamiento y prueba, criterios del ajuste OLS, diagnósticos de residuos, coeficientes, residuos de prueba y el resumen completo del modelo.

In [3]:
import mlflow
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.stattools import durbin_watson, jarque_bera, omni_normtest

RUTA_OLS = Path("data_sets_transformados/secondary_market_modelado_one_hot.csv")
CORTE_TEMPORAL = "2025-01"
df_ols = pd.read_csv(RUTA_OLS)
df_ols = df_ols.sort_values("periodo").reset_index(drop=True)

df_train_ols = df_ols[df_ols["periodo"] < CORTE_TEMPORAL].copy()
df_test_ols = df_ols[df_ols["periodo"] >= CORTE_TEMPORAL].copy()
if df_train_ols.empty or df_test_ols.empty:
    raise ValueError("El corte temporal no produjo conjuntos de entrenamiento y prueba válidos")

columnas_excluir_ols = ["price_per_sqm", "periodo"]
X_train_ols = df_train_ols.drop(columns=columnas_excluir_ols)
X_test_ols = df_test_ols.drop(columns=columnas_excluir_ols)
y_train_ols = df_train_ols["price_per_sqm"]
y_test_ols = df_test_ols["price_per_sqm"]

X_train_ols_sm = sm.add_constant(X_train_ols, has_constant="add")
X_test_ols_sm = sm.add_constant(X_test_ols, has_constant="add")
modelo_ols_temporal = sm.OLS(y_train_ols, X_train_ols_sm)
resultados_ols_temporal = modelo_ols_temporal.fit()
predicciones_train_ols = resultados_ols_temporal.predict(X_train_ols_sm)
predicciones_test_ols = resultados_ols_temporal.predict(X_test_ols_sm)

def calcular_metricas_regresion(y_real, predicciones, prefijo):
    errores_porcentuales = (y_real - predicciones) / y_real
    return {
        f"mae_{prefijo}": float(mean_absolute_error(y_real, predicciones)),
        f"rmse_{prefijo}": float(mean_squared_error(y_real, predicciones) ** 0.5),
        f"r2_{prefijo}": float(r2_score(y_real, predicciones)),
        f"mape_{prefijo}": float(errores_porcentuales.abs().mean() * 100)
    }

metricas_ols_temporal = {
    **calcular_metricas_regresion(y_train_ols, predicciones_train_ols, "train"),
    **calcular_metricas_regresion(y_test_ols, predicciones_test_ols, "test"),
    "r2_ols": float(resultados_ols_temporal.rsquared),
    "r2_ajustado_ols": float(resultados_ols_temporal.rsquared_adj),
    "aic_ols": float(resultados_ols_temporal.aic),
    "bic_ols": float(resultados_ols_temporal.bic),
    "f_statistic_ols": float(resultados_ols_temporal.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_temporal.f_pvalue),
}

omnibus_ols, prob_omnibus_ols = omni_normtest(resultados_ols_temporal.resid)
jarque_bera_ols, prob_jb_ols, skew_ols, kurtosis_ols = jarque_bera(resultados_ols_temporal.resid)
metricas_ols_temporal.update({
    "omnibus_ols": float(omnibus_ols),
    "prob_omnibus_ols": float(prob_omnibus_ols),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_temporal.resid)),
    "jarque_bera_ols": float(jarque_bera_ols),
    "prob_jb_ols": float(prob_jb_ols),
    "skew_ols": float(skew_ols),
    "kurtosis_ols": float(kurtosis_ols),
    "condition_number_ols": float(resultados_ols_temporal.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_temporal.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_temporal.pvalues > 0.05).sum())
})

coeficientes_ols_temporal = pd.DataFrame({
    "variable": resultados_ols_temporal.params.index,
    "coeficiente": resultados_ols_temporal.params.to_numpy(),
    "p_value": resultados_ols_temporal.pvalues.to_numpy(),
    "significativa_0_05": resultados_ols_temporal.pvalues.to_numpy() <= 0.05
})
residuos_test_ols = pd.DataFrame({
    "periodo": df_test_ols["periodo"].to_numpy(),
    "real": y_test_ols.to_numpy(),
    "prediccion": predicciones_test_ols.to_numpy(),
    "residuo": (y_test_ols.to_numpy() - predicciones_test_ols.to_numpy())
})

tabla_metricas_ols_temporal = pd.DataFrame({"metrica": list(metricas_ols_temporal), "valor": list(metricas_ols_temporal.values())})
display(tabla_metricas_ols_temporal)
print(resultados_ols_temporal.summary())

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_modelado_one_hot"):
    mlflow.log_params({
        "dataset": str(RUTA_OLS),
        "objetivo": "price_per_sqm",
        "corte_temporal": CORTE_TEMPORAL,
        "periodo_entrenamiento_desde": str(df_train_ols["periodo"].min()),
        "periodo_entrenamiento_hasta": str(df_train_ols["periodo"].max()),
        "periodo_prueba_desde": str(df_test_ols["periodo"].min()),
        "periodo_prueba_hasta": str(df_test_ols["periodo"].max()),
        "filas_entrenamiento": len(df_train_ols),
        "filas_prueba": len(df_test_ols),
        "n_predictores": X_train_ols.shape[1],
        "tipo_modelo": "statsmodels_OLS"
    })
    mlflow.log_metrics(metricas_ols_temporal)
    mlflow.log_table(tabla_metricas_ols_temporal, artifact_file="metricas/metricas_ols_temporal.json")
    mlflow.log_table(coeficientes_ols_temporal, artifact_file="modelo/coeficientes_ols_temporal.json")
    mlflow.log_table(residuos_test_ols, artifact_file="diagnostico/residuos_test_ols.json")
    mlflow.log_text(resultados_ols_temporal.summary().as_text(), "modelo/resumen_ols_temporal.txt")

print("OLS temporal registrado en MLflow: experimento 'Inmobiliario - OLS Temporal'")

2026/10/12 23:07:46 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


,metrica,valor
0,mae_train,2.314073e+04
1,rmse_train,3.353420e+04
2,r2_train,9.082387e-01
3,mape_train,1.491517e+01
4,mae_test,3.708559e+04
5,rmse_test,4.513263e+04
6,r2_test,8.598280e-01
7,mape_test,2.317410e+01
8,r2_ols,9.082387e-01
9,r2_ajustado_ols,9.076402e-01


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1517.
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        23:07:48   Log-Likelihood:            -4.6952e+05
No. Observations:               39658   AIC:                         9.396e+05
Df Residuals:                   39400   BIC:                         9.418e+05
Df Model:                         257                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

🏃 View run ols_temporal_modelado_one_hot at: http://127.0.0.1:5000/#/experiments/4/runs/c46c10ae37fc435cbe692c40eb3dad5c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
OLS temporal registrado en MLflow: experimento 'Inmobiliario - OLS Temporal'


### Interpretación

Las métricas de prueba son las que deben utilizarse para valorar la capacidad predictiva fuera de muestra. El R², AIC, BIC y los diagnósticos describen el ajuste OLS; no convierten por sí solos al modelo en definitivo. La corrida de MLflow permite comparar este resultado temporal con futuras versiones del dataset y del modelo.

## 4. Filtrado de predictores con p-value >= 0,9

Usamos los p-values del OLS temporal anterior para identificar predictores sin evidencia estadística suficiente bajo este ajuste. Se eliminan únicamente columnas predictoras con `p-value >= 0,9`; nunca se elimina la constante, el objetivo ni `periodo`.

El archivo one-hot original se conserva. Esta sección crea una copia filtrada, vuelve a ajustar OLS con el mismo corte temporal y registra una nueva corrida para comparar sus métricas con la anterior.

In [4]:
UMBRAL_PVALUE = 0.90
RUTA_SALIDA_PVALUE = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_090.csv")

variables_eliminar_pvalue = sorted(
    coeficientes_ols_temporal.loc[
        (coeficientes_ols_temporal["p_value"] >= UMBRAL_PVALUE)
        & (~coeficientes_ols_temporal["variable"].isin(["const", "price_per_sqm", "periodo"])),
        "variable"
    ].tolist()
)

df_ols_filtrado = df_ols.drop(columns=variables_eliminar_pvalue)
df_ols_filtrado.to_csv(RUTA_SALIDA_PVALUE, index=False)

df_train_filtrado = df_ols_filtrado[df_ols_filtrado["periodo"] < CORTE_TEMPORAL].copy()
df_test_filtrado = df_ols_filtrado[df_ols_filtrado["periodo"] >= CORTE_TEMPORAL].copy()
X_train_filtrado = df_train_filtrado.drop(columns=["price_per_sqm", "periodo"])
X_test_filtrado = df_test_filtrado.drop(columns=["price_per_sqm", "periodo"])
y_train_filtrado = df_train_filtrado["price_per_sqm"]
y_test_filtrado = df_test_filtrado["price_per_sqm"]

X_train_filtrado_sm = sm.add_constant(X_train_filtrado, has_constant="add")
X_test_filtrado_sm = sm.add_constant(X_test_filtrado, has_constant="add")
modelo_ols_filtrado = sm.OLS(y_train_filtrado, X_train_filtrado_sm)
resultados_ols_filtrado = modelo_ols_filtrado.fit()
predicciones_train_filtrado = resultados_ols_filtrado.predict(X_train_filtrado_sm)
predicciones_test_filtrado = resultados_ols_filtrado.predict(X_test_filtrado_sm)

metricas_ols_filtrado = {
    **calcular_metricas_regresion(y_train_filtrado, predicciones_train_filtrado, "train"),
    **calcular_metricas_regresion(y_test_filtrado, predicciones_test_filtrado, "test"),
    "r2_ols": float(resultados_ols_filtrado.rsquared),
    "r2_ajustado_ols": float(resultados_ols_filtrado.rsquared_adj),
    "aic_ols": float(resultados_ols_filtrado.aic),
    "bic_ols": float(resultados_ols_filtrado.bic),
    "f_statistic_ols": float(resultados_ols_filtrado.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_filtrado.f_pvalue)
}
omnibus_filtrado, prob_omnibus_filtrado = omni_normtest(resultados_ols_filtrado.resid)
jarque_bera_filtrado, prob_jb_filtrado, skew_filtrado, kurtosis_filtrado = jarque_bera(resultados_ols_filtrado.resid)
metricas_ols_filtrado.update({
    "omnibus_ols": float(omnibus_filtrado),
    "prob_omnibus_ols": float(prob_omnibus_filtrado),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_filtrado.resid)),
    "jarque_bera_ols": float(jarque_bera_filtrado),
    "prob_jb_ols": float(prob_jb_filtrado),
    "skew_ols": float(skew_filtrado),
    "kurtosis_ols": float(kurtosis_filtrado),
    "condition_number_ols": float(resultados_ols_filtrado.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_filtrado.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_filtrado.pvalues > 0.05).sum())
})

coeficientes_ols_filtrado = pd.DataFrame({
    "variable": resultados_ols_filtrado.params.index,
    "coeficiente": resultados_ols_filtrado.params.to_numpy(),
    "p_value": resultados_ols_filtrado.pvalues.to_numpy(),
    "significativa_0_05": resultados_ols_filtrado.pvalues.to_numpy() <= 0.05
})
residuos_test_filtrado = pd.DataFrame({
    "periodo": df_test_filtrado["periodo"].to_numpy(),
    "real": y_test_filtrado.to_numpy(),
    "prediccion": predicciones_test_filtrado.to_numpy(),
    "residuo": (y_test_filtrado.to_numpy() - predicciones_test_filtrado.to_numpy())
})

comparacion_ols = pd.DataFrame([
    {"modelo": "OLS anterior", **metricas_ols_temporal},
    {"modelo": "OLS p-value filtrado", **metricas_ols_filtrado}
])
delta_metricas = {
    f"delta_{metrica}": metricas_ols_filtrado[metrica] - metricas_ols_temporal[metrica]
    for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]
}
tabla_variables_eliminadas = coeficientes_ols_temporal[
    coeficientes_ols_temporal["variable"].isin(variables_eliminar_pvalue)
].sort_values("p_value", ascending=False)

display(tabla_variables_eliminadas)
display(comparacion_ols[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
print(resultados_ols_filtrado.summary())

RUN_OLS_ANTERIOR_ID = "8632cd73b7b84347bd6cd3ee0b2cfe4e"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_pvalue_filtrado_090"):
    mlflow.set_tag("run_anterior_id", RUN_OLS_ANTERIOR_ID)
    mlflow.log_params({
        "dataset": str(RUTA_SALIDA_PVALUE),
        "dataset_origen": str(RUTA_OLS),
        "objetivo": "price_per_sqm",
        "corte_temporal": CORTE_TEMPORAL,
        "umbral_pvalue_eliminacion": UMBRAL_PVALUE,
        "n_variables_eliminadas_pvalue": len(variables_eliminar_pvalue),
        "n_predictores": X_train_filtrado.shape[1],
        "filas_entrenamiento": len(df_train_filtrado),
        "filas_prueba": len(df_test_filtrado),
        "tipo_modelo": "statsmodels_OLS_pvalue_filtrado"
    })
    mlflow.log_metrics({**metricas_ols_filtrado, **delta_metricas})
    mlflow.log_table(tabla_variables_eliminadas, artifact_file="seleccion/variables_eliminadas_pvalue_090.json")
    mlflow.log_table(comparacion_ols, artifact_file="comparacion/comparacion_ols.json")
    mlflow.log_table(coeficientes_ols_filtrado, artifact_file="modelo/coeficientes_ols_filtrado.json")
    mlflow.log_table(residuos_test_filtrado, artifact_file="diagnostico/residuos_test_filtrado.json")
    mlflow.log_text(resultados_ols_filtrado.summary().as_text(), "modelo/resumen_ols_filtrado.txt")

print(f"Variables eliminadas por p-value >= {UMBRAL_PVALUE}: {len(variables_eliminar_pvalue)}")
print(f"Archivo filtrado guardado en: {RUTA_SALIDA_PVALUE}")
print("OLS filtrado registrado en MLflow para comparación")

,variable,coeficiente,p_value,significativa_0_05
83,district_Novofyodorovskoye,-213.007177,0.984772,False
69,district_Maryina Roshcha,154.746066,0.958605,False
194,metro_station_Park Pobedy,-155.820254,0.953916,False
245,metro_station_Ugreshskaya,160.066982,0.953830,False
34,district_Danilovsky,281.294278,0.932565,False
238,metro_station_Troparyovo,218.831280,0.931241,False
205,metro_station_Profsoyuznaya,-226.715496,0.926320,False
239,metro_station_Trubnaya,-248.214532,0.925719,False
246,metro_station_Ulitsa 1905 Goda,307.355430,0.908387,False


,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS anterior,37085.586046,45132.632765,0.859828,23.174104
1,OLS p-value filtrado,37085.141086,45132.235880,0.859830,23.174184


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1573.
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        23:07:50   Log-Likelihood:            -4.6952e+05
No. Observations:               39658   AIC:                         9.395e+05
Df Residuals:                   39409   BIC:                         9.417e+05
Df Model:                         248                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

🏃 View run ols_temporal_pvalue_filtrado_090 at: http://127.0.0.1:5000/#/experiments/4/runs/04b678f6c4474626be4a11c497e37497
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Variables eliminadas por p-value >= 0.9: 9
Archivo filtrado guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_090.csv
OLS filtrado registrado en MLflow para comparación


### Interpretación

El nuevo modelo debe compararse principalmente por MAE, RMSE, R² y MAPE en el conjunto temporal de prueba. El filtrado por p-value reduce la cantidad de predictores, pero solo conviene conservarlo si mantiene o mejora el desempeño fuera de muestra y no deteriora la estabilidad del modelo.

## 5. Filtrado progresivo de predictores con p-value >= 0,7

Partimos del dataset y del modelo que ya fueron filtrados con `p-value >= 0,9`. Sobre ese resultado eliminamos los predictores restantes cuyo p-value sea mayor o igual a `0,7`, conservando siempre la constante, `price_per_sqm` y `periodo`.

El nuevo modelo utiliza el mismo corte temporal y se registra en el mismo experimento MLflow para compararlo con la versión de `p-value >= 0,9`.

In [5]:
UMBRAL_PVALUE_070 = 0.70
RUTA_SALIDA_PVALUE_070 = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_070.csv")

variables_eliminar_pvalue_070 = sorted(
    coeficientes_ols_filtrado.loc[
        (coeficientes_ols_filtrado["p_value"] >= UMBRAL_PVALUE_070)
        & (~coeficientes_ols_filtrado["variable"].isin(["const", "price_per_sqm", "periodo"])),
        "variable"
    ].tolist()
)

df_ols_filtrado_070 = df_ols_filtrado.drop(columns=variables_eliminar_pvalue_070)
df_ols_filtrado_070.to_csv(RUTA_SALIDA_PVALUE_070, index=False)

df_train_070 = df_ols_filtrado_070[df_ols_filtrado_070["periodo"] < CORTE_TEMPORAL].copy()
df_test_070 = df_ols_filtrado_070[df_ols_filtrado_070["periodo"] >= CORTE_TEMPORAL].copy()
X_train_070 = df_train_070.drop(columns=["price_per_sqm", "periodo"])
X_test_070 = df_test_070.drop(columns=["price_per_sqm", "periodo"])
y_train_070 = df_train_070["price_per_sqm"]
y_test_070 = df_test_070["price_per_sqm"]

X_train_070_sm = sm.add_constant(X_train_070, has_constant="add")
X_test_070_sm = sm.add_constant(X_test_070, has_constant="add")
modelo_ols_070 = sm.OLS(y_train_070, X_train_070_sm)
resultados_ols_070 = modelo_ols_070.fit()
predicciones_train_070 = resultados_ols_070.predict(X_train_070_sm)
predicciones_test_070 = resultados_ols_070.predict(X_test_070_sm)

metricas_ols_070 = {
    **calcular_metricas_regresion(y_train_070, predicciones_train_070, "train"),
    **calcular_metricas_regresion(y_test_070, predicciones_test_070, "test"),
    "r2_ols": float(resultados_ols_070.rsquared),
    "r2_ajustado_ols": float(resultados_ols_070.rsquared_adj),
    "aic_ols": float(resultados_ols_070.aic),
    "bic_ols": float(resultados_ols_070.bic),
    "f_statistic_ols": float(resultados_ols_070.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_070.f_pvalue)
}
omnibus_070, prob_omnibus_070 = omni_normtest(resultados_ols_070.resid)
jarque_bera_070, prob_jb_070, skew_070, kurtosis_070 = jarque_bera(resultados_ols_070.resid)
metricas_ols_070.update({
    "omnibus_ols": float(omnibus_070),
    "prob_omnibus_ols": float(prob_omnibus_070),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_070.resid)),
    "jarque_bera_ols": float(jarque_bera_070),
    "prob_jb_ols": float(prob_jb_070),
    "skew_ols": float(skew_070),
    "kurtosis_ols": float(kurtosis_070),
    "condition_number_ols": float(resultados_ols_070.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_070.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_070.pvalues > 0.05).sum())
})

coeficientes_ols_070 = pd.DataFrame({
    "variable": resultados_ols_070.params.index,
    "coeficiente": resultados_ols_070.params.to_numpy(),
    "p_value": resultados_ols_070.pvalues.to_numpy(),
    "significativa_0_05": resultados_ols_070.pvalues.to_numpy() <= 0.05
})
residuos_test_070 = pd.DataFrame({
    "periodo": df_test_070["periodo"].to_numpy(),
    "real": y_test_070.to_numpy(),
    "prediccion": predicciones_test_070.to_numpy(),
    "residuo": (y_test_070.to_numpy() - predicciones_test_070.to_numpy())
})

comparacion_ols_070 = pd.DataFrame([
    {"modelo": "OLS p-value 0.90", **metricas_ols_filtrado},
    {"modelo": "OLS p-value 0.70", **metricas_ols_070}
])
delta_metricas_070 = {
    f"delta_{metrica}": metricas_ols_070[metrica] - metricas_ols_filtrado[metrica]
    for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]
}
tabla_variables_eliminadas_070 = coeficientes_ols_filtrado[
    coeficientes_ols_filtrado["variable"].isin(variables_eliminar_pvalue_070)
].sort_values("p_value", ascending=False)

display(tabla_variables_eliminadas_070)
display(comparacion_ols_070[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
print(resultados_ols_070.summary())

RUN_OLS_P90_ID = "45a11b38934a495eb7a148c8fa9d3f92"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_pvalue_filtrado_070"):
    mlflow.set_tag("run_anterior_id", RUN_OLS_P90_ID)
    mlflow.log_params({
        "dataset": str(RUTA_SALIDA_PVALUE_070),
        "dataset_origen": str(RUTA_SALIDA_PVALUE),
        "objetivo": "price_per_sqm",
        "corte_temporal": CORTE_TEMPORAL,
        "umbral_pvalue_eliminacion": UMBRAL_PVALUE_070,
        "n_variables_eliminadas_pvalue": len(variables_eliminar_pvalue_070),
        "n_predictores": X_train_070.shape[1],
        "filas_entrenamiento": len(df_train_070),
        "filas_prueba": len(df_test_070),
        "tipo_modelo": "statsmodels_OLS_pvalue_filtrado_070"
    })
    mlflow.log_metrics({**metricas_ols_070, **delta_metricas_070})
    mlflow.log_table(tabla_variables_eliminadas_070, artifact_file="seleccion/variables_eliminadas_pvalue_070.json")
    mlflow.log_table(comparacion_ols_070, artifact_file="comparacion/comparacion_ols_070.json")
    mlflow.log_table(coeficientes_ols_070, artifact_file="modelo/coeficientes_ols_070.json")
    mlflow.log_table(residuos_test_070, artifact_file="diagnostico/residuos_test_070.json")
    mlflow.log_text(resultados_ols_070.summary().as_text(), "modelo/resumen_ols_070.txt")

print(f"Variables eliminadas por p-value >= {UMBRAL_PVALUE_070}: {len(variables_eliminar_pvalue_070)}")
print(f"Archivo filtrado guardado en: {RUTA_SALIDA_PVALUE_070}")
print("OLS filtrado 0.70 registrado en MLflow para comparación")

,variable,coeficiente,p_value,significativa_0_05
3,total_area,7.848934,0.848591,False
4,living_area,10.980802,0.846329,False
84,district_Ochakovo-Matveyevskoye,685.915188,0.763805,False
154,metro_station_Borovitskaya,640.929623,0.760793,False
91,district_Pokrovskoye-Streshnevo,-833.377776,0.744008,False
171,metro_station_Kozhukhovskaya,-720.057679,0.735430,False
223,metro_station_Sretensky Bulvar,736.784795,0.727455,False


,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS p-value 0.90,37085.141086,45132.235880,0.859830,23.174184
1,OLS p-value 0.70,37085.870554,45133.296237,0.859824,23.174562


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1619.
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        23:07:52   Log-Likelihood:            -4.6952e+05
No. Observations:               39658   AIC:                         9.395e+05
Df Residuals:                   39416   BIC:                         9.416e+05
Df Model:                         241                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

🏃 View run ols_temporal_pvalue_filtrado_070 at: http://127.0.0.1:5000/#/experiments/4/runs/f4917697d2c142c8a77cea946dd5e7e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Variables eliminadas por p-value >= 0.7: 7
Archivo filtrado guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_070.csv
OLS filtrado 0.70 registrado en MLflow para comparación


### Interpretación

Este segundo filtrado es más agresivo que el de `p-value >= 0,9`. La decisión debe basarse en la comparación temporal de MAE, RMSE, R² y MAPE, no únicamente en reducir la cantidad de columnas.

## 6. Eliminación explícita de `rooms`

El modelo con umbral `p-value >= 0,7` mostró que `rooms` quedó con un p-value de aproximadamente `0,823` después del re-ajuste. En esta sección eliminamos explícitamente esa variable, volvemos a ejecutar el mismo OLS temporal y registramos la nueva versión para compararla con el modelo anterior.

In [6]:
VARIABLE_ELIMINADA = "rooms"
RUTA_SALIDA_SIN_ROOMS = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_070_sin_rooms.csv")

if VARIABLE_ELIMINADA not in df_ols_filtrado_070.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA!r} no está disponible para eliminar")

df_ols_sin_rooms = df_ols_filtrado_070.drop(columns=[VARIABLE_ELIMINADA])
df_ols_sin_rooms.to_csv(RUTA_SALIDA_SIN_ROOMS, index=False)

df_train_sin_rooms = df_ols_sin_rooms[df_ols_sin_rooms["periodo"] < CORTE_TEMPORAL].copy()
df_test_sin_rooms = df_ols_sin_rooms[df_ols_sin_rooms["periodo"] >= CORTE_TEMPORAL].copy()
X_train_sin_rooms = df_train_sin_rooms.drop(columns=["price_per_sqm", "periodo"])
X_test_sin_rooms = df_test_sin_rooms.drop(columns=["price_per_sqm", "periodo"])
y_train_sin_rooms = df_train_sin_rooms["price_per_sqm"]
y_test_sin_rooms = df_test_sin_rooms["price_per_sqm"]

X_train_sin_rooms_sm = sm.add_constant(X_train_sin_rooms, has_constant="add")
X_test_sin_rooms_sm = sm.add_constant(X_test_sin_rooms, has_constant="add")
modelo_ols_sin_rooms = sm.OLS(y_train_sin_rooms, X_train_sin_rooms_sm)
resultados_ols_sin_rooms = modelo_ols_sin_rooms.fit()
predicciones_train_sin_rooms = resultados_ols_sin_rooms.predict(X_train_sin_rooms_sm)
predicciones_test_sin_rooms = resultados_ols_sin_rooms.predict(X_test_sin_rooms_sm)

metricas_ols_sin_rooms = {
    **calcular_metricas_regresion(y_train_sin_rooms, predicciones_train_sin_rooms, "train"),
    **calcular_metricas_regresion(y_test_sin_rooms, predicciones_test_sin_rooms, "test"),
    "r2_ols": float(resultados_ols_sin_rooms.rsquared),
    "r2_ajustado_ols": float(resultados_ols_sin_rooms.rsquared_adj),
    "aic_ols": float(resultados_ols_sin_rooms.aic),
    "bic_ols": float(resultados_ols_sin_rooms.bic),
    "f_statistic_ols": float(resultados_ols_sin_rooms.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_sin_rooms.f_pvalue)
}
omnibus_sin_rooms, prob_omnibus_sin_rooms = omni_normtest(resultados_ols_sin_rooms.resid)
jarque_bera_sin_rooms, prob_jb_sin_rooms, skew_sin_rooms, kurtosis_sin_rooms = jarque_bera(resultados_ols_sin_rooms.resid)
metricas_ols_sin_rooms.update({
    "omnibus_ols": float(omnibus_sin_rooms),
    "prob_omnibus_ols": float(prob_omnibus_sin_rooms),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_sin_rooms.resid)),
    "jarque_bera_ols": float(jarque_bera_sin_rooms),
    "prob_jb_ols": float(prob_jb_sin_rooms),
    "skew_ols": float(skew_sin_rooms),
    "kurtosis_ols": float(kurtosis_sin_rooms),
    "condition_number_ols": float(resultados_ols_sin_rooms.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_sin_rooms.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_sin_rooms.pvalues > 0.05).sum())
})

coeficientes_ols_sin_rooms = pd.DataFrame({
    "variable": resultados_ols_sin_rooms.params.index,
    "coeficiente": resultados_ols_sin_rooms.params.to_numpy(),
    "p_value": resultados_ols_sin_rooms.pvalues.to_numpy(),
    "significativa_0_05": resultados_ols_sin_rooms.pvalues.to_numpy() <= 0.05
})
residuos_test_sin_rooms = pd.DataFrame({
    "periodo": df_test_sin_rooms["periodo"].to_numpy(),
    "real": y_test_sin_rooms.to_numpy(),
    "prediccion": predicciones_test_sin_rooms.to_numpy(),
    "residuo": (y_test_sin_rooms.to_numpy() - predicciones_test_sin_rooms.to_numpy())
})
comparacion_ols_sin_rooms = pd.DataFrame([
    {"modelo": "OLS p-value 0.70", **metricas_ols_070},
    {"modelo": "OLS sin rooms", **metricas_ols_sin_rooms}
])
delta_metricas_sin_rooms = {
    f"delta_{metrica}": metricas_ols_sin_rooms[metrica] - metricas_ols_070[metrica]
    for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]
}

display(comparacion_ols_sin_rooms[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
print(resultados_ols_sin_rooms.summary())

RUN_OLS_P70_ID = "bc655f410d474a07b8ed1f153c4d084e"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_sin_rooms"):
    mlflow.set_tag("run_anterior_id", RUN_OLS_P70_ID)
    mlflow.log_params({
        "dataset": str(RUTA_SALIDA_SIN_ROOMS),
        "dataset_origen": str(RUTA_SALIDA_PVALUE_070),
        "objetivo": "price_per_sqm",
        "corte_temporal": CORTE_TEMPORAL,
        "variable_eliminada": VARIABLE_ELIMINADA,
        "n_predictores": X_train_sin_rooms.shape[1],
        "filas_entrenamiento": len(df_train_sin_rooms),
        "filas_prueba": len(df_test_sin_rooms),
        "tipo_modelo": "statsmodels_OLS_sin_rooms"
    })
    mlflow.log_metrics({**metricas_ols_sin_rooms, **delta_metricas_sin_rooms})
    mlflow.log_table(comparacion_ols_sin_rooms, artifact_file="comparacion/comparacion_ols_sin_rooms.json")
    mlflow.log_table(coeficientes_ols_sin_rooms, artifact_file="modelo/coeficientes_ols_sin_rooms.json")
    mlflow.log_table(residuos_test_sin_rooms, artifact_file="diagnostico/residuos_test_sin_rooms.json")
    mlflow.log_text(resultados_ols_sin_rooms.summary().as_text(), "modelo/resumen_ols_sin_rooms.txt")

print(f"Archivo sin rooms guardado en: {RUTA_SALIDA_SIN_ROOMS}")
print("OLS sin rooms registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS p-value 0.70,37085.870554,45133.296237,0.859824,23.174562
1,OLS sin rooms,37086.334875,45133.843386,0.859820,23.174916


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1626.
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        23:07:54   Log-Likelihood:            -4.6952e+05
No. Observations:               39658   AIC:                         9.395e+05
Df Residuals:                   39417   BIC:                         9.416e+05
Df Model:                         240                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

🏃 View run ols_temporal_sin_rooms at: http://127.0.0.1:5000/#/experiments/4/runs/364ebea47ba04427a404907bad78752e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Archivo sin rooms guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_070_sin_rooms.csv
OLS sin rooms registrado en MLflow para comparación


### Interpretación

La eliminación de `rooms` se evalúa comparando las métricas temporales de prueba contra el modelo `p-value >= 0,7`. La variable deja de participar en el modelo, pero el resto de los coeficientes y sus p-values pueden cambiar al volver a ajustar el OLS.

## 7. Filtrado de predictores con p-value >= 0,2 sobre el modelo sin `rooms`

Partimos del modelo que ya elimina `rooms` y quitamos los predictores restantes con `p-value >= 0,2`. Se conserva el mismo corte temporal y el archivo anterior permanece intacto para poder comparar las variantes.

In [7]:
UMBRAL_PVALUE_020 = 0.20
RUTA_SALIDA_PVALUE_020 = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_020_sin_rooms.csv")

variables_eliminar_pvalue_020 = sorted(
    coeficientes_ols_sin_rooms.loc[
        (coeficientes_ols_sin_rooms["p_value"] >= UMBRAL_PVALUE_020)
        & (~coeficientes_ols_sin_rooms["variable"].isin(["const", "price_per_sqm", "periodo"])),
        "variable"
    ].tolist()
)

df_ols_filtrado_020 = df_ols_sin_rooms.drop(columns=variables_eliminar_pvalue_020)
df_ols_filtrado_020.to_csv(RUTA_SALIDA_PVALUE_020, index=False)

df_train_020 = df_ols_filtrado_020[df_ols_filtrado_020["periodo"] < CORTE_TEMPORAL].copy()
df_test_020 = df_ols_filtrado_020[df_ols_filtrado_020["periodo"] >= CORTE_TEMPORAL].copy()
X_train_020 = df_train_020.drop(columns=["price_per_sqm", "periodo"])
X_test_020 = df_test_020.drop(columns=["price_per_sqm", "periodo"])
y_train_020 = df_train_020["price_per_sqm"]
y_test_020 = df_test_020["price_per_sqm"]

X_train_020_sm = sm.add_constant(X_train_020, has_constant="add")
X_test_020_sm = sm.add_constant(X_test_020, has_constant="add")
modelo_ols_020 = sm.OLS(y_train_020, X_train_020_sm)
resultados_ols_020 = modelo_ols_020.fit()
predicciones_train_020 = resultados_ols_020.predict(X_train_020_sm)
predicciones_test_020 = resultados_ols_020.predict(X_test_020_sm)

metricas_ols_020 = {
    **calcular_metricas_regresion(y_train_020, predicciones_train_020, "train"),
    **calcular_metricas_regresion(y_test_020, predicciones_test_020, "test"),
    "r2_ols": float(resultados_ols_020.rsquared),
    "r2_ajustado_ols": float(resultados_ols_020.rsquared_adj),
    "aic_ols": float(resultados_ols_020.aic),
    "bic_ols": float(resultados_ols_020.bic),
    "f_statistic_ols": float(resultados_ols_020.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_020.f_pvalue)
}
omnibus_020, prob_omnibus_020 = omni_normtest(resultados_ols_020.resid)
jarque_bera_020, prob_jb_020, skew_020, kurtosis_020 = jarque_bera(resultados_ols_020.resid)
metricas_ols_020.update({
    "omnibus_ols": float(omnibus_020),
    "prob_omnibus_ols": float(prob_omnibus_020),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_020.resid)),
    "jarque_bera_ols": float(jarque_bera_020),
    "prob_jb_ols": float(prob_jb_020),
    "skew_ols": float(skew_020),
    "kurtosis_ols": float(kurtosis_020),
    "condition_number_ols": float(resultados_ols_020.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_020.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_020.pvalues > 0.05).sum())
})

coeficientes_ols_020 = pd.DataFrame({
    "variable": resultados_ols_020.params.index,
    "coeficiente": resultados_ols_020.params.to_numpy(),
    "p_value": resultados_ols_020.pvalues.to_numpy(),
    "significativa_0_05": resultados_ols_020.pvalues.to_numpy() <= 0.05
})
residuos_test_020 = pd.DataFrame({
    "periodo": df_test_020["periodo"].to_numpy(),
    "real": y_test_020.to_numpy(),
    "prediccion": predicciones_test_020.to_numpy(),
    "residuo": (y_test_020.to_numpy() - predicciones_test_020.to_numpy())
})
comparacion_ols_020 = pd.DataFrame([
    {"modelo": "OLS sin rooms", **metricas_ols_sin_rooms},
    {"modelo": "OLS p-value 0.20 sin rooms", **metricas_ols_020}
])
delta_metricas_020 = {
    f"delta_{metrica}": metricas_ols_020[metrica] - metricas_ols_sin_rooms[metrica]
    for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]
}
tabla_variables_eliminadas_020 = coeficientes_ols_sin_rooms[
    coeficientes_ols_sin_rooms["variable"].isin(variables_eliminar_pvalue_020)
].sort_values("p_value", ascending=False)

display(tabla_variables_eliminadas_020)
display(comparacion_ols_020[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
print(resultados_ols_020.summary())

RUN_OLS_SIN_ROOMS_ID = "b31b9d738b2a4d23afe15d9690a3b9ce"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_pvalue_filtrado_020_sin_rooms"):
    mlflow.set_tag("run_anterior_id", RUN_OLS_SIN_ROOMS_ID)
    mlflow.log_params({
        "dataset": str(RUTA_SALIDA_PVALUE_020),
        "dataset_origen": str(RUTA_SALIDA_SIN_ROOMS),
        "objetivo": "price_per_sqm",
        "corte_temporal": CORTE_TEMPORAL,
        "umbral_pvalue_eliminacion": UMBRAL_PVALUE_020,
        "n_variables_eliminadas_pvalue": len(variables_eliminar_pvalue_020),
        "n_predictores": X_train_020.shape[1],
        "filas_entrenamiento": len(df_train_020),
        "filas_prueba": len(df_test_020),
        "tipo_modelo": "statsmodels_OLS_pvalue_filtrado_020_sin_rooms"
    })
    mlflow.log_metrics({**metricas_ols_020, **delta_metricas_020})
    mlflow.log_table(tabla_variables_eliminadas_020, artifact_file="seleccion/variables_eliminadas_pvalue_020.json")
    mlflow.log_table(comparacion_ols_020, artifact_file="comparacion/comparacion_ols_020.json")
    mlflow.log_table(coeficientes_ols_020, artifact_file="modelo/coeficientes_ols_020.json")
    mlflow.log_table(residuos_test_020, artifact_file="diagnostico/residuos_test_020.json")
    mlflow.log_text(resultados_ols_020.summary().as_text(), "modelo/resumen_ols_020.txt")

print(f"Variables eliminadas por p-value >= {UMBRAL_PVALUE_020}: {len(variables_eliminar_pvalue_020)}")
print(f"Archivo filtrado guardado en: {RUTA_SALIDA_PVALUE_020}")
print("OLS filtrado 0.20 registrado en MLflow para comparación")

,variable,coeficiente,p_value,significativa_0_05
179,metro_station_Novoslobodskaya,703.655845,0.714917,False
148,metro_station_Belorusskaya,869.593966,0.662087,False
2,lon,-3452.973992,0.657554,False
14,district_Alekseevsky,-1066.515338,0.632213,False
205,metro_station_Sevastopolskaya,849.093646,0.615449,False
163,metro_station_Kitay-Gorod,1269.493225,0.575240,False
212,metro_station_Smolenskaya (F),1110.348128,0.569916,False
160,metro_station_Kakhovskaya,-968.186026,0.565852,False
156,metro_station_Elektrozavodskaya,1208.289504,0.548883,False
146,metro_station_Bagrationovskaya,-1257.756730,0.526888,False


,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sin rooms,37086.334875,45133.843386,0.859820,23.174916
1,OLS p-value 0.20 sin rooms,37093.445411,45146.594650,0.859741,23.177022


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1904.
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        23:07:56   Log-Likelihood:            -4.6953e+05
No. Observations:               39658   AIC:                         9.395e+05
Df Residuals:                   39452   BIC:                         9.412e+05
Df Model:                         205                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

🏃 View run ols_temporal_pvalue_filtrado_020_sin_rooms at: http://127.0.0.1:5000/#/experiments/4/runs/12514da87fe64b64b9a9b03a3d0f1961
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Variables eliminadas por p-value >= 0.2: 35
Archivo filtrado guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_020_sin_rooms.csv
OLS filtrado 0.20 registrado en MLflow para comparación


### Interpretación

Este filtro es más agresivo y se aplica sobre la versión que ya no contiene `rooms`. La decisión final debe considerar si la reducción de variables conserva o mejora el desempeño temporal de prueba y la estabilidad del modelo.

## 8. Variables ordenadas por p-value del modelo de la seccion 33 - mercado estandar

Esta seccion carga el dataset estandar de la seccion 32, vuelve a ajustar el OLS temporal solo para obtener sus coeficientes y muestra las variables ordenadas desde el p-value mas alto al mas bajo. Es una tabla de consulta; no genera una nueva corrida ni registra artefactos en MLflow.

In [1]:
RUTA_TABLA_PVALUES = Path("data_sets_transformados/secondary_market_modelado_seccion_33.csv")
CORTE_TEMPORAL_TABLA = "2025-01"
df_tabla_pvalues = pd.read_csv(RUTA_TABLA_PVALUES)
df_train_tabla_pvalues = df_tabla_pvalues[df_tabla_pvalues["periodo"] < CORTE_TEMPORAL_TABLA].copy()
X_train_tabla_pvalues = df_train_tabla_pvalues.drop(columns=["price_per_sqm", "periodo"])
y_train_tabla_pvalues = df_train_tabla_pvalues["price_per_sqm"]
X_train_tabla_pvalues_sm = sm.add_constant(X_train_tabla_pvalues, has_constant="add")
resultados_ols_tabla_pvalues = sm.OLS(y_train_tabla_pvalues, X_train_tabla_pvalues_sm).fit()

tabla_pvalues_sin_living_ordenada = pd.DataFrame({
    "variable": resultados_ols_tabla_pvalues.params.index,
    "p_value": resultados_ols_tabla_pvalues.pvalues.to_numpy()
})
tabla_pvalues_sin_living_ordenada = (
    tabla_pvalues_sin_living_ordenada[tabla_pvalues_sin_living_ordenada["variable"] != "const"]
    .sort_values("p_value", ascending=False)
    .reset_index(drop=True)
)
tabla_pvalues_sin_living_ordenada["p_value"] = tabla_pvalues_sin_living_ordenada["p_value"].round(5)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(tabla_pvalues_sin_living_ordenada)

,variable,p_value
0,metro_station_Elektrozavodskaya,0.99867
1,metro_station_Profsoyuznaya,0.97709
2,metro_station_Molodyozhnaya,0.93891
3,metro_station_Serpukhovskaya,0.92788
4,district_Kievsky,0.90167
5,metro_station_Pechatniki,0.88680
6,metro_station_Borovitskaya,0.82076
7,metro_station_Chistye Prudy,0.80874
8,metro_station_Frunzenskaya,0.80734
9,metro_station_Delovoy Tsentr,0.79559


### Interpretación

Las primeras filas son las variables con mayor p-value dentro del modelo sin `living_area` ni `lon`. Esta tabla sirve para revisar el orden y discutir variables, pero no elimina columnas ni modifica las corridas existentes.

## 9. OLS sin `living_area`

Partimos del modelo `ols_temporal_pvalue_filtrado_090` y eliminamos únicamente `living_area`. La variable `total_area` se conserva porque representa la superficie total. Se mantiene el mismo corte temporal para comparar las métricas de forma consistente.

In [9]:
# Configuración de esta variante del modelo.
VARIABLE_ELIMINADA_LIVING = "living_area"
RUTA_SALIDA_SIN_LIVING = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_090_sin_living_area.csv")

# Trabajamos sobre una copia del dataset p90 y no sobrescribimos el archivo anterior.
if VARIABLE_ELIMINADA_LIVING not in df_ols_filtrado.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_LIVING!r} no está disponible en el modelo p90")
df_ols_sin_living = df_ols_filtrado.drop(columns=[VARIABLE_ELIMINADA_LIVING]).copy()
df_ols_sin_living.to_csv(RUTA_SALIDA_SIN_LIVING, index=False)

# Separamos entrenamiento y prueba respetando el mismo corte temporal mensual.
df_train_sin_living = df_ols_sin_living[df_ols_sin_living["periodo"] < CORTE_TEMPORAL].copy()
df_test_sin_living = df_ols_sin_living[df_ols_sin_living["periodo"] >= CORTE_TEMPORAL].copy()
X_train_sin_living = df_train_sin_living.drop(columns=["price_per_sqm", "periodo"])
X_test_sin_living = df_test_sin_living.drop(columns=["price_per_sqm", "periodo"])
y_train_sin_living = df_train_sin_living["price_per_sqm"]
y_test_sin_living = df_test_sin_living["price_per_sqm"]

# Ajustamos OLS con constante explícita.
X_train_sin_living_sm = sm.add_constant(X_train_sin_living, has_constant="add")
X_test_sin_living_sm = sm.add_constant(X_test_sin_living, has_constant="add")
modelo_ols_sin_living = sm.OLS(y_train_sin_living, X_train_sin_living_sm)
resultados_ols_sin_living = modelo_ols_sin_living.fit()
predicciones_train_sin_living = resultados_ols_sin_living.predict(X_train_sin_living_sm)
predicciones_test_sin_living = resultados_ols_sin_living.predict(X_test_sin_living_sm)

# Calculamos las métricas predictivas y los diagnósticos OLS.
metricas_ols_sin_living = {
    **calcular_metricas_regresion(y_train_sin_living, predicciones_train_sin_living, "train"),
    **calcular_metricas_regresion(y_test_sin_living, predicciones_test_sin_living, "test"),
    "r2_ols": float(resultados_ols_sin_living.rsquared),
    "r2_ajustado_ols": float(resultados_ols_sin_living.rsquared_adj),
    "aic_ols": float(resultados_ols_sin_living.aic),
    "bic_ols": float(resultados_ols_sin_living.bic),
    "f_statistic_ols": float(resultados_ols_sin_living.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_sin_living.f_pvalue)
}
omnibus_sin_living, prob_omnibus_sin_living = omni_normtest(resultados_ols_sin_living.resid)
jarque_bera_sin_living, prob_jb_sin_living, skew_sin_living, kurtosis_sin_living = jarque_bera(resultados_ols_sin_living.resid)
metricas_ols_sin_living.update({
    "omnibus_ols": float(omnibus_sin_living),
    "prob_omnibus_ols": float(prob_omnibus_sin_living),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_sin_living.resid)),
    "jarque_bera_ols": float(jarque_bera_sin_living),
    "prob_jb_ols": float(prob_jb_sin_living),
    "skew_ols": float(skew_sin_living),
    "kurtosis_ols": float(kurtosis_sin_living),
    "condition_number_ols": float(resultados_ols_sin_living.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_sin_living.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_sin_living.pvalues > 0.05).sum())
})

coeficientes_ols_sin_living = pd.DataFrame({
    "variable": resultados_ols_sin_living.params.index,
    "coeficiente": resultados_ols_sin_living.params.to_numpy(),
    "p_value": resultados_ols_sin_living.pvalues.to_numpy(),
    "significativa_0_05": resultados_ols_sin_living.pvalues.to_numpy() <= 0.05
})
residuos_test_sin_living = pd.DataFrame({
    "periodo": df_test_sin_living["periodo"].to_numpy(),
    "real": y_test_sin_living.to_numpy(),
    "prediccion": predicciones_test_sin_living.to_numpy(),
    "residuo": (y_test_sin_living.to_numpy() - predicciones_test_sin_living.to_numpy())
})

comparacion_ols_sin_living = pd.DataFrame([
    {"modelo": "OLS p-value 0.90", **metricas_ols_temporal},
    {"modelo": "OLS sin living_area", **metricas_ols_sin_living}
])
delta_metricas_sin_living = {
    f"delta_{metrica}": metricas_ols_sin_living[metrica] - metricas_ols_temporal[metrica]
    for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]
}

display(comparacion_ols_sin_living[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
print(resultados_ols_sin_living.summary())

# Registramos la nueva variante y sus artefactos en el mismo experimento.
RUN_OLS_P90_ID = "45a11b38934a495eb7a148c8fa9d3f92"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_sin_living_area"):
    mlflow.set_tag("run_anterior_id", RUN_OLS_P90_ID)
    mlflow.log_params({
        "dataset": str(RUTA_SALIDA_SIN_LIVING),
        "dataset_origen": str(RUTA_OLS),
        "objetivo": "price_per_sqm",
        "corte_temporal": CORTE_TEMPORAL,
        "variable_eliminada": VARIABLE_ELIMINADA_LIVING,
        "n_predictores": X_train_sin_living.shape[1],
        "filas_entrenamiento": len(df_train_sin_living),
        "filas_prueba": len(df_test_sin_living),
        "tipo_modelo": "statsmodels_OLS_sin_living_area"
    })
    mlflow.log_metrics({**metricas_ols_sin_living, **delta_metricas_sin_living})
    mlflow.log_table(comparacion_ols_sin_living, artifact_file="comparacion/comparacion_ols_sin_living_area.json")
    mlflow.log_table(coeficientes_ols_sin_living, artifact_file="modelo/coeficientes_ols_sin_living_area.json")
    mlflow.log_table(residuos_test_sin_living, artifact_file="diagnostico/residuos_test_sin_living_area.json")
    mlflow.log_text(resultados_ols_sin_living.summary().as_text(), "modelo/resumen_ols_sin_living_area.txt")

print(f"Archivo sin living_area guardado en: {RUTA_SALIDA_SIN_LIVING}")
print("OLS sin living_area registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS p-value 0.90,37085.586046,45132.632765,0.859828,23.174104
1,OLS sin living_area,37084.571782,45132.405888,0.859829,23.173753


                            OLS Regression Results                            
Dep. Variable:          price_per_sqm   R-squared:                       0.908
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1579.
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        23:07:59   Log-Likelihood:            -4.6952e+05
No. Observations:               39658   AIC:                         9.395e+05
Df Residuals:                   39410   BIC:                         9.417e+05
Df Model:                         247                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

🏃 View run ols_temporal_sin_living_area at: http://127.0.0.1:5000/#/experiments/4/runs/f0682766f6094c9b86a6e78392c0dd78
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Archivo sin living_area guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_090_sin_living_area.csv
OLS sin living_area registrado en MLflow para comparación


### Interpretación

La comparación debe centrarse en las métricas temporales de prueba y en el número de condición. Si quitar `living_area` reduce la colinealidad sin empeorar de forma relevante MAE, RMSE, R² y MAPE, puede ser una simplificación razonable del modelo.

## 10. OLS sin `lon`

Partimos del último dataset sin `living_area` y eliminamos `lon`. `lat` se conserva para evaluar si la coordenada longitudinal aporta información adicional junto con las variables geográficas categóricas.

In [10]:
VARIABLE_ELIMINADA_LON = "lon"
RUTA_SALIDA_SIN_LON = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_090_sin_living_area_sin_lon.csv")
df_ols_sin_living_para_lon = pd.read_csv(RUTA_SALIDA_SIN_LIVING)
if VARIABLE_ELIMINADA_LON not in df_ols_sin_living_para_lon.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_LON!r} no está disponible")
df_ols_sin_lon = df_ols_sin_living_para_lon.drop(columns=[VARIABLE_ELIMINADA_LON]).copy()
df_ols_sin_lon.to_csv(RUTA_SALIDA_SIN_LON, index=False)
df_train_sin_lon = df_ols_sin_lon[df_ols_sin_lon["periodo"] < CORTE_TEMPORAL].copy()
df_test_sin_lon = df_ols_sin_lon[df_ols_sin_lon["periodo"] >= CORTE_TEMPORAL].copy()
X_train_sin_lon = df_train_sin_lon.drop(columns=["price_per_sqm", "periodo"])
X_test_sin_lon = df_test_sin_lon.drop(columns=["price_per_sqm", "periodo"])
y_train_sin_lon = df_train_sin_lon["price_per_sqm"]
y_test_sin_lon = df_test_sin_lon["price_per_sqm"]
X_train_sin_lon_sm = sm.add_constant(X_train_sin_lon, has_constant="add")
X_test_sin_lon_sm = sm.add_constant(X_test_sin_lon, has_constant="add")
resultados_ols_sin_lon = sm.OLS(y_train_sin_lon, X_train_sin_lon_sm).fit()
predicciones_train_sin_lon = resultados_ols_sin_lon.predict(X_train_sin_lon_sm)
predicciones_test_sin_lon = resultados_ols_sin_lon.predict(X_test_sin_lon_sm)
metricas_ols_sin_lon = {
    **calcular_metricas_regresion(y_train_sin_lon, predicciones_train_sin_lon, "train"),
    **calcular_metricas_regresion(y_test_sin_lon, predicciones_test_sin_lon, "test"),
    "r2_ols": float(resultados_ols_sin_lon.rsquared),
    "r2_ajustado_ols": float(resultados_ols_sin_lon.rsquared_adj),
    "aic_ols": float(resultados_ols_sin_lon.aic),
    "bic_ols": float(resultados_ols_sin_lon.bic),
    "f_statistic_ols": float(resultados_ols_sin_lon.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_sin_lon.f_pvalue),
    "condition_number_ols": float(resultados_ols_sin_lon.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_sin_lon.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_sin_lon.pvalues > 0.05).sum())
}
omnibus_sin_lon, prob_omnibus_sin_lon = omni_normtest(resultados_ols_sin_lon.resid)
jarque_bera_sin_lon, prob_jb_sin_lon, skew_sin_lon, kurtosis_sin_lon = jarque_bera(resultados_ols_sin_lon.resid)
metricas_ols_sin_lon.update({
    "omnibus_ols": float(omnibus_sin_lon),
    "prob_omnibus_ols": float(prob_omnibus_sin_lon),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_sin_lon.resid)),
    "jarque_bera_ols": float(jarque_bera_sin_lon),
    "prob_jb_ols": float(prob_jb_sin_lon),
    "skew_ols": float(skew_sin_lon),
    "kurtosis_ols": float(kurtosis_sin_lon)
})
coeficientes_ols_sin_lon = pd.DataFrame({"variable": resultados_ols_sin_lon.params.index, "coeficiente": resultados_ols_sin_lon.params.to_numpy(), "p_value": resultados_ols_sin_lon.pvalues.to_numpy(), "significativa_0_05": resultados_ols_sin_lon.pvalues.to_numpy() <= 0.05})
residuos_test_sin_lon = pd.DataFrame({"periodo": df_test_sin_lon["periodo"].to_numpy(), "real": y_test_sin_lon.to_numpy(), "prediccion": predicciones_test_sin_lon.to_numpy(), "residuo": y_test_sin_lon.to_numpy() - predicciones_test_sin_lon.to_numpy()})
comparacion_ols_sin_lon = pd.DataFrame([{"modelo": "OLS sin living_area", **metricas_ols_sin_living}, {"modelo": "OLS sin living_area ni lon", **metricas_ols_sin_lon}])
delta_metricas_sin_lon = {f"delta_{metrica}": metricas_ols_sin_lon[metrica] - metricas_ols_sin_living[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_sin_lon[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_sin_living_area_sin_lon"):
    mlflow.set_tag("run_anterior_id", "3e5b2762c30f42df8b4805ec3d3c5f3e")
    mlflow.log_params({"dataset": str(RUTA_SALIDA_SIN_LON), "dataset_origen": str(RUTA_SALIDA_SIN_LIVING), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_LON, "n_predictores": X_train_sin_lon.shape[1], "filas_entrenamiento": len(df_train_sin_lon), "filas_prueba": len(df_test_sin_lon), "tipo_modelo": "statsmodels_OLS_sin_living_area_sin_lon"})
    mlflow.log_metrics({**metricas_ols_sin_lon, **delta_metricas_sin_lon})
    mlflow.log_table(comparacion_ols_sin_lon, artifact_file="comparacion/comparacion_ols_sin_living_area_sin_lon.json")
    mlflow.log_table(coeficientes_ols_sin_lon, artifact_file="modelo/coeficientes_ols_sin_living_area_sin_lon.json")
    mlflow.log_table(residuos_test_sin_lon, artifact_file="diagnostico/residuos_test_sin_living_area_sin_lon.json")
    mlflow.log_text(resultados_ols_sin_lon.summary().as_text(), "modelo/resumen_ols_sin_living_area_sin_lon.txt")
print(f"Archivo sin living_area ni lon guardado en: {RUTA_SALIDA_SIN_LON}")
print("OLS sin living_area ni lon registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sin living_area,37084.571782,45132.405888,0.859829,23.173753
1,OLS sin living_area ni lon,37084.667719,45131.738718,0.859834,23.172933


🏃 View run ols_temporal_sin_living_area_sin_lon at: http://127.0.0.1:5000/#/experiments/4/runs/73a2ec1cafaf4e78b6784f79494d2bf7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Archivo sin living_area ni lon guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_090_sin_living_area_sin_lon.csv
OLS sin living_area ni lon registrado en MLflow para comparación


### Interpretación

La eliminación de `lon` se evalúa comparando las métricas temporales y el número de condición contra el modelo anterior. La tabla de p-values de la sección 8 refleja esta variante sin `living_area` ni `lon`.

## 11. OLS sin `district_Pokrovskoye-Streshnevo`

Partimos del modelo sin `living_area` ni `lon` y eliminamos únicamente la dummy `district_Pokrovskoye-Streshnevo`. Se conserva el resto de las variables de distrito y estación de metro.

In [11]:
VARIABLE_ELIMINADA_DISTRICT = "district_Pokrovskoye-Streshnevo"
RUTA_SALIDA_SIN_DISTRICT = Path("data_sets_transformados/secondary_market_modelado_one_hot_pvalue_090_sin_living_area_sin_lon_sin_district_pokrovskoye_streshnevo.csv")
df_ols_sin_lon_para_district = pd.read_csv(RUTA_SALIDA_SIN_LON)
if VARIABLE_ELIMINADA_DISTRICT not in df_ols_sin_lon_para_district.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_DISTRICT!r} no está disponible")
df_ols_sin_district = df_ols_sin_lon_para_district.drop(columns=[VARIABLE_ELIMINADA_DISTRICT]).copy()
df_ols_sin_district.to_csv(RUTA_SALIDA_SIN_DISTRICT, index=False)
df_train_sin_district = df_ols_sin_district[df_ols_sin_district["periodo"] < CORTE_TEMPORAL].copy()
df_test_sin_district = df_ols_sin_district[df_ols_sin_district["periodo"] >= CORTE_TEMPORAL].copy()
X_train_sin_district = df_train_sin_district.drop(columns=["price_per_sqm", "periodo"])
X_test_sin_district = df_test_sin_district.drop(columns=["price_per_sqm", "periodo"])
y_train_sin_district = df_train_sin_district["price_per_sqm"]
y_test_sin_district = df_test_sin_district["price_per_sqm"]
X_train_sin_district_sm = sm.add_constant(X_train_sin_district, has_constant="add")
X_test_sin_district_sm = sm.add_constant(X_test_sin_district, has_constant="add")
resultados_ols_sin_district = sm.OLS(y_train_sin_district, X_train_sin_district_sm).fit()
predicciones_train_sin_district = resultados_ols_sin_district.predict(X_train_sin_district_sm)
predicciones_test_sin_district = resultados_ols_sin_district.predict(X_test_sin_district_sm)
metricas_ols_sin_district = {
    **calcular_metricas_regresion(y_train_sin_district, predicciones_train_sin_district, "train"),
    **calcular_metricas_regresion(y_test_sin_district, predicciones_test_sin_district, "test"),
    "r2_ols": float(resultados_ols_sin_district.rsquared),
    "r2_ajustado_ols": float(resultados_ols_sin_district.rsquared_adj),
    "aic_ols": float(resultados_ols_sin_district.aic),
    "bic_ols": float(resultados_ols_sin_district.bic),
    "f_statistic_ols": float(resultados_ols_sin_district.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_sin_district.f_pvalue),
    "condition_number_ols": float(resultados_ols_sin_district.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_sin_district.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_sin_district.pvalues > 0.05).sum())
}
omnibus_sin_district, prob_omnibus_sin_district = omni_normtest(resultados_ols_sin_district.resid)
jarque_bera_sin_district, prob_jb_sin_district, skew_sin_district, kurtosis_sin_district = jarque_bera(resultados_ols_sin_district.resid)
metricas_ols_sin_district.update({
    "omnibus_ols": float(omnibus_sin_district),
    "prob_omnibus_ols": float(prob_omnibus_sin_district),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_sin_district.resid)),
    "jarque_bera_ols": float(jarque_bera_sin_district),
    "prob_jb_ols": float(prob_jb_sin_district),
    "skew_ols": float(skew_sin_district),
    "kurtosis_ols": float(kurtosis_sin_district)
})
coeficientes_ols_sin_district = pd.DataFrame({"variable": resultados_ols_sin_district.params.index, "coeficiente": resultados_ols_sin_district.params.to_numpy(), "p_value": resultados_ols_sin_district.pvalues.to_numpy(), "significativa_0_05": resultados_ols_sin_district.pvalues.to_numpy() <= 0.05})
residuos_test_sin_district = pd.DataFrame({"periodo": df_test_sin_district["periodo"].to_numpy(), "real": y_test_sin_district.to_numpy(), "prediccion": predicciones_test_sin_district.to_numpy(), "residuo": y_test_sin_district.to_numpy() - predicciones_test_sin_district.to_numpy()})
comparacion_ols_sin_district = pd.DataFrame([{"modelo": "OLS sin living_area ni lon", **metricas_ols_sin_lon}, {"modelo": "OLS sin living_area, lon ni district_Pokrovskoye-Streshnevo", **metricas_ols_sin_district}])
delta_metricas_sin_district = {f"delta_{metrica}": metricas_ols_sin_district[metrica] - metricas_ols_sin_lon[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_sin_district[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_sin_living_area_sin_lon_sin_district_pokrovskoye"):
    mlflow.set_tag("run_anterior_id", "d5feb1cdd6db4f2a884b26c4ecfe3fef")
    mlflow.log_params({"dataset": str(RUTA_SALIDA_SIN_DISTRICT), "dataset_origen": str(RUTA_SALIDA_SIN_LON), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_DISTRICT, "n_predictores": X_train_sin_district.shape[1], "filas_entrenamiento": len(df_train_sin_district), "filas_prueba": len(df_test_sin_district), "tipo_modelo": "statsmodels_OLS_sin_living_area_sin_lon_sin_district"})
    mlflow.log_metrics({**metricas_ols_sin_district, **delta_metricas_sin_district})
    mlflow.log_table(comparacion_ols_sin_district, artifact_file="comparacion/comparacion_ols_sin_living_area_sin_lon_sin_district.json")
    mlflow.log_table(coeficientes_ols_sin_district, artifact_file="modelo/coeficientes_ols_sin_living_area_sin_lon_sin_district.json")
    mlflow.log_table(residuos_test_sin_district, artifact_file="diagnostico/residuos_test_sin_living_area_sin_lon_sin_district.json")
    mlflow.log_text(resultados_ols_sin_district.summary().as_text(), "modelo/resumen_ols_sin_living_area_sin_lon_sin_district.txt")
print(f"Archivo sin living_area, lon ni district_Pokrovskoye-Streshnevo guardado en: {RUTA_SALIDA_SIN_DISTRICT}")
print("OLS sin la dummy de distrito registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sin living_area ni lon,37084.667719,45131.738718,0.859834,23.172933
1,"OLS sin living_area, lon ni district_Pokrovsko...",37084.746277,45131.606415,0.859834,23.172829


🏃 View run ols_temporal_sin_living_area_sin_lon_sin_district_pokrovskoye at: http://127.0.0.1:5000/#/experiments/4/runs/d6beaaeb974c40009471707897db75e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Archivo sin living_area, lon ni district_Pokrovskoye-Streshnevo guardado en: data_sets_transformados\secondary_market_modelado_one_hot_pvalue_090_sin_living_area_sin_lon_sin_district_pokrovskoye_streshnevo.csv
OLS sin la dummy de distrito registrado en MLflow para comparación


### Interpretación

La eliminación de la dummy de distrito se evalúa comparando las métricas temporales y el número de condición contra el modelo sin `living_area` ni `lon`.

## 12. OLS con eliminación agrupada de dummies de `metro_station`

Partimos del dataset de la sección 11 y eliminamos simultáneamente `metro_station_Borovitskaya`, `metro_station_Kozhukhovskaya` y `metro_station_Sretensky Bulvar`. El nuevo dataset usa el número de sección en su nombre.

In [12]:
VARIABLES_ELIMINADAS_SECCION_12 = [
    "metro_station_Borovitskaya",
    "metro_station_Kozhukhovskaya",
    "metro_station_Sretensky Bulvar"
]
RUTA_SALIDA_SECCION_12 = Path("data_sets_transformados/secondary_market_modelado_seccion_12.csv")
df_ols_seccion_12_origen = pd.read_csv(RUTA_SALIDA_SIN_DISTRICT)
columnas_faltantes_seccion_12 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_12 if columna not in df_ols_seccion_12_origen.columns]
if columnas_faltantes_seccion_12:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_12}")
df_ols_seccion_12 = df_ols_seccion_12_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_12).copy()
df_ols_seccion_12.to_csv(RUTA_SALIDA_SECCION_12, index=False)
df_train_seccion_12 = df_ols_seccion_12[df_ols_seccion_12["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_12 = df_ols_seccion_12[df_ols_seccion_12["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_12 = df_train_seccion_12.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_12 = df_test_seccion_12.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_12 = df_train_seccion_12["price_per_sqm"]
y_test_seccion_12 = df_test_seccion_12["price_per_sqm"]
X_train_seccion_12_sm = sm.add_constant(X_train_seccion_12, has_constant="add")
X_test_seccion_12_sm = sm.add_constant(X_test_seccion_12, has_constant="add")
resultados_ols_seccion_12 = sm.OLS(y_train_seccion_12, X_train_seccion_12_sm).fit()
predicciones_train_seccion_12 = resultados_ols_seccion_12.predict(X_train_seccion_12_sm)
predicciones_test_seccion_12 = resultados_ols_seccion_12.predict(X_test_seccion_12_sm)
metricas_ols_seccion_12 = {
    **calcular_metricas_regresion(y_train_seccion_12, predicciones_train_seccion_12, "train"),
    **calcular_metricas_regresion(y_test_seccion_12, predicciones_test_seccion_12, "test"),
    "r2_ols": float(resultados_ols_seccion_12.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_12.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_12.aic),
    "bic_ols": float(resultados_ols_seccion_12.bic),
    "f_statistic_ols": float(resultados_ols_seccion_12.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_12.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_12.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_12.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_12.pvalues > 0.05).sum())
}
omnibus_seccion_12, prob_omnibus_seccion_12 = omni_normtest(resultados_ols_seccion_12.resid)
jarque_bera_seccion_12, prob_jb_seccion_12, skew_seccion_12, kurtosis_seccion_12 = jarque_bera(resultados_ols_seccion_12.resid)
metricas_ols_seccion_12.update({
    "omnibus_ols": float(omnibus_seccion_12),
    "prob_omnibus_ols": float(prob_omnibus_seccion_12),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_12.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_12),
    "prob_jb_ols": float(prob_jb_seccion_12),
    "skew_ols": float(skew_seccion_12),
    "kurtosis_ols": float(kurtosis_seccion_12)
})
coeficientes_ols_seccion_12 = pd.DataFrame({"variable": resultados_ols_seccion_12.params.index, "coeficiente": resultados_ols_seccion_12.params.to_numpy(), "p_value": resultados_ols_seccion_12.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_12.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_12 = pd.DataFrame({"periodo": df_test_seccion_12["periodo"].to_numpy(), "real": y_test_seccion_12.to_numpy(), "prediccion": predicciones_test_seccion_12.to_numpy(), "residuo": y_test_seccion_12.to_numpy() - predicciones_test_seccion_12.to_numpy()})
comparacion_ols_seccion_12 = pd.DataFrame([{"modelo": "OLS sección 11", **metricas_ols_sin_district}, {"modelo": "OLS sección 12", **metricas_ols_seccion_12}])
delta_metricas_seccion_12 = {f"delta_{metrica}": metricas_ols_seccion_12[metrica] - metricas_ols_sin_district[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_12[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_12"):
    mlflow.set_tag("run_anterior_id", "6b350dfdd37548998fbf34db587091b2")
    mlflow.log_params({"seccion": 12, "dataset": str(RUTA_SALIDA_SECCION_12), "dataset_origen": str(RUTA_SALIDA_SIN_DISTRICT), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_12), "n_predictores": X_train_seccion_12.shape[1], "filas_entrenamiento": len(df_train_seccion_12), "filas_prueba": len(df_test_seccion_12), "tipo_modelo": "statsmodels_OLS_seccion_12"})
    mlflow.log_metrics({**metricas_ols_seccion_12, **delta_metricas_seccion_12})
    mlflow.log_table(comparacion_ols_seccion_12, artifact_file="comparacion/comparacion_ols_seccion_12.json")
    mlflow.log_table(coeficientes_ols_seccion_12, artifact_file="modelo/coeficientes_ols_seccion_12.json")
    mlflow.log_table(residuos_test_seccion_12, artifact_file="diagnostico/residuos_test_seccion_12.json")
    mlflow.log_text(resultados_ols_seccion_12.summary().as_text(), "modelo/resumen_ols_seccion_12.txt")
print(f"Dataset de la sección 12 guardado en: {RUTA_SALIDA_SECCION_12}")
print("OLS de la sección 12 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 11,37084.746277,45131.606415,0.859834,23.172829
1,OLS sección 12,37086.131497,45133.296465,0.859824,23.173331


🏃 View run ols_temporal_seccion_12 at: http://127.0.0.1:5000/#/experiments/4/runs/24d38fdbde904f5d9cf31170e5595980
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 12 guardado en: data_sets_transformados\secondary_market_modelado_seccion_12.csv
OLS de la sección 12 registrado en MLflow para comparación


### Interpretación

La eliminación agrupada se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 11.

## 13. OLS sin `metro_station_Borovitskaya`

Partimos del dataset de la sección 11 y eliminamos únicamente `metro_station_Borovitskaya`. El nuevo dataset usa el número de sección en su nombre.

In [13]:
VARIABLE_ELIMINADA_SECCION_13 = "metro_station_Borovitskaya"
RUTA_SALIDA_SECCION_13 = Path("data_sets_transformados/secondary_market_modelado_seccion_13.csv")
df_ols_seccion_13_origen = pd.read_csv(RUTA_SALIDA_SIN_DISTRICT)
if VARIABLE_ELIMINADA_SECCION_13 not in df_ols_seccion_13_origen.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_SECCION_13!r} no está disponible")
df_ols_seccion_13 = df_ols_seccion_13_origen.drop(columns=[VARIABLE_ELIMINADA_SECCION_13]).copy()
df_ols_seccion_13.to_csv(RUTA_SALIDA_SECCION_13, index=False)
df_train_seccion_13 = df_ols_seccion_13[df_ols_seccion_13["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_13 = df_ols_seccion_13[df_ols_seccion_13["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_13 = df_train_seccion_13.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_13 = df_test_seccion_13.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_13 = df_train_seccion_13["price_per_sqm"]
y_test_seccion_13 = df_test_seccion_13["price_per_sqm"]
X_train_seccion_13_sm = sm.add_constant(X_train_seccion_13, has_constant="add")
X_test_seccion_13_sm = sm.add_constant(X_test_seccion_13, has_constant="add")
resultados_ols_seccion_13 = sm.OLS(y_train_seccion_13, X_train_seccion_13_sm).fit()
predicciones_train_seccion_13 = resultados_ols_seccion_13.predict(X_train_seccion_13_sm)
predicciones_test_seccion_13 = resultados_ols_seccion_13.predict(X_test_seccion_13_sm)
metricas_ols_seccion_13 = {
    **calcular_metricas_regresion(y_train_seccion_13, predicciones_train_seccion_13, "train"),
    **calcular_metricas_regresion(y_test_seccion_13, predicciones_test_seccion_13, "test"),
    "r2_ols": float(resultados_ols_seccion_13.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_13.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_13.aic),
    "bic_ols": float(resultados_ols_seccion_13.bic),
    "f_statistic_ols": float(resultados_ols_seccion_13.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_13.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_13.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_13.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_13.pvalues > 0.05).sum())
}
omnibus_seccion_13, prob_omnibus_seccion_13 = omni_normtest(resultados_ols_seccion_13.resid)
jarque_bera_seccion_13, prob_jb_seccion_13, skew_seccion_13, kurtosis_seccion_13 = jarque_bera(resultados_ols_seccion_13.resid)
metricas_ols_seccion_13.update({
    "omnibus_ols": float(omnibus_seccion_13),
    "prob_omnibus_ols": float(prob_omnibus_seccion_13),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_13.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_13),
    "prob_jb_ols": float(prob_jb_seccion_13),
    "skew_ols": float(skew_seccion_13),
    "kurtosis_ols": float(kurtosis_seccion_13)
})
coeficientes_ols_seccion_13 = pd.DataFrame({"variable": resultados_ols_seccion_13.params.index, "coeficiente": resultados_ols_seccion_13.params.to_numpy(), "p_value": resultados_ols_seccion_13.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_13.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_13 = pd.DataFrame({"periodo": df_test_seccion_13["periodo"].to_numpy(), "real": y_test_seccion_13.to_numpy(), "prediccion": predicciones_test_seccion_13.to_numpy(), "residuo": y_test_seccion_13.to_numpy() - predicciones_test_seccion_13.to_numpy()})
comparacion_ols_seccion_13 = pd.DataFrame([{"modelo": "OLS sección 11", **metricas_ols_sin_district}, {"modelo": "OLS sección 13", **metricas_ols_seccion_13}])
delta_metricas_seccion_13 = {f"delta_{metrica}": metricas_ols_seccion_13[metrica] - metricas_ols_sin_district[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_13[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_13"):
    mlflow.set_tag("run_anterior_id", "6b350dfdd37548998fbf34db587091b2")
    mlflow.log_params({"seccion": 13, "dataset": str(RUTA_SALIDA_SECCION_13), "dataset_origen": str(RUTA_SALIDA_SIN_DISTRICT), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_SECCION_13, "n_predictores": X_train_seccion_13.shape[1], "filas_entrenamiento": len(df_train_seccion_13), "filas_prueba": len(df_test_seccion_13), "tipo_modelo": "statsmodels_OLS_seccion_13"})
    mlflow.log_metrics({**metricas_ols_seccion_13, **delta_metricas_seccion_13})
    mlflow.log_table(comparacion_ols_seccion_13, artifact_file="comparacion/comparacion_ols_seccion_13.json")
    mlflow.log_table(coeficientes_ols_seccion_13, artifact_file="modelo/coeficientes_ols_seccion_13.json")
    mlflow.log_table(residuos_test_seccion_13, artifact_file="diagnostico/residuos_test_seccion_13.json")
    mlflow.log_text(resultados_ols_seccion_13.summary().as_text(), "modelo/resumen_ols_seccion_13.txt")
print(f"Dataset de la sección 13 guardado en: {RUTA_SALIDA_SECCION_13}")
print("OLS de la sección 13 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 11,37084.746277,45131.606415,0.859834,23.172829
1,OLS sección 13,37084.817321,45132.177475,0.859831,23.172807


🏃 View run ols_temporal_seccion_13 at: http://127.0.0.1:5000/#/experiments/4/runs/51649632e4ed4b24a9c51f4911d6fc50
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 13 guardado en: data_sets_transformados\secondary_market_modelado_seccion_13.csv
OLS de la sección 13 registrado en MLflow para comparación


### Interpretación

La eliminación individual se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 11.

## 14. OLS sin `metro_station_Sretensky Bulvar`

Partimos del dataset de la sección 11 y eliminamos únicamente `metro_station_Sretensky Bulvar`. El nuevo dataset usa el número de sección en su nombre.

In [14]:
VARIABLE_ELIMINADA_SECCION_14 = "metro_station_Sretensky Bulvar"
RUTA_SALIDA_SECCION_14 = Path("data_sets_transformados/secondary_market_modelado_seccion_14.csv")
df_ols_seccion_14_origen = pd.read_csv(RUTA_SALIDA_SIN_DISTRICT)
if VARIABLE_ELIMINADA_SECCION_14 not in df_ols_seccion_14_origen.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_SECCION_14!r} no está disponible")
df_ols_seccion_14 = df_ols_seccion_14_origen.drop(columns=[VARIABLE_ELIMINADA_SECCION_14]).copy()
df_ols_seccion_14.to_csv(RUTA_SALIDA_SECCION_14, index=False)
df_train_seccion_14 = df_ols_seccion_14[df_ols_seccion_14["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_14 = df_ols_seccion_14[df_ols_seccion_14["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_14 = df_train_seccion_14.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_14 = df_test_seccion_14.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_14 = df_train_seccion_14["price_per_sqm"]
y_test_seccion_14 = df_test_seccion_14["price_per_sqm"]
X_train_seccion_14_sm = sm.add_constant(X_train_seccion_14, has_constant="add")
X_test_seccion_14_sm = sm.add_constant(X_test_seccion_14, has_constant="add")
resultados_ols_seccion_14 = sm.OLS(y_train_seccion_14, X_train_seccion_14_sm).fit()
predicciones_train_seccion_14 = resultados_ols_seccion_14.predict(X_train_seccion_14_sm)
predicciones_test_seccion_14 = resultados_ols_seccion_14.predict(X_test_seccion_14_sm)
metricas_ols_seccion_14 = {
    **calcular_metricas_regresion(y_train_seccion_14, predicciones_train_seccion_14, "train"),
    **calcular_metricas_regresion(y_test_seccion_14, predicciones_test_seccion_14, "test"),
    "r2_ols": float(resultados_ols_seccion_14.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_14.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_14.aic),
    "bic_ols": float(resultados_ols_seccion_14.bic),
    "f_statistic_ols": float(resultados_ols_seccion_14.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_14.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_14.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_14.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_14.pvalues > 0.05).sum())
}
omnibus_seccion_14, prob_omnibus_seccion_14 = omni_normtest(resultados_ols_seccion_14.resid)
jarque_bera_seccion_14, prob_jb_seccion_14, skew_seccion_14, kurtosis_seccion_14 = jarque_bera(resultados_ols_seccion_14.resid)
metricas_ols_seccion_14.update({
    "omnibus_ols": float(omnibus_seccion_14),
    "prob_omnibus_ols": float(prob_omnibus_seccion_14),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_14.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_14),
    "prob_jb_ols": float(prob_jb_seccion_14),
    "skew_ols": float(skew_seccion_14),
    "kurtosis_ols": float(kurtosis_seccion_14)
})
coeficientes_ols_seccion_14 = pd.DataFrame({"variable": resultados_ols_seccion_14.params.index, "coeficiente": resultados_ols_seccion_14.params.to_numpy(), "p_value": resultados_ols_seccion_14.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_14.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_14 = pd.DataFrame({"periodo": df_test_seccion_14["periodo"].to_numpy(), "real": y_test_seccion_14.to_numpy(), "prediccion": predicciones_test_seccion_14.to_numpy(), "residuo": y_test_seccion_14.to_numpy() - predicciones_test_seccion_14.to_numpy()})
comparacion_ols_seccion_14 = pd.DataFrame([{"modelo": "OLS sección 11", **metricas_ols_sin_district}, {"modelo": "OLS sección 14", **metricas_ols_seccion_14}])
delta_metricas_seccion_14 = {f"delta_{metrica}": metricas_ols_seccion_14[metrica] - metricas_ols_sin_district[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_14[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_14"):
    mlflow.set_tag("run_anterior_id", "6b350dfdd37548998fbf34db587091b2")
    mlflow.log_params({"seccion": 14, "dataset": str(RUTA_SALIDA_SECCION_14), "dataset_origen": str(RUTA_SALIDA_SIN_DISTRICT), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_SECCION_14, "n_predictores": X_train_seccion_14.shape[1], "filas_entrenamiento": len(df_train_seccion_14), "filas_prueba": len(df_test_seccion_14), "tipo_modelo": "statsmodels_OLS_seccion_14"})
    mlflow.log_metrics({**metricas_ols_seccion_14, **delta_metricas_seccion_14})
    mlflow.log_table(comparacion_ols_seccion_14, artifact_file="comparacion/comparacion_ols_seccion_14.json")
    mlflow.log_table(coeficientes_ols_seccion_14, artifact_file="modelo/coeficientes_ols_seccion_14.json")
    mlflow.log_table(residuos_test_seccion_14, artifact_file="diagnostico/residuos_test_seccion_14.json")
    mlflow.log_text(resultados_ols_seccion_14.summary().as_text(), "modelo/resumen_ols_seccion_14.txt")
print(f"Dataset de la sección 14 guardado en: {RUTA_SALIDA_SECCION_14}")
print("OLS de la sección 14 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 11,37084.746277,45131.606415,0.859834,23.172829
1,OLS sección 14,37083.704336,45130.953077,0.859838,23.171620


🏃 View run ols_temporal_seccion_14 at: http://127.0.0.1:5000/#/experiments/4/runs/fbfc9ab040a340be8acbedebe7832d74
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 14 guardado en: data_sets_transformados\secondary_market_modelado_seccion_14.csv
OLS de la sección 14 registrado en MLflow para comparación


### Interpretación

La eliminación individual se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 11.

## 15. OLS sin `metro_station_Novoslobodskaya`

Partimos del dataset de la sección 14 y eliminamos `metro_station_Novoslobodskaya` como segundo paso acumulativo. El nuevo dataset usa el número de sección en su nombre.

In [15]:
VARIABLE_ELIMINADA_SECCION_15 = "metro_station_Novoslobodskaya"
RUTA_SALIDA_SECCION_15 = Path("data_sets_transformados/secondary_market_modelado_seccion_15.csv")
RUTA_ORIGEN_SECCION_15 = Path("data_sets_transformados/secondary_market_modelado_seccion_14.csv")
df_ols_seccion_15_origen = pd.read_csv(RUTA_ORIGEN_SECCION_15)
if VARIABLE_ELIMINADA_SECCION_15 not in df_ols_seccion_15_origen.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_SECCION_15!r} no está disponible")
df_ols_seccion_15 = df_ols_seccion_15_origen.drop(columns=[VARIABLE_ELIMINADA_SECCION_15]).copy()
df_ols_seccion_15.to_csv(RUTA_SALIDA_SECCION_15, index=False)
df_train_seccion_15 = df_ols_seccion_15[df_ols_seccion_15["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_15 = df_ols_seccion_15[df_ols_seccion_15["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_15 = df_train_seccion_15.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_15 = df_test_seccion_15.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_15 = df_train_seccion_15["price_per_sqm"]
y_test_seccion_15 = df_test_seccion_15["price_per_sqm"]
X_train_seccion_15_sm = sm.add_constant(X_train_seccion_15, has_constant="add")
X_test_seccion_15_sm = sm.add_constant(X_test_seccion_15, has_constant="add")
resultados_ols_seccion_15 = sm.OLS(y_train_seccion_15, X_train_seccion_15_sm).fit()
predicciones_train_seccion_15 = resultados_ols_seccion_15.predict(X_train_seccion_15_sm)
predicciones_test_seccion_15 = resultados_ols_seccion_15.predict(X_test_seccion_15_sm)
metricas_ols_seccion_15 = {
    **calcular_metricas_regresion(y_train_seccion_15, predicciones_train_seccion_15, "train"),
    **calcular_metricas_regresion(y_test_seccion_15, predicciones_test_seccion_15, "test"),
    "r2_ols": float(resultados_ols_seccion_15.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_15.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_15.aic),
    "bic_ols": float(resultados_ols_seccion_15.bic),
    "f_statistic_ols": float(resultados_ols_seccion_15.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_15.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_15.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_15.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_15.pvalues > 0.05).sum())
}
omnibus_seccion_15, prob_omnibus_seccion_15 = omni_normtest(resultados_ols_seccion_15.resid)
jarque_bera_seccion_15, prob_jb_seccion_15, skew_seccion_15, kurtosis_seccion_15 = jarque_bera(resultados_ols_seccion_15.resid)
metricas_ols_seccion_15.update({
    "omnibus_ols": float(omnibus_seccion_15),
    "prob_omnibus_ols": float(prob_omnibus_seccion_15),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_15.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_15),
    "prob_jb_ols": float(prob_jb_seccion_15),
    "skew_ols": float(skew_seccion_15),
    "kurtosis_ols": float(kurtosis_seccion_15)
})
coeficientes_ols_seccion_15 = pd.DataFrame({"variable": resultados_ols_seccion_15.params.index, "coeficiente": resultados_ols_seccion_15.params.to_numpy(), "p_value": resultados_ols_seccion_15.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_15.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_15 = pd.DataFrame({"periodo": df_test_seccion_15["periodo"].to_numpy(), "real": y_test_seccion_15.to_numpy(), "prediccion": predicciones_test_seccion_15.to_numpy(), "residuo": y_test_seccion_15.to_numpy() - predicciones_test_seccion_15.to_numpy()})
comparacion_ols_seccion_15 = pd.DataFrame([{"modelo": "OLS sección 14", **metricas_ols_seccion_14}, {"modelo": "OLS sección 15", **metricas_ols_seccion_15}])
delta_metricas_seccion_15 = {f"delta_{metrica}": metricas_ols_seccion_15[metrica] - metricas_ols_seccion_14[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_15[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_15"):
    mlflow.set_tag("run_anterior_id", "5a6927ef54e64c77aa3a01b8b32c8713")
    mlflow.log_params({"seccion": 15, "dataset": str(RUTA_SALIDA_SECCION_15), "dataset_origen": str(RUTA_ORIGEN_SECCION_15), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_SECCION_15, "n_predictores": X_train_seccion_15.shape[1], "filas_entrenamiento": len(df_train_seccion_15), "filas_prueba": len(df_test_seccion_15), "tipo_modelo": "statsmodels_OLS_seccion_15_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_15, **delta_metricas_seccion_15})
    mlflow.log_table(comparacion_ols_seccion_15, artifact_file="comparacion/comparacion_ols_seccion_15.json")
    mlflow.log_table(coeficientes_ols_seccion_15, artifact_file="modelo/coeficientes_ols_seccion_15.json")
    mlflow.log_table(residuos_test_seccion_15, artifact_file="diagnostico/residuos_test_seccion_15.json")
    mlflow.log_text(resultados_ols_seccion_15.summary().as_text(), "modelo/resumen_ols_seccion_15.txt")
print(f"Dataset de la sección 15 guardado en: {RUTA_SALIDA_SECCION_15}")
print("OLS de la sección 15 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 14,37083.704336,45130.953077,0.859838,23.171620
1,OLS sección 15,37082.913534,45130.296893,0.859843,23.171044


🏃 View run ols_temporal_seccion_15 at: http://127.0.0.1:5000/#/experiments/4/runs/f362b35b2b4a4c46882d43ec9e730a95
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 15 guardado en: data_sets_transformados\secondary_market_modelado_seccion_15.csv
OLS de la sección 15 registrado en MLflow para comparación


### Interpretación

La eliminación individual se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 11.

## 16. OLS sin `metro_station_Borovitskaya`

Partimos del dataset acumulativo de la sección 15 y eliminamos `metro_station_Borovitskaya`. El nuevo dataset usa el número de sección en su nombre.

In [16]:
VARIABLE_ELIMINADA_SECCION_16 = "metro_station_Borovitskaya"
RUTA_ORIGEN_SECCION_16 = Path("data_sets_transformados/secondary_market_modelado_seccion_15.csv")
RUTA_SALIDA_SECCION_16 = Path("data_sets_transformados/secondary_market_modelado_seccion_16.csv")
df_ols_seccion_16_origen = pd.read_csv(RUTA_ORIGEN_SECCION_16)
if VARIABLE_ELIMINADA_SECCION_16 not in df_ols_seccion_16_origen.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_SECCION_16!r} no está disponible")
df_ols_seccion_16 = df_ols_seccion_16_origen.drop(columns=[VARIABLE_ELIMINADA_SECCION_16]).copy()
df_ols_seccion_16.to_csv(RUTA_SALIDA_SECCION_16, index=False)
df_train_seccion_16 = df_ols_seccion_16[df_ols_seccion_16["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_16 = df_ols_seccion_16[df_ols_seccion_16["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_16 = df_train_seccion_16.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_16 = df_test_seccion_16.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_16 = df_train_seccion_16["price_per_sqm"]
y_test_seccion_16 = df_test_seccion_16["price_per_sqm"]
X_train_seccion_16_sm = sm.add_constant(X_train_seccion_16, has_constant="add")
X_test_seccion_16_sm = sm.add_constant(X_test_seccion_16, has_constant="add")
resultados_ols_seccion_16 = sm.OLS(y_train_seccion_16, X_train_seccion_16_sm).fit()
predicciones_train_seccion_16 = resultados_ols_seccion_16.predict(X_train_seccion_16_sm)
predicciones_test_seccion_16 = resultados_ols_seccion_16.predict(X_test_seccion_16_sm)
metricas_ols_seccion_16 = {
    **calcular_metricas_regresion(y_train_seccion_16, predicciones_train_seccion_16, "train"),
    **calcular_metricas_regresion(y_test_seccion_16, predicciones_test_seccion_16, "test"),
    "r2_ols": float(resultados_ols_seccion_16.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_16.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_16.aic),
    "bic_ols": float(resultados_ols_seccion_16.bic),
    "f_statistic_ols": float(resultados_ols_seccion_16.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_16.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_16.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_16.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_16.pvalues > 0.05).sum())
}
omnibus_seccion_16, prob_omnibus_seccion_16 = omni_normtest(resultados_ols_seccion_16.resid)
jarque_bera_seccion_16, prob_jb_seccion_16, skew_seccion_16, kurtosis_seccion_16 = jarque_bera(resultados_ols_seccion_16.resid)
metricas_ols_seccion_16.update({
    "omnibus_ols": float(omnibus_seccion_16),
    "prob_omnibus_ols": float(prob_omnibus_seccion_16),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_16.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_16),
    "prob_jb_ols": float(prob_jb_seccion_16),
    "skew_ols": float(skew_seccion_16),
    "kurtosis_ols": float(kurtosis_seccion_16)
})
coeficientes_ols_seccion_16 = pd.DataFrame({"variable": resultados_ols_seccion_16.params.index, "coeficiente": resultados_ols_seccion_16.params.to_numpy(), "p_value": resultados_ols_seccion_16.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_16.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_16 = pd.DataFrame({"periodo": df_test_seccion_16["periodo"].to_numpy(), "real": y_test_seccion_16.to_numpy(), "prediccion": predicciones_test_seccion_16.to_numpy(), "residuo": y_test_seccion_16.to_numpy() - predicciones_test_seccion_16.to_numpy()})
comparacion_ols_seccion_16 = pd.DataFrame([{"modelo": "OLS sección 15", **metricas_ols_seccion_15}, {"modelo": "OLS sección 16", **metricas_ols_seccion_16}])
delta_metricas_seccion_16 = {f"delta_{metrica}": metricas_ols_seccion_16[metrica] - metricas_ols_seccion_15[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_16[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_16"):
    mlflow.set_tag("run_anterior_id", "7841515ae07b4790843dadec44ce01a2")
    mlflow.log_params({"seccion": 16, "dataset": str(RUTA_SALIDA_SECCION_16), "dataset_origen": str(RUTA_ORIGEN_SECCION_16), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_SECCION_16, "n_predictores": X_train_seccion_16.shape[1], "filas_entrenamiento": len(df_train_seccion_16), "filas_prueba": len(df_test_seccion_16), "tipo_modelo": "statsmodels_OLS_seccion_16_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_16, **delta_metricas_seccion_16})
    mlflow.log_table(comparacion_ols_seccion_16, artifact_file="comparacion/comparacion_ols_seccion_16.json")
    mlflow.log_table(coeficientes_ols_seccion_16, artifact_file="modelo/coeficientes_ols_seccion_16.json")
    mlflow.log_table(residuos_test_seccion_16, artifact_file="diagnostico/residuos_test_seccion_16.json")
    mlflow.log_text(resultados_ols_seccion_16.summary().as_text(), "modelo/resumen_ols_seccion_16.txt")
print(f"Dataset de la sección 16 guardado en: {RUTA_SALIDA_SECCION_16}")
print("OLS de la sección 16 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 15,37082.913534,45130.296893,0.859843,23.171044
1,OLS sección 16,37083.088468,45130.828457,0.859839,23.171151


🏃 View run ols_temporal_seccion_16 at: http://127.0.0.1:5000/#/experiments/4/runs/3c7f44cc30654985ba27c02a0ec9da7e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 16 guardado en: data_sets_transformados\secondary_market_modelado_seccion_16.csv
OLS de la sección 16 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 15.

## 17. OLS sin `metro_station_Belorusskaya`

Partimos del dataset acumulativo de la sección 16 y eliminamos `metro_station_Belorusskaya`. El nuevo dataset usa el número de sección en su nombre.

In [17]:
VARIABLE_ELIMINADA_SECCION_17 = "metro_station_Belorusskaya"
RUTA_ORIGEN_SECCION_17 = Path("data_sets_transformados/secondary_market_modelado_seccion_16.csv")
RUTA_SALIDA_SECCION_17 = Path("data_sets_transformados/secondary_market_modelado_seccion_17.csv")
df_ols_seccion_17_origen = pd.read_csv(RUTA_ORIGEN_SECCION_17)
if VARIABLE_ELIMINADA_SECCION_17 not in df_ols_seccion_17_origen.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_SECCION_17!r} no está disponible")
df_ols_seccion_17 = df_ols_seccion_17_origen.drop(columns=[VARIABLE_ELIMINADA_SECCION_17]).copy()
df_ols_seccion_17.to_csv(RUTA_SALIDA_SECCION_17, index=False)
df_train_seccion_17 = df_ols_seccion_17[df_ols_seccion_17["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_17 = df_ols_seccion_17[df_ols_seccion_17["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_17 = df_train_seccion_17.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_17 = df_test_seccion_17.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_17 = df_train_seccion_17["price_per_sqm"]
y_test_seccion_17 = df_test_seccion_17["price_per_sqm"]
X_train_seccion_17_sm = sm.add_constant(X_train_seccion_17, has_constant="add")
X_test_seccion_17_sm = sm.add_constant(X_test_seccion_17, has_constant="add")
resultados_ols_seccion_17 = sm.OLS(y_train_seccion_17, X_train_seccion_17_sm).fit()
predicciones_train_seccion_17 = resultados_ols_seccion_17.predict(X_train_seccion_17_sm)
predicciones_test_seccion_17 = resultados_ols_seccion_17.predict(X_test_seccion_17_sm)
metricas_ols_seccion_17 = {
    **calcular_metricas_regresion(y_train_seccion_17, predicciones_train_seccion_17, "train"),
    **calcular_metricas_regresion(y_test_seccion_17, predicciones_test_seccion_17, "test"),
    "r2_ols": float(resultados_ols_seccion_17.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_17.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_17.aic),
    "bic_ols": float(resultados_ols_seccion_17.bic),
    "f_statistic_ols": float(resultados_ols_seccion_17.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_17.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_17.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_17.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_17.pvalues > 0.05).sum())
}
omnibus_seccion_17, prob_omnibus_seccion_17 = omni_normtest(resultados_ols_seccion_17.resid)
jarque_bera_seccion_17, prob_jb_seccion_17, skew_seccion_17, kurtosis_seccion_17 = jarque_bera(resultados_ols_seccion_17.resid)
metricas_ols_seccion_17.update({
    "omnibus_ols": float(omnibus_seccion_17),
    "prob_omnibus_ols": float(prob_omnibus_seccion_17),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_17.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_17),
    "prob_jb_ols": float(prob_jb_seccion_17),
    "skew_ols": float(skew_seccion_17),
    "kurtosis_ols": float(kurtosis_seccion_17)
})
coeficientes_ols_seccion_17 = pd.DataFrame({"variable": resultados_ols_seccion_17.params.index, "coeficiente": resultados_ols_seccion_17.params.to_numpy(), "p_value": resultados_ols_seccion_17.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_17.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_17 = pd.DataFrame({"periodo": df_test_seccion_17["periodo"].to_numpy(), "real": y_test_seccion_17.to_numpy(), "prediccion": predicciones_test_seccion_17.to_numpy(), "residuo": y_test_seccion_17.to_numpy() - predicciones_test_seccion_17.to_numpy()})
comparacion_ols_seccion_17 = pd.DataFrame([{"modelo": "OLS sección 16", **metricas_ols_seccion_16}, {"modelo": "OLS sección 17", **metricas_ols_seccion_17}])
delta_metricas_seccion_17 = {f"delta_{metrica}": metricas_ols_seccion_17[metrica] - metricas_ols_seccion_16[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_17[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_17"):
    mlflow.set_tag("run_anterior_id", "386e759d672e4f07a2bce9f7800564a8")
    mlflow.log_params({"seccion": 17, "dataset": str(RUTA_SALIDA_SECCION_17), "dataset_origen": str(RUTA_ORIGEN_SECCION_17), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_SECCION_17, "n_predictores": X_train_seccion_17.shape[1], "filas_entrenamiento": len(df_train_seccion_17), "filas_prueba": len(df_test_seccion_17), "tipo_modelo": "statsmodels_OLS_seccion_17_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_17, **delta_metricas_seccion_17})
    mlflow.log_table(comparacion_ols_seccion_17, artifact_file="comparacion/comparacion_ols_seccion_17.json")
    mlflow.log_table(coeficientes_ols_seccion_17, artifact_file="modelo/coeficientes_ols_seccion_17.json")
    mlflow.log_table(residuos_test_seccion_17, artifact_file="diagnostico/residuos_test_seccion_17.json")
    mlflow.log_text(resultados_ols_seccion_17.summary().as_text(), "modelo/resumen_ols_seccion_17.txt")
print(f"Dataset de la sección 17 guardado en: {RUTA_SALIDA_SECCION_17}")
print("OLS de la sección 17 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 16,37083.088468,45130.828457,0.859839,23.171151
1,OLS sección 17,37081.504127,45129.477642,0.859848,23.169997


🏃 View run ols_temporal_seccion_17 at: http://127.0.0.1:5000/#/experiments/4/runs/5989b6d2ab134941a73d3abb1e3f30b1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 17 guardado en: data_sets_transformados\secondary_market_modelado_seccion_17.csv
OLS de la sección 17 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 16.

## 18. OLS sin `metro_station_Sevastopolskaya`

Partimos del dataset acumulativo de la sección 17 y eliminamos `metro_station_Sevastopolskaya`. El nuevo dataset usa el número de sección en su nombre.

In [18]:
VARIABLE_ELIMINADA_SECCION_18 = "metro_station_Sevastopolskaya"
RUTA_ORIGEN_SECCION_18 = Path("data_sets_transformados/secondary_market_modelado_seccion_17.csv")
RUTA_SALIDA_SECCION_18 = Path("data_sets_transformados/secondary_market_modelado_seccion_18.csv")
df_ols_seccion_18_origen = pd.read_csv(RUTA_ORIGEN_SECCION_18)
if VARIABLE_ELIMINADA_SECCION_18 not in df_ols_seccion_18_origen.columns:
    raise KeyError(f"La columna {VARIABLE_ELIMINADA_SECCION_18!r} no está disponible")
df_ols_seccion_18 = df_ols_seccion_18_origen.drop(columns=[VARIABLE_ELIMINADA_SECCION_18]).copy()
df_ols_seccion_18.to_csv(RUTA_SALIDA_SECCION_18, index=False)
df_train_seccion_18 = df_ols_seccion_18[df_ols_seccion_18["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_18 = df_ols_seccion_18[df_ols_seccion_18["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_18 = df_train_seccion_18.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_18 = df_test_seccion_18.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_18 = df_train_seccion_18["price_per_sqm"]
y_test_seccion_18 = df_test_seccion_18["price_per_sqm"]
X_train_seccion_18_sm = sm.add_constant(X_train_seccion_18, has_constant="add")
X_test_seccion_18_sm = sm.add_constant(X_test_seccion_18, has_constant="add")
resultados_ols_seccion_18 = sm.OLS(y_train_seccion_18, X_train_seccion_18_sm).fit()
predicciones_train_seccion_18 = resultados_ols_seccion_18.predict(X_train_seccion_18_sm)
predicciones_test_seccion_18 = resultados_ols_seccion_18.predict(X_test_seccion_18_sm)
metricas_ols_seccion_18 = {
    **calcular_metricas_regresion(y_train_seccion_18, predicciones_train_seccion_18, "train"),
    **calcular_metricas_regresion(y_test_seccion_18, predicciones_test_seccion_18, "test"),
    "r2_ols": float(resultados_ols_seccion_18.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_18.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_18.aic),
    "bic_ols": float(resultados_ols_seccion_18.bic),
    "f_statistic_ols": float(resultados_ols_seccion_18.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_18.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_18.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_18.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_18.pvalues > 0.05).sum())
}
omnibus_seccion_18, prob_omnibus_seccion_18 = omni_normtest(resultados_ols_seccion_18.resid)
jarque_bera_seccion_18, prob_jb_seccion_18, skew_seccion_18, kurtosis_seccion_18 = jarque_bera(resultados_ols_seccion_18.resid)
metricas_ols_seccion_18.update({
    "omnibus_ols": float(omnibus_seccion_18),
    "prob_omnibus_ols": float(prob_omnibus_seccion_18),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_18.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_18),
    "prob_jb_ols": float(prob_jb_seccion_18),
    "skew_ols": float(skew_seccion_18),
    "kurtosis_ols": float(kurtosis_seccion_18)
})
coeficientes_ols_seccion_18 = pd.DataFrame({"variable": resultados_ols_seccion_18.params.index, "coeficiente": resultados_ols_seccion_18.params.to_numpy(), "p_value": resultados_ols_seccion_18.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_18.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_18 = pd.DataFrame({"periodo": df_test_seccion_18["periodo"].to_numpy(), "real": y_test_seccion_18.to_numpy(), "prediccion": predicciones_test_seccion_18.to_numpy(), "residuo": y_test_seccion_18.to_numpy() - predicciones_test_seccion_18.to_numpy()})
comparacion_ols_seccion_18 = pd.DataFrame([{"modelo": "OLS sección 17", **metricas_ols_seccion_17}, {"modelo": "OLS sección 18", **metricas_ols_seccion_18}])
delta_metricas_seccion_18 = {f"delta_{metrica}": metricas_ols_seccion_18[metrica] - metricas_ols_seccion_17[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_18[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_18"):
    mlflow.set_tag("run_anterior_id", "27110ebe9f424d31a10ec9ba41d5a74a")
    mlflow.log_params({"seccion": 18, "dataset": str(RUTA_SALIDA_SECCION_18), "dataset_origen": str(RUTA_ORIGEN_SECCION_18), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variable_eliminada": VARIABLE_ELIMINADA_SECCION_18, "n_predictores": X_train_seccion_18.shape[1], "filas_entrenamiento": len(df_train_seccion_18), "filas_prueba": len(df_test_seccion_18), "tipo_modelo": "statsmodels_OLS_seccion_18_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_18, **delta_metricas_seccion_18})
    mlflow.log_table(comparacion_ols_seccion_18, artifact_file="comparacion/comparacion_ols_seccion_18.json")
    mlflow.log_table(coeficientes_ols_seccion_18, artifact_file="modelo/coeficientes_ols_seccion_18.json")
    mlflow.log_table(residuos_test_seccion_18, artifact_file="diagnostico/residuos_test_seccion_18.json")
    mlflow.log_text(resultados_ols_seccion_18.summary().as_text(), "modelo/resumen_ols_seccion_18.txt")
print(f"Dataset de la sección 18 guardado en: {RUTA_SALIDA_SECCION_18}")
print("OLS de la sección 18 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 17,37081.504127,45129.477642,0.859848,23.169997
1,OLS sección 18,37081.053061,45129.098683,0.859850,23.169435


🏃 View run ols_temporal_seccion_18 at: http://127.0.0.1:5000/#/experiments/4/runs/007c83a834694761a651b573027ccb37
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 18 guardado en: data_sets_transformados\secondary_market_modelado_seccion_18.csv
OLS de la sección 18 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 17.

## 19. OLS sin tres candidatas de la sección 8

Partimos del dataset acumulativo de la sección 18 y eliminamos `district_Ochakovo-Matveyevskoye`, `metro_station_Kitay-Gorod` y `metro_station_Smolenskaya (F)`. El nuevo dataset usa el número de sección en su nombre.

In [19]:
VARIABLES_ELIMINADAS_SECCION_19 = [
    "district_Ochakovo-Matveyevskoye",
    "metro_station_Kitay-Gorod",
    "metro_station_Smolenskaya (F)"
]
RUTA_ORIGEN_SECCION_19 = Path("data_sets_transformados/secondary_market_modelado_seccion_18.csv")
RUTA_SALIDA_SECCION_19 = Path("data_sets_transformados/secondary_market_modelado_seccion_19.csv")
df_ols_seccion_19_origen = pd.read_csv(RUTA_ORIGEN_SECCION_19)
columnas_faltantes_seccion_19 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_19 if columna not in df_ols_seccion_19_origen.columns]
if columnas_faltantes_seccion_19:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_19}")
df_ols_seccion_19 = df_ols_seccion_19_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_19).copy()
df_ols_seccion_19.to_csv(RUTA_SALIDA_SECCION_19, index=False)
df_train_seccion_19 = df_ols_seccion_19[df_ols_seccion_19["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_19 = df_ols_seccion_19[df_ols_seccion_19["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_19 = df_train_seccion_19.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_19 = df_test_seccion_19.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_19 = df_train_seccion_19["price_per_sqm"]
y_test_seccion_19 = df_test_seccion_19["price_per_sqm"]
X_train_seccion_19_sm = sm.add_constant(X_train_seccion_19, has_constant="add")
X_test_seccion_19_sm = sm.add_constant(X_test_seccion_19, has_constant="add")
resultados_ols_seccion_19 = sm.OLS(y_train_seccion_19, X_train_seccion_19_sm).fit()
predicciones_train_seccion_19 = resultados_ols_seccion_19.predict(X_train_seccion_19_sm)
predicciones_test_seccion_19 = resultados_ols_seccion_19.predict(X_test_seccion_19_sm)
metricas_ols_seccion_19 = {
    **calcular_metricas_regresion(y_train_seccion_19, predicciones_train_seccion_19, "train"),
    **calcular_metricas_regresion(y_test_seccion_19, predicciones_test_seccion_19, "test"),
    "r2_ols": float(resultados_ols_seccion_19.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_19.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_19.aic),
    "bic_ols": float(resultados_ols_seccion_19.bic),
    "f_statistic_ols": float(resultados_ols_seccion_19.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_19.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_19.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_19.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_19.pvalues > 0.05).sum())
}
omnibus_seccion_19, prob_omnibus_seccion_19 = omni_normtest(resultados_ols_seccion_19.resid)
jarque_bera_seccion_19, prob_jb_seccion_19, skew_seccion_19, kurtosis_seccion_19 = jarque_bera(resultados_ols_seccion_19.resid)
metricas_ols_seccion_19.update({
    "omnibus_ols": float(omnibus_seccion_19),
    "prob_omnibus_ols": float(prob_omnibus_seccion_19),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_19.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_19),
    "prob_jb_ols": float(prob_jb_seccion_19),
    "skew_ols": float(skew_seccion_19),
    "kurtosis_ols": float(kurtosis_seccion_19)
})
coeficientes_ols_seccion_19 = pd.DataFrame({"variable": resultados_ols_seccion_19.params.index, "coeficiente": resultados_ols_seccion_19.params.to_numpy(), "p_value": resultados_ols_seccion_19.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_19.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_19 = pd.DataFrame({"periodo": df_test_seccion_19["periodo"].to_numpy(), "real": y_test_seccion_19.to_numpy(), "prediccion": predicciones_test_seccion_19.to_numpy(), "residuo": y_test_seccion_19.to_numpy() - predicciones_test_seccion_19.to_numpy()})
comparacion_ols_seccion_19 = pd.DataFrame([{"modelo": "OLS sección 18", **metricas_ols_seccion_18}, {"modelo": "OLS sección 19", **metricas_ols_seccion_19}])
delta_metricas_seccion_19 = {f"delta_{metrica}": metricas_ols_seccion_19[metrica] - metricas_ols_seccion_18[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_19[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_19"):
    mlflow.set_tag("run_anterior_id", "32c10618074e4a35bb627541e766e934")
    mlflow.log_params({"seccion": 19, "dataset": str(RUTA_SALIDA_SECCION_19), "dataset_origen": str(RUTA_ORIGEN_SECCION_19), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_19), "n_predictores": X_train_seccion_19.shape[1], "filas_entrenamiento": len(df_train_seccion_19), "filas_prueba": len(df_test_seccion_19), "tipo_modelo": "statsmodels_OLS_seccion_19_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_19, **delta_metricas_seccion_19})
    mlflow.log_table(comparacion_ols_seccion_19, artifact_file="comparacion/comparacion_ols_seccion_19.json")
    mlflow.log_table(coeficientes_ols_seccion_19, artifact_file="modelo/coeficientes_ols_seccion_19.json")
    mlflow.log_table(residuos_test_seccion_19, artifact_file="diagnostico/residuos_test_seccion_19.json")
    mlflow.log_text(resultados_ols_seccion_19.summary().as_text(), "modelo/resumen_ols_seccion_19.txt")
print(f"Dataset de la sección 19 guardado en: {RUTA_SALIDA_SECCION_19}")
print("OLS de la sección 19 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 18,37081.053061,45129.098683,0.859850,23.169435
1,OLS sección 19,37079.847236,45129.182624,0.859849,23.169380


🏃 View run ols_temporal_seccion_19 at: http://127.0.0.1:5000/#/experiments/4/runs/8bfe94adfe4e45659ea3f9e25e03f479
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 19 guardado en: data_sets_transformados\secondary_market_modelado_seccion_19.csv
OLS de la sección 19 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las tres candidatas se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 18.

## 20. OLS sin tres candidatas de la sección 8

Partimos del dataset acumulativo de la sección 19 y eliminamos `metro_station_Elektrozavodskaya`, `metro_station_Pechatniki` y `metro_station_Dostoevskaya`. El nuevo dataset usa el número de sección en su nombre.

In [20]:
VARIABLES_ELIMINADAS_SECCION_20 = [
    "metro_station_Elektrozavodskaya",
    "metro_station_Pechatniki",
    "metro_station_Dostoevskaya"
]
RUTA_ORIGEN_SECCION_20 = Path("data_sets_transformados/secondary_market_modelado_seccion_19.csv")
RUTA_SALIDA_SECCION_20 = Path("data_sets_transformados/secondary_market_modelado_seccion_20.csv")
df_ols_seccion_20_origen = pd.read_csv(RUTA_ORIGEN_SECCION_20)
columnas_faltantes_seccion_20 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_20 if columna not in df_ols_seccion_20_origen.columns]
if columnas_faltantes_seccion_20:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_20}")
df_ols_seccion_20 = df_ols_seccion_20_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_20).copy()
df_ols_seccion_20.to_csv(RUTA_SALIDA_SECCION_20, index=False)
df_train_seccion_20 = df_ols_seccion_20[df_ols_seccion_20["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_20 = df_ols_seccion_20[df_ols_seccion_20["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_20 = df_train_seccion_20.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_20 = df_test_seccion_20.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_20 = df_train_seccion_20["price_per_sqm"]
y_test_seccion_20 = df_test_seccion_20["price_per_sqm"]
X_train_seccion_20_sm = sm.add_constant(X_train_seccion_20, has_constant="add")
X_test_seccion_20_sm = sm.add_constant(X_test_seccion_20, has_constant="add")
resultados_ols_seccion_20 = sm.OLS(y_train_seccion_20, X_train_seccion_20_sm).fit()
predicciones_train_seccion_20 = resultados_ols_seccion_20.predict(X_train_seccion_20_sm)
predicciones_test_seccion_20 = resultados_ols_seccion_20.predict(X_test_seccion_20_sm)
metricas_ols_seccion_20 = {
    **calcular_metricas_regresion(y_train_seccion_20, predicciones_train_seccion_20, "train"),
    **calcular_metricas_regresion(y_test_seccion_20, predicciones_test_seccion_20, "test"),
    "r2_ols": float(resultados_ols_seccion_20.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_20.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_20.aic),
    "bic_ols": float(resultados_ols_seccion_20.bic),
    "f_statistic_ols": float(resultados_ols_seccion_20.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_20.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_20.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_20.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_20.pvalues > 0.05).sum())
}
omnibus_seccion_20, prob_omnibus_seccion_20 = omni_normtest(resultados_ols_seccion_20.resid)
jarque_bera_seccion_20, prob_jb_seccion_20, skew_seccion_20, kurtosis_seccion_20 = jarque_bera(resultados_ols_seccion_20.resid)
metricas_ols_seccion_20.update({
    "omnibus_ols": float(omnibus_seccion_20),
    "prob_omnibus_ols": float(prob_omnibus_seccion_20),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_20.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_20),
    "prob_jb_ols": float(prob_jb_seccion_20),
    "skew_ols": float(skew_seccion_20),
    "kurtosis_ols": float(kurtosis_seccion_20)
})
coeficientes_ols_seccion_20 = pd.DataFrame({"variable": resultados_ols_seccion_20.params.index, "coeficiente": resultados_ols_seccion_20.params.to_numpy(), "p_value": resultados_ols_seccion_20.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_20.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_20 = pd.DataFrame({"periodo": df_test_seccion_20["periodo"].to_numpy(), "real": y_test_seccion_20.to_numpy(), "prediccion": predicciones_test_seccion_20.to_numpy(), "residuo": y_test_seccion_20.to_numpy() - predicciones_test_seccion_20.to_numpy()})
comparacion_ols_seccion_20 = pd.DataFrame([{"modelo": "OLS sección 19", **metricas_ols_seccion_19}, {"modelo": "OLS sección 20", **metricas_ols_seccion_20}])
delta_metricas_seccion_20 = {f"delta_{metrica}": metricas_ols_seccion_20[metrica] - metricas_ols_seccion_19[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_20[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_20"):
    mlflow.set_tag("run_anterior_id", "cdef663475e34e24bae61bb7fc353318")
    mlflow.log_params({"seccion": 20, "dataset": str(RUTA_SALIDA_SECCION_20), "dataset_origen": str(RUTA_ORIGEN_SECCION_20), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_20), "n_predictores": X_train_seccion_20.shape[1], "filas_entrenamiento": len(df_train_seccion_20), "filas_prueba": len(df_test_seccion_20), "tipo_modelo": "statsmodels_OLS_seccion_20_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_20, **delta_metricas_seccion_20})
    mlflow.log_table(comparacion_ols_seccion_20, artifact_file="comparacion/comparacion_ols_seccion_20.json")
    mlflow.log_table(coeficientes_ols_seccion_20, artifact_file="modelo/coeficientes_ols_seccion_20.json")
    mlflow.log_table(residuos_test_seccion_20, artifact_file="diagnostico/residuos_test_seccion_20.json")
    mlflow.log_text(resultados_ols_seccion_20.summary().as_text(), "modelo/resumen_ols_seccion_20.txt")
print(f"Dataset de la sección 20 guardado en: {RUTA_SALIDA_SECCION_20}")
print("OLS de la sección 20 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 19,37079.847236,45129.182624,0.859849,23.169380
1,OLS sección 20,37076.824980,45126.757949,0.859864,23.166935


🏃 View run ols_temporal_seccion_20 at: http://127.0.0.1:5000/#/experiments/4/runs/997b1ed48d944e488d1d7cb5375c8dc8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 20 guardado en: data_sets_transformados\secondary_market_modelado_seccion_20.csv
OLS de la sección 20 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las tres candidatas se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 19.

## 21. OLS sin tres candidatas de la sección 8

Partimos del dataset acumulativo de la sección 20 y eliminamos `metro_station_Lubyanka`, `district_Sokolniki` y `metro_station_Chistye Prudy`. El nuevo dataset usa el número de sección en su nombre.

In [21]:
VARIABLES_ELIMINADAS_SECCION_21 = [
    "metro_station_Lubyanka",
    "district_Sokolniki",
    "metro_station_Chistye Prudy"
]
RUTA_ORIGEN_SECCION_21 = Path("data_sets_transformados/secondary_market_modelado_seccion_20.csv")
RUTA_SALIDA_SECCION_21 = Path("data_sets_transformados/secondary_market_modelado_seccion_21.csv")
df_ols_seccion_21_origen = pd.read_csv(RUTA_ORIGEN_SECCION_21)
columnas_faltantes_seccion_21 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_21 if columna not in df_ols_seccion_21_origen.columns]
if columnas_faltantes_seccion_21:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_21}")
df_ols_seccion_21 = df_ols_seccion_21_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_21).copy()
df_ols_seccion_21.to_csv(RUTA_SALIDA_SECCION_21, index=False)
df_train_seccion_21 = df_ols_seccion_21[df_ols_seccion_21["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_21 = df_ols_seccion_21[df_ols_seccion_21["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_21 = df_train_seccion_21.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_21 = df_test_seccion_21.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_21 = df_train_seccion_21["price_per_sqm"]
y_test_seccion_21 = df_test_seccion_21["price_per_sqm"]
X_train_seccion_21_sm = sm.add_constant(X_train_seccion_21, has_constant="add")
X_test_seccion_21_sm = sm.add_constant(X_test_seccion_21, has_constant="add")
resultados_ols_seccion_21 = sm.OLS(y_train_seccion_21, X_train_seccion_21_sm).fit()
predicciones_train_seccion_21 = resultados_ols_seccion_21.predict(X_train_seccion_21_sm)
predicciones_test_seccion_21 = resultados_ols_seccion_21.predict(X_test_seccion_21_sm)
metricas_ols_seccion_21 = {
    **calcular_metricas_regresion(y_train_seccion_21, predicciones_train_seccion_21, "train"),
    **calcular_metricas_regresion(y_test_seccion_21, predicciones_test_seccion_21, "test"),
    "r2_ols": float(resultados_ols_seccion_21.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_21.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_21.aic),
    "bic_ols": float(resultados_ols_seccion_21.bic),
    "f_statistic_ols": float(resultados_ols_seccion_21.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_21.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_21.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_21.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_21.pvalues > 0.05).sum())
}
omnibus_seccion_21, prob_omnibus_seccion_21 = omni_normtest(resultados_ols_seccion_21.resid)
jarque_bera_seccion_21, prob_jb_seccion_21, skew_seccion_21, kurtosis_seccion_21 = jarque_bera(resultados_ols_seccion_21.resid)
metricas_ols_seccion_21.update({
    "omnibus_ols": float(omnibus_seccion_21),
    "prob_omnibus_ols": float(prob_omnibus_seccion_21),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_21.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_21),
    "prob_jb_ols": float(prob_jb_seccion_21),
    "skew_ols": float(skew_seccion_21),
    "kurtosis_ols": float(kurtosis_seccion_21)
})
coeficientes_ols_seccion_21 = pd.DataFrame({"variable": resultados_ols_seccion_21.params.index, "coeficiente": resultados_ols_seccion_21.params.to_numpy(), "p_value": resultados_ols_seccion_21.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_21.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_21 = pd.DataFrame({"periodo": df_test_seccion_21["periodo"].to_numpy(), "real": y_test_seccion_21.to_numpy(), "prediccion": predicciones_test_seccion_21.to_numpy(), "residuo": y_test_seccion_21.to_numpy() - predicciones_test_seccion_21.to_numpy()})
comparacion_ols_seccion_21 = pd.DataFrame([{"modelo": "OLS sección 20", **metricas_ols_seccion_20}, {"modelo": "OLS sección 21", **metricas_ols_seccion_21}])
delta_metricas_seccion_21 = {f"delta_{metrica}": metricas_ols_seccion_21[metrica] - metricas_ols_seccion_20[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_21[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_21"):
    mlflow.set_tag("run_anterior_id", "79d57bfdd7254954a9639304df0e39c3")
    mlflow.log_params({"seccion": 21, "dataset": str(RUTA_SALIDA_SECCION_21), "dataset_origen": str(RUTA_ORIGEN_SECCION_21), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_21), "n_predictores": X_train_seccion_21.shape[1], "filas_entrenamiento": len(df_train_seccion_21), "filas_prueba": len(df_test_seccion_21), "tipo_modelo": "statsmodels_OLS_seccion_21_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_21, **delta_metricas_seccion_21})
    mlflow.log_table(comparacion_ols_seccion_21, artifact_file="comparacion/comparacion_ols_seccion_21.json")
    mlflow.log_table(coeficientes_ols_seccion_21, artifact_file="modelo/coeficientes_ols_seccion_21.json")
    mlflow.log_table(residuos_test_seccion_21, artifact_file="diagnostico/residuos_test_seccion_21.json")
    mlflow.log_text(resultados_ols_seccion_21.summary().as_text(), "modelo/resumen_ols_seccion_21.txt")
print(f"Dataset de la sección 21 guardado en: {RUTA_SALIDA_SECCION_21}")
print("OLS de la sección 21 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 20,37076.824980,45126.757949,0.859864,23.166935
1,OLS sección 21,37075.255959,45125.466604,0.859873,23.166044


🏃 View run ols_temporal_seccion_21 at: http://127.0.0.1:5000/#/experiments/4/runs/d8f58fb11b2a48bc9cada57b65055a87
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 21 guardado en: data_sets_transformados\secondary_market_modelado_seccion_21.csv
OLS de la sección 21 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las tres candidatas se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 20.

## 22. OLS sin tres candidatas de la sección 8

Partimos del dataset acumulativo de la sección 21 y eliminamos `building_age`, `district_Zyuzino` y `metro_station_Molodyozhnaya`.

In [22]:
VARIABLES_ELIMINADAS_SECCION_22 = [
    "building_age",
    "district_Zyuzino",
    "metro_station_Molodyozhnaya"
]
RUTA_ORIGEN_SECCION_22 = Path("data_sets_transformados/secondary_market_modelado_seccion_21.csv")
RUTA_SALIDA_SECCION_22 = Path("data_sets_transformados/secondary_market_modelado_seccion_22.csv")
df_ols_seccion_22_origen = pd.read_csv(RUTA_ORIGEN_SECCION_22)
columnas_faltantes_seccion_22 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_22 if columna not in df_ols_seccion_22_origen.columns]
if columnas_faltantes_seccion_22:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_22}")
df_ols_seccion_22 = df_ols_seccion_22_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_22).copy()
df_ols_seccion_22.to_csv(RUTA_SALIDA_SECCION_22, index=False)
df_train_seccion_22 = df_ols_seccion_22[df_ols_seccion_22["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_22 = df_ols_seccion_22[df_ols_seccion_22["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_22 = df_train_seccion_22.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_22 = df_test_seccion_22.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_22 = df_train_seccion_22["price_per_sqm"]
y_test_seccion_22 = df_test_seccion_22["price_per_sqm"]
X_train_seccion_22_sm = sm.add_constant(X_train_seccion_22, has_constant="add")
X_test_seccion_22_sm = sm.add_constant(X_test_seccion_22, has_constant="add")
resultados_ols_seccion_22 = sm.OLS(y_train_seccion_22, X_train_seccion_22_sm).fit()
predicciones_train_seccion_22 = resultados_ols_seccion_22.predict(X_train_seccion_22_sm)
predicciones_test_seccion_22 = resultados_ols_seccion_22.predict(X_test_seccion_22_sm)
metricas_ols_seccion_22 = {
    **calcular_metricas_regresion(y_train_seccion_22, predicciones_train_seccion_22, "train"),
    **calcular_metricas_regresion(y_test_seccion_22, predicciones_test_seccion_22, "test"),
    "r2_ols": float(resultados_ols_seccion_22.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_22.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_22.aic),
    "bic_ols": float(resultados_ols_seccion_22.bic),
    "f_statistic_ols": float(resultados_ols_seccion_22.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_22.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_22.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_22.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_22.pvalues > 0.05).sum())
}
omnibus_seccion_22, prob_omnibus_seccion_22 = omni_normtest(resultados_ols_seccion_22.resid)
jarque_bera_seccion_22, prob_jb_seccion_22, skew_seccion_22, kurtosis_seccion_22 = jarque_bera(resultados_ols_seccion_22.resid)
metricas_ols_seccion_22.update({
    "omnibus_ols": float(omnibus_seccion_22),
    "prob_omnibus_ols": float(prob_omnibus_seccion_22),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_22.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_22),
    "prob_jb_ols": float(prob_jb_seccion_22),
    "skew_ols": float(skew_seccion_22),
    "kurtosis_ols": float(kurtosis_seccion_22)
})
coeficientes_ols_seccion_22 = pd.DataFrame({"variable": resultados_ols_seccion_22.params.index, "coeficiente": resultados_ols_seccion_22.params.to_numpy(), "p_value": resultados_ols_seccion_22.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_22.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_22 = pd.DataFrame({"periodo": df_test_seccion_22["periodo"].to_numpy(), "real": y_test_seccion_22.to_numpy(), "prediccion": predicciones_test_seccion_22.to_numpy(), "residuo": y_test_seccion_22.to_numpy() - predicciones_test_seccion_22.to_numpy()})
comparacion_ols_seccion_22 = pd.DataFrame([{"modelo": "OLS sección 21", **metricas_ols_seccion_21}, {"modelo": "OLS sección 22", **metricas_ols_seccion_22}])
delta_metricas_seccion_22 = {f"delta_{metrica}": metricas_ols_seccion_22[metrica] - metricas_ols_seccion_21[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_22[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_22"):
    mlflow.set_tag("run_anterior_id", "123db2e67e9c4bcfb7c8736776d6ff57")
    mlflow.log_params({"seccion": 22, "dataset": str(RUTA_SALIDA_SECCION_22), "dataset_origen": str(RUTA_ORIGEN_SECCION_22), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_22), "n_predictores": X_train_seccion_22.shape[1], "filas_entrenamiento": len(df_train_seccion_22), "filas_prueba": len(df_test_seccion_22), "tipo_modelo": "statsmodels_OLS_seccion_22_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_22, **delta_metricas_seccion_22})
    mlflow.log_table(comparacion_ols_seccion_22, artifact_file="comparacion/comparacion_ols_seccion_22.json")
    mlflow.log_table(coeficientes_ols_seccion_22, artifact_file="modelo/coeficientes_ols_seccion_22.json")
    mlflow.log_table(residuos_test_seccion_22, artifact_file="diagnostico/residuos_test_seccion_22.json")
    mlflow.log_text(resultados_ols_seccion_22.summary().as_text(), "modelo/resumen_ols_seccion_22.txt")
print(f"Dataset de la sección 22 guardado en: {RUTA_SALIDA_SECCION_22}")
print("OLS de la sección 22 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 21,37075.255959,45125.466604,0.859873,23.166044
1,OLS sección 22,37073.634354,45123.905470,0.859882,23.165532


🏃 View run ols_temporal_seccion_22 at: http://127.0.0.1:5000/#/experiments/4/runs/65eec57ee34c4c1eb966c495cce3eafd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 22 guardado en: data_sets_transformados\secondary_market_modelado_seccion_22.csv
OLS de la sección 22 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las tres candidatas se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 21.

## 23. OLS sin dos candidatas de la sección 8

Partimos del dataset acumulativo de la sección 22 y eliminamos `metro_station_Teatralnaya` y `metro_station_Smolenskaya (AP)`.

In [23]:
VARIABLES_ELIMINADAS_SECCION_23 = [
    "metro_station_Teatralnaya",
    "metro_station_Smolenskaya (AP)"
]
RUTA_ORIGEN_SECCION_23 = Path("data_sets_transformados/secondary_market_modelado_seccion_22.csv")
RUTA_SALIDA_SECCION_23 = Path("data_sets_transformados/secondary_market_modelado_seccion_23.csv")
df_ols_seccion_23_origen = pd.read_csv(RUTA_ORIGEN_SECCION_23)
columnas_faltantes_seccion_23 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_23 if columna not in df_ols_seccion_23_origen.columns]
if columnas_faltantes_seccion_23:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_23}")
df_ols_seccion_23 = df_ols_seccion_23_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_23).copy()
df_ols_seccion_23.to_csv(RUTA_SALIDA_SECCION_23, index=False)
df_train_seccion_23 = df_ols_seccion_23[df_ols_seccion_23["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_23 = df_ols_seccion_23[df_ols_seccion_23["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_23 = df_train_seccion_23.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_23 = df_test_seccion_23.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_23 = df_train_seccion_23["price_per_sqm"]
y_test_seccion_23 = df_test_seccion_23["price_per_sqm"]
X_train_seccion_23_sm = sm.add_constant(X_train_seccion_23, has_constant="add")
X_test_seccion_23_sm = sm.add_constant(X_test_seccion_23, has_constant="add")
resultados_ols_seccion_23 = sm.OLS(y_train_seccion_23, X_train_seccion_23_sm).fit()
predicciones_train_seccion_23 = resultados_ols_seccion_23.predict(X_train_seccion_23_sm)
predicciones_test_seccion_23 = resultados_ols_seccion_23.predict(X_test_seccion_23_sm)
metricas_ols_seccion_23 = {
    **calcular_metricas_regresion(y_train_seccion_23, predicciones_train_seccion_23, "train"),
    **calcular_metricas_regresion(y_test_seccion_23, predicciones_test_seccion_23, "test"),
    "r2_ols": float(resultados_ols_seccion_23.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_23.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_23.aic),
    "bic_ols": float(resultados_ols_seccion_23.bic),
    "f_statistic_ols": float(resultados_ols_seccion_23.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_23.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_23.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_23.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_23.pvalues > 0.05).sum())
}
omnibus_seccion_23, prob_omnibus_seccion_23 = omni_normtest(resultados_ols_seccion_23.resid)
jarque_bera_seccion_23, prob_jb_seccion_23, skew_seccion_23, kurtosis_seccion_23 = jarque_bera(resultados_ols_seccion_23.resid)
metricas_ols_seccion_23.update({
    "omnibus_ols": float(omnibus_seccion_23),
    "prob_omnibus_ols": float(prob_omnibus_seccion_23),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_23.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_23),
    "prob_jb_ols": float(prob_jb_seccion_23),
    "skew_ols": float(skew_seccion_23),
    "kurtosis_ols": float(kurtosis_seccion_23)
})
coeficientes_ols_seccion_23 = pd.DataFrame({"variable": resultados_ols_seccion_23.params.index, "coeficiente": resultados_ols_seccion_23.params.to_numpy(), "p_value": resultados_ols_seccion_23.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_23.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_23 = pd.DataFrame({"periodo": df_test_seccion_23["periodo"].to_numpy(), "real": y_test_seccion_23.to_numpy(), "prediccion": predicciones_test_seccion_23.to_numpy(), "residuo": y_test_seccion_23.to_numpy() - predicciones_test_seccion_23.to_numpy()})
comparacion_ols_seccion_23 = pd.DataFrame([{"modelo": "OLS sección 22", **metricas_ols_seccion_22}, {"modelo": "OLS sección 23", **metricas_ols_seccion_23}])
delta_metricas_seccion_23 = {f"delta_{metrica}": metricas_ols_seccion_23[metrica] - metricas_ols_seccion_22[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_23[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_23"):
    mlflow.set_tag("run_anterior_id", "12dea48cdde64f178bfd00a2e111760e")
    mlflow.log_params({"seccion": 23, "dataset": str(RUTA_SALIDA_SECCION_23), "dataset_origen": str(RUTA_ORIGEN_SECCION_23), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_23), "n_predictores": X_train_seccion_23.shape[1], "filas_entrenamiento": len(df_train_seccion_23), "filas_prueba": len(df_test_seccion_23), "tipo_modelo": "statsmodels_OLS_seccion_23_acumulativo"})
    mlflow.log_metrics({**metricas_ols_seccion_23, **delta_metricas_seccion_23})
    mlflow.log_table(comparacion_ols_seccion_23, artifact_file="comparacion/comparacion_ols_seccion_23.json")
    mlflow.log_table(coeficientes_ols_seccion_23, artifact_file="modelo/coeficientes_ols_seccion_23.json")
    mlflow.log_table(residuos_test_seccion_23, artifact_file="diagnostico/residuos_test_seccion_23.json")
    mlflow.log_text(resultados_ols_seccion_23.summary().as_text(), "modelo/resumen_ols_seccion_23.txt")
print(f"Dataset de la sección 23 guardado en: {RUTA_SALIDA_SECCION_23}")
print("OLS de la sección 23 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 22,37073.634354,45123.905470,0.859882,23.165532
1,OLS sección 23,37074.438694,45126.590672,0.859866,23.166574


🏃 View run ols_temporal_seccion_23 at: http://127.0.0.1:5000/#/experiments/4/runs/c7a8ad9a33b24428be3f616686cec0ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 23 guardado en: data_sets_transformados\secondary_market_modelado_seccion_23.csv
OLS de la sección 23 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las dos candidatas se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 22.

## 24. OLS sin `metro_station_Teatralnaya` desde la sección 22

Partimos directamente del dataset acumulativo de la sección 22 y eliminamos únicamente `metro_station_Teatralnaya`.

In [24]:
VARIABLES_ELIMINADAS_SECCION_24 = ["metro_station_Teatralnaya"]
RUTA_ORIGEN_SECCION_24 = Path("data_sets_transformados/secondary_market_modelado_seccion_22.csv")
RUTA_SALIDA_SECCION_24 = Path("data_sets_transformados/secondary_market_modelado_seccion_24.csv")
df_ols_seccion_24_origen = pd.read_csv(RUTA_ORIGEN_SECCION_24)
columnas_faltantes_seccion_24 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_24 if columna not in df_ols_seccion_24_origen.columns]
if columnas_faltantes_seccion_24:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_24}")
df_ols_seccion_24 = df_ols_seccion_24_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_24).copy()
df_ols_seccion_24.to_csv(RUTA_SALIDA_SECCION_24, index=False)
df_train_seccion_24 = df_ols_seccion_24[df_ols_seccion_24["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_24 = df_ols_seccion_24[df_ols_seccion_24["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_24 = df_train_seccion_24.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_24 = df_test_seccion_24.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_24 = df_train_seccion_24["price_per_sqm"]
y_test_seccion_24 = df_test_seccion_24["price_per_sqm"]
X_train_seccion_24_sm = sm.add_constant(X_train_seccion_24, has_constant="add")
X_test_seccion_24_sm = sm.add_constant(X_test_seccion_24, has_constant="add")
resultados_ols_seccion_24 = sm.OLS(y_train_seccion_24, X_train_seccion_24_sm).fit()
predicciones_train_seccion_24 = resultados_ols_seccion_24.predict(X_train_seccion_24_sm)
predicciones_test_seccion_24 = resultados_ols_seccion_24.predict(X_test_seccion_24_sm)
metricas_ols_seccion_24 = {
    **calcular_metricas_regresion(y_train_seccion_24, predicciones_train_seccion_24, "train"),
    **calcular_metricas_regresion(y_test_seccion_24, predicciones_test_seccion_24, "test"),
    "r2_ols": float(resultados_ols_seccion_24.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_24.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_24.aic),
    "bic_ols": float(resultados_ols_seccion_24.bic),
    "f_statistic_ols": float(resultados_ols_seccion_24.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_24.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_24.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_24.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_24.pvalues > 0.05).sum())
}
omnibus_seccion_24, prob_omnibus_seccion_24 = omni_normtest(resultados_ols_seccion_24.resid)
jarque_bera_seccion_24, prob_jb_seccion_24, skew_seccion_24, kurtosis_seccion_24 = jarque_bera(resultados_ols_seccion_24.resid)
metricas_ols_seccion_24.update({
    "omnibus_ols": float(omnibus_seccion_24),
    "prob_omnibus_ols": float(prob_omnibus_seccion_24),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_24.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_24),
    "prob_jb_ols": float(prob_jb_seccion_24),
    "skew_ols": float(skew_seccion_24),
    "kurtosis_ols": float(kurtosis_seccion_24)
})
coeficientes_ols_seccion_24 = pd.DataFrame({"variable": resultados_ols_seccion_24.params.index, "coeficiente": resultados_ols_seccion_24.params.to_numpy(), "p_value": resultados_ols_seccion_24.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_24.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_24 = pd.DataFrame({"periodo": df_test_seccion_24["periodo"].to_numpy(), "real": y_test_seccion_24.to_numpy(), "prediccion": predicciones_test_seccion_24.to_numpy(), "residuo": y_test_seccion_24.to_numpy() - predicciones_test_seccion_24.to_numpy()})
comparacion_ols_seccion_24 = pd.DataFrame([{"modelo": "OLS sección 22", **metricas_ols_seccion_22}, {"modelo": "OLS sección 24", **metricas_ols_seccion_24}])
delta_metricas_seccion_24 = {f"delta_{metrica}": metricas_ols_seccion_24[metrica] - metricas_ols_seccion_22[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_24[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_24"):
    mlflow.set_tag("run_anterior_id", "12dea48cdde64f178bfd00a2e111760e")
    mlflow.log_params({"seccion": 24, "dataset": str(RUTA_SALIDA_SECCION_24), "dataset_origen": str(RUTA_ORIGEN_SECCION_24), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_24), "n_predictores": X_train_seccion_24.shape[1], "filas_entrenamiento": len(df_train_seccion_24), "filas_prueba": len(df_test_seccion_24), "tipo_modelo": "statsmodels_OLS_seccion_24_desde_seccion_22"})
    mlflow.log_metrics({**metricas_ols_seccion_24, **delta_metricas_seccion_24})
    mlflow.log_table(comparacion_ols_seccion_24, artifact_file="comparacion/comparacion_ols_seccion_24.json")
    mlflow.log_table(coeficientes_ols_seccion_24, artifact_file="modelo/coeficientes_ols_seccion_24.json")
    mlflow.log_table(residuos_test_seccion_24, artifact_file="diagnostico/residuos_test_seccion_24.json")
    mlflow.log_text(resultados_ols_seccion_24.summary().as_text(), "modelo/resumen_ols_seccion_24.txt")
print(f"Dataset de la sección 24 guardado en: {RUTA_SALIDA_SECCION_24}")
print("OLS de la sección 24 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 22,37073.634354,45123.905470,0.859882,23.165532
1,OLS sección 24,37075.470169,45125.892866,0.859870,23.167252


🏃 View run ols_temporal_seccion_24 at: http://127.0.0.1:5000/#/experiments/4/runs/aa5cbfa6cc1c4b20a5d0fda50c064b01
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 24 guardado en: data_sets_transformados\secondary_market_modelado_seccion_24.csv
OLS de la sección 24 registrado en MLflow para comparación


### Interpretación

La eliminación de `metro_station_Teatralnaya` se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 22.

## 25. OLS sin tres candidatas de la sección 8

Partimos del dataset acumulativo de la sección 24 y eliminamos `metro_station_Smolenskaya (AP)`, `metro_station_Mayakovskaya` y `metro_station_Dubrovka`.

In [25]:
VARIABLES_ELIMINADAS_SECCION_25 = [
    "metro_station_Smolenskaya (AP)",
    "metro_station_Mayakovskaya",
    "metro_station_Dubrovka"
]
RUTA_ORIGEN_SECCION_25 = Path("data_sets_transformados/secondary_market_modelado_seccion_24.csv")
RUTA_SALIDA_SECCION_25 = Path("data_sets_transformados/secondary_market_modelado_seccion_25.csv")
df_ols_seccion_25_origen = pd.read_csv(RUTA_ORIGEN_SECCION_25)
columnas_faltantes_seccion_25 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_25 if columna not in df_ols_seccion_25_origen.columns]
if columnas_faltantes_seccion_25:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_25}")
df_ols_seccion_25 = df_ols_seccion_25_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_25).copy()
df_ols_seccion_25.to_csv(RUTA_SALIDA_SECCION_25, index=False)
df_train_seccion_25 = df_ols_seccion_25[df_ols_seccion_25["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_25 = df_ols_seccion_25[df_ols_seccion_25["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_25 = df_train_seccion_25.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_25 = df_test_seccion_25.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_25 = df_train_seccion_25["price_per_sqm"]
y_test_seccion_25 = df_test_seccion_25["price_per_sqm"]
X_train_seccion_25_sm = sm.add_constant(X_train_seccion_25, has_constant="add")
X_test_seccion_25_sm = sm.add_constant(X_test_seccion_25, has_constant="add")
resultados_ols_seccion_25 = sm.OLS(y_train_seccion_25, X_train_seccion_25_sm).fit()
predicciones_train_seccion_25 = resultados_ols_seccion_25.predict(X_train_seccion_25_sm)
predicciones_test_seccion_25 = resultados_ols_seccion_25.predict(X_test_seccion_25_sm)
metricas_ols_seccion_25 = {
    **calcular_metricas_regresion(y_train_seccion_25, predicciones_train_seccion_25, "train"),
    **calcular_metricas_regresion(y_test_seccion_25, predicciones_test_seccion_25, "test"),
    "r2_ols": float(resultados_ols_seccion_25.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_25.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_25.aic),
    "bic_ols": float(resultados_ols_seccion_25.bic),
    "f_statistic_ols": float(resultados_ols_seccion_25.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_25.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_25.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_25.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_25.pvalues > 0.05).sum())
}
omnibus_seccion_25, prob_omnibus_seccion_25 = omni_normtest(resultados_ols_seccion_25.resid)
jarque_bera_seccion_25, prob_jb_seccion_25, skew_seccion_25, kurtosis_seccion_25 = jarque_bera(resultados_ols_seccion_25.resid)
metricas_ols_seccion_25.update({
    "omnibus_ols": float(omnibus_seccion_25),
    "prob_omnibus_ols": float(prob_omnibus_seccion_25),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_25.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_25),
    "prob_jb_ols": float(prob_jb_seccion_25),
    "skew_ols": float(skew_seccion_25),
    "kurtosis_ols": float(kurtosis_seccion_25)
})
coeficientes_ols_seccion_25 = pd.DataFrame({"variable": resultados_ols_seccion_25.params.index, "coeficiente": resultados_ols_seccion_25.params.to_numpy(), "p_value": resultados_ols_seccion_25.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_25.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_25 = pd.DataFrame({"periodo": df_test_seccion_25["periodo"].to_numpy(), "real": y_test_seccion_25.to_numpy(), "prediccion": predicciones_test_seccion_25.to_numpy(), "residuo": y_test_seccion_25.to_numpy() - predicciones_test_seccion_25.to_numpy()})
comparacion_ols_seccion_25 = pd.DataFrame([{"modelo": "OLS sección 24", **metricas_ols_seccion_24}, {"modelo": "OLS sección 25", **metricas_ols_seccion_25}])
delta_metricas_seccion_25 = {f"delta_{metrica}": metricas_ols_seccion_25[metrica] - metricas_ols_seccion_24[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_25[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_25"):
    mlflow.set_tag("run_anterior_id", "0233e7b41eb24bbebbe1802409ffa833")
    mlflow.log_params({"seccion": 25, "dataset": str(RUTA_SALIDA_SECCION_25), "dataset_origen": str(RUTA_ORIGEN_SECCION_25), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_25), "n_predictores": X_train_seccion_25.shape[1], "filas_entrenamiento": len(df_train_seccion_25), "filas_prueba": len(df_test_seccion_25), "tipo_modelo": "statsmodels_OLS_seccion_25_desde_seccion_24"})
    mlflow.log_metrics({**metricas_ols_seccion_25, **delta_metricas_seccion_25})
    mlflow.log_table(comparacion_ols_seccion_25, artifact_file="comparacion/comparacion_ols_seccion_25.json")
    mlflow.log_table(coeficientes_ols_seccion_25, artifact_file="modelo/coeficientes_ols_seccion_25.json")
    mlflow.log_table(residuos_test_seccion_25, artifact_file="diagnostico/residuos_test_seccion_25.json")
    mlflow.log_text(resultados_ols_seccion_25.summary().as_text(), "modelo/resumen_ols_seccion_25.txt")
print(f"Dataset de la sección 25 guardado en: {RUTA_SALIDA_SECCION_25}")
print("OLS de la sección 25 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 24,37075.470169,45125.892866,0.859870,23.167252
1,OLS sección 25,37072.694776,45125.764307,0.859871,23.165666


🏃 View run ols_temporal_seccion_25 at: http://127.0.0.1:5000/#/experiments/4/runs/e405b69c92dd4ec69dfab06e62ef7071
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 25 guardado en: data_sets_transformados\secondary_market_modelado_seccion_25.csv
OLS de la sección 25 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las tres candidatas se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 24.

## 26. OLS sin `total_area` desde la sección 25

Partimos del dataset acumulativo de la sección 25 y eliminamos únicamente `total_area`.

In [26]:
VARIABLES_ELIMINADAS_SECCION_26 = ["total_area"]
RUTA_ORIGEN_SECCION_26 = Path("data_sets_transformados/secondary_market_modelado_seccion_25.csv")
RUTA_SALIDA_SECCION_26 = Path("data_sets_transformados/secondary_market_modelado_seccion_26.csv")
df_ols_seccion_26_origen = pd.read_csv(RUTA_ORIGEN_SECCION_26)
columnas_faltantes_seccion_26 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_26 if columna not in df_ols_seccion_26_origen.columns]
if columnas_faltantes_seccion_26:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_26}")
df_ols_seccion_26 = df_ols_seccion_26_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_26).copy()
df_ols_seccion_26.to_csv(RUTA_SALIDA_SECCION_26, index=False)
df_train_seccion_26 = df_ols_seccion_26[df_ols_seccion_26["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_26 = df_ols_seccion_26[df_ols_seccion_26["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_26 = df_train_seccion_26.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_26 = df_test_seccion_26.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_26 = df_train_seccion_26["price_per_sqm"]
y_test_seccion_26 = df_test_seccion_26["price_per_sqm"]
X_train_seccion_26_sm = sm.add_constant(X_train_seccion_26, has_constant="add")
X_test_seccion_26_sm = sm.add_constant(X_test_seccion_26, has_constant="add")
resultados_ols_seccion_26 = sm.OLS(y_train_seccion_26, X_train_seccion_26_sm).fit()
predicciones_train_seccion_26 = resultados_ols_seccion_26.predict(X_train_seccion_26_sm)
predicciones_test_seccion_26 = resultados_ols_seccion_26.predict(X_test_seccion_26_sm)
metricas_ols_seccion_26 = {
    **calcular_metricas_regresion(y_train_seccion_26, predicciones_train_seccion_26, "train"),
    **calcular_metricas_regresion(y_test_seccion_26, predicciones_test_seccion_26, "test"),
    "r2_ols": float(resultados_ols_seccion_26.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_26.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_26.aic),
    "bic_ols": float(resultados_ols_seccion_26.bic),
    "f_statistic_ols": float(resultados_ols_seccion_26.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_26.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_26.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_26.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_26.pvalues > 0.05).sum())
}
omnibus_seccion_26, prob_omnibus_seccion_26 = omni_normtest(resultados_ols_seccion_26.resid)
jarque_bera_seccion_26, prob_jb_seccion_26, skew_seccion_26, kurtosis_seccion_26 = jarque_bera(resultados_ols_seccion_26.resid)
metricas_ols_seccion_26.update({
    "omnibus_ols": float(omnibus_seccion_26),
    "prob_omnibus_ols": float(prob_omnibus_seccion_26),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_26.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_26),
    "prob_jb_ols": float(prob_jb_seccion_26),
    "skew_ols": float(skew_seccion_26),
    "kurtosis_ols": float(kurtosis_seccion_26)
})
coeficientes_ols_seccion_26 = pd.DataFrame({"variable": resultados_ols_seccion_26.params.index, "coeficiente": resultados_ols_seccion_26.params.to_numpy(), "p_value": resultados_ols_seccion_26.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_26.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_26 = pd.DataFrame({"periodo": df_test_seccion_26["periodo"].to_numpy(), "real": y_test_seccion_26.to_numpy(), "prediccion": predicciones_test_seccion_26.to_numpy(), "residuo": y_test_seccion_26.to_numpy() - predicciones_test_seccion_26.to_numpy()})
comparacion_ols_seccion_26 = pd.DataFrame([{"modelo": "OLS sección 25", **metricas_ols_seccion_25}, {"modelo": "OLS sección 26", **metricas_ols_seccion_26}])
delta_metricas_seccion_26 = {f"delta_{metrica}": metricas_ols_seccion_26[metrica] - metricas_ols_seccion_25[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_26[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_26"):
    mlflow.set_tag("run_anterior_id", "b9816ee7a5ed4b628d6dddb768a4cfec")
    mlflow.log_params({"seccion": 26, "dataset": str(RUTA_SALIDA_SECCION_26), "dataset_origen": str(RUTA_ORIGEN_SECCION_26), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_26), "n_predictores": X_train_seccion_26.shape[1], "filas_entrenamiento": len(df_train_seccion_26), "filas_prueba": len(df_test_seccion_26), "tipo_modelo": "statsmodels_OLS_seccion_26_desde_seccion_25"})
    mlflow.log_metrics({**metricas_ols_seccion_26, **delta_metricas_seccion_26})
    mlflow.log_table(comparacion_ols_seccion_26, artifact_file="comparacion/comparacion_ols_seccion_26.json")
    mlflow.log_table(coeficientes_ols_seccion_26, artifact_file="modelo/coeficientes_ols_seccion_26.json")
    mlflow.log_table(residuos_test_seccion_26, artifact_file="diagnostico/residuos_test_seccion_26.json")
    mlflow.log_text(resultados_ols_seccion_26.summary().as_text(), "modelo/resumen_ols_seccion_26.txt")
print(f"Dataset de la sección 26 guardado en: {RUTA_SALIDA_SECCION_26}")
print("OLS de la sección 26 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 25,37072.694776,45125.764307,0.859871,23.165666
1,OLS sección 26,37072.523484,45125.073874,0.859875,23.165776


🏃 View run ols_temporal_seccion_26 at: http://127.0.0.1:5000/#/experiments/4/runs/8997b0e5202949df9aad478e0fb9ef27
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 26 guardado en: data_sets_transformados\secondary_market_modelado_seccion_26.csv
OLS de la sección 26 registrado en MLflow para comparación


### Interpretación

La eliminación de `total_area` se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 25.

## 27. OLS sin `total_area` desde la sección 22

Partimos directamente del dataset acumulativo de la sección 22 y eliminamos únicamente `total_area`.

In [27]:
VARIABLES_ELIMINADAS_SECCION_27 = ["total_area"]
RUTA_ORIGEN_SECCION_27 = Path("data_sets_transformados/secondary_market_modelado_seccion_22.csv")
RUTA_SALIDA_SECCION_27 = Path("data_sets_transformados/secondary_market_modelado_seccion_27.csv")
df_ols_seccion_27_origen = pd.read_csv(RUTA_ORIGEN_SECCION_27)
columnas_faltantes_seccion_27 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_27 if columna not in df_ols_seccion_27_origen.columns]
if columnas_faltantes_seccion_27:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_27}")
df_ols_seccion_27 = df_ols_seccion_27_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_27).copy()
df_ols_seccion_27.to_csv(RUTA_SALIDA_SECCION_27, index=False)
df_train_seccion_27 = df_ols_seccion_27[df_ols_seccion_27["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_27 = df_ols_seccion_27[df_ols_seccion_27["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_27 = df_train_seccion_27.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_27 = df_test_seccion_27.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_27 = df_train_seccion_27["price_per_sqm"]
y_test_seccion_27 = df_test_seccion_27["price_per_sqm"]
X_train_seccion_27_sm = sm.add_constant(X_train_seccion_27, has_constant="add")
X_test_seccion_27_sm = sm.add_constant(X_test_seccion_27, has_constant="add")
resultados_ols_seccion_27 = sm.OLS(y_train_seccion_27, X_train_seccion_27_sm).fit()
predicciones_train_seccion_27 = resultados_ols_seccion_27.predict(X_train_seccion_27_sm)
predicciones_test_seccion_27 = resultados_ols_seccion_27.predict(X_test_seccion_27_sm)
metricas_ols_seccion_27 = {
    **calcular_metricas_regresion(y_train_seccion_27, predicciones_train_seccion_27, "train"),
    **calcular_metricas_regresion(y_test_seccion_27, predicciones_test_seccion_27, "test"),
    "r2_ols": float(resultados_ols_seccion_27.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_27.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_27.aic),
    "bic_ols": float(resultados_ols_seccion_27.bic),
    "f_statistic_ols": float(resultados_ols_seccion_27.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_27.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_27.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_27.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_27.pvalues > 0.05).sum())
}
omnibus_seccion_27, prob_omnibus_seccion_27 = omni_normtest(resultados_ols_seccion_27.resid)
jarque_bera_seccion_27, prob_jb_seccion_27, skew_seccion_27, kurtosis_seccion_27 = jarque_bera(resultados_ols_seccion_27.resid)
metricas_ols_seccion_27.update({
    "omnibus_ols": float(omnibus_seccion_27),
    "prob_omnibus_ols": float(prob_omnibus_seccion_27),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_27.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_27),
    "prob_jb_ols": float(prob_jb_seccion_27),
    "skew_ols": float(skew_seccion_27),
    "kurtosis_ols": float(kurtosis_seccion_27)
})
coeficientes_ols_seccion_27 = pd.DataFrame({"variable": resultados_ols_seccion_27.params.index, "coeficiente": resultados_ols_seccion_27.params.to_numpy(), "p_value": resultados_ols_seccion_27.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_27.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_27 = pd.DataFrame({"periodo": df_test_seccion_27["periodo"].to_numpy(), "real": y_test_seccion_27.to_numpy(), "prediccion": predicciones_test_seccion_27.to_numpy(), "residuo": y_test_seccion_27.to_numpy() - predicciones_test_seccion_27.to_numpy()})
comparacion_ols_seccion_27 = pd.DataFrame([{"modelo": "OLS sección 22", **metricas_ols_seccion_22}, {"modelo": "OLS sección 27", **metricas_ols_seccion_27}])
delta_metricas_seccion_27 = {f"delta_{metrica}": metricas_ols_seccion_27[metrica] - metricas_ols_seccion_22[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_27[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_27"):
    mlflow.set_tag("run_anterior_id", "12dea48cdde64f178bfd00a2e111760e")
    mlflow.log_params({"seccion": 27, "dataset": str(RUTA_SALIDA_SECCION_27), "dataset_origen": str(RUTA_ORIGEN_SECCION_27), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_27), "n_predictores": X_train_seccion_27.shape[1], "filas_entrenamiento": len(df_train_seccion_27), "filas_prueba": len(df_test_seccion_27), "tipo_modelo": "statsmodels_OLS_seccion_27_desde_seccion_22"})
    mlflow.log_metrics({**metricas_ols_seccion_27, **delta_metricas_seccion_27})
    mlflow.log_table(comparacion_ols_seccion_27, artifact_file="comparacion/comparacion_ols_seccion_27.json")
    mlflow.log_table(coeficientes_ols_seccion_27, artifact_file="modelo/coeficientes_ols_seccion_27.json")
    mlflow.log_table(residuos_test_seccion_27, artifact_file="diagnostico/residuos_test_seccion_27.json")
    mlflow.log_text(resultados_ols_seccion_27.summary().as_text(), "modelo/resumen_ols_seccion_27.txt")
print(f"Dataset de la sección 27 guardado en: {RUTA_SALIDA_SECCION_27}")
print("OLS de la sección 27 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 22,37073.634354,45123.905470,0.859882,23.165532
1,OLS sección 27,37073.461485,45123.234408,0.859886,23.165613


🏃 View run ols_temporal_seccion_27 at: http://127.0.0.1:5000/#/experiments/4/runs/7bbe48e4ece84fb4be52eb8731dd8c20
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 27 guardado en: data_sets_transformados\secondary_market_modelado_seccion_27.csv
OLS de la sección 27 registrado en MLflow para comparación


### Interpretación

La eliminación de `total_area` se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 22.

## 28. OLS sin `rooms` desde la sección 27

Partimos del dataset acumulativo de la sección 27 y eliminamos únicamente `rooms`.

In [28]:
VARIABLES_ELIMINADAS_SECCION_28 = ["rooms"]
RUTA_ORIGEN_SECCION_28 = Path("data_sets_transformados/secondary_market_modelado_seccion_27.csv")
RUTA_SALIDA_SECCION_28 = Path("data_sets_transformados/secondary_market_modelado_seccion_28.csv")
df_ols_seccion_28_origen = pd.read_csv(RUTA_ORIGEN_SECCION_28)
columnas_faltantes_seccion_28 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_28 if columna not in df_ols_seccion_28_origen.columns]
if columnas_faltantes_seccion_28:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_28}")
df_ols_seccion_28 = df_ols_seccion_28_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_28).copy()
df_ols_seccion_28.to_csv(RUTA_SALIDA_SECCION_28, index=False)
df_train_seccion_28 = df_ols_seccion_28[df_ols_seccion_28["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_28 = df_ols_seccion_28[df_ols_seccion_28["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_28 = df_train_seccion_28.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_28 = df_test_seccion_28.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_28 = df_train_seccion_28["price_per_sqm"]
y_test_seccion_28 = df_test_seccion_28["price_per_sqm"]
X_train_seccion_28_sm = sm.add_constant(X_train_seccion_28, has_constant="add")
X_test_seccion_28_sm = sm.add_constant(X_test_seccion_28, has_constant="add")
resultados_ols_seccion_28 = sm.OLS(y_train_seccion_28, X_train_seccion_28_sm).fit()
predicciones_train_seccion_28 = resultados_ols_seccion_28.predict(X_train_seccion_28_sm)
predicciones_test_seccion_28 = resultados_ols_seccion_28.predict(X_test_seccion_28_sm)
metricas_ols_seccion_28 = {
    **calcular_metricas_regresion(y_train_seccion_28, predicciones_train_seccion_28, "train"),
    **calcular_metricas_regresion(y_test_seccion_28, predicciones_test_seccion_28, "test"),
    "r2_ols": float(resultados_ols_seccion_28.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_28.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_28.aic),
    "bic_ols": float(resultados_ols_seccion_28.bic),
    "f_statistic_ols": float(resultados_ols_seccion_28.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_28.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_28.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_28.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_28.pvalues > 0.05).sum())
}
omnibus_seccion_28, prob_omnibus_seccion_28 = omni_normtest(resultados_ols_seccion_28.resid)
jarque_bera_seccion_28, prob_jb_seccion_28, skew_seccion_28, kurtosis_seccion_28 = jarque_bera(resultados_ols_seccion_28.resid)
metricas_ols_seccion_28.update({
    "omnibus_ols": float(omnibus_seccion_28),
    "prob_omnibus_ols": float(prob_omnibus_seccion_28),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_28.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_28),
    "prob_jb_ols": float(prob_jb_seccion_28),
    "skew_ols": float(skew_seccion_28),
    "kurtosis_ols": float(kurtosis_seccion_28)
})
coeficientes_ols_seccion_28 = pd.DataFrame({"variable": resultados_ols_seccion_28.params.index, "coeficiente": resultados_ols_seccion_28.params.to_numpy(), "p_value": resultados_ols_seccion_28.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_28.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_28 = pd.DataFrame({"periodo": df_test_seccion_28["periodo"].to_numpy(), "real": y_test_seccion_28.to_numpy(), "prediccion": predicciones_test_seccion_28.to_numpy(), "residuo": y_test_seccion_28.to_numpy() - predicciones_test_seccion_28.to_numpy()})
comparacion_ols_seccion_28 = pd.DataFrame([{"modelo": "OLS sección 27", **metricas_ols_seccion_27}, {"modelo": "OLS sección 28", **metricas_ols_seccion_28}])
delta_metricas_seccion_28 = {f"delta_{metrica}": metricas_ols_seccion_28[metrica] - metricas_ols_seccion_27[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_28[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_28"):
    mlflow.set_tag("run_anterior_id", "d41a0b6a1c4d4de6ac75794edec5e4ce")
    mlflow.log_params({"seccion": 28, "dataset": str(RUTA_SALIDA_SECCION_28), "dataset_origen": str(RUTA_ORIGEN_SECCION_28), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_28), "n_predictores": X_train_seccion_28.shape[1], "filas_entrenamiento": len(df_train_seccion_28), "filas_prueba": len(df_test_seccion_28), "tipo_modelo": "statsmodels_OLS_seccion_28_desde_seccion_27"})
    mlflow.log_metrics({**metricas_ols_seccion_28, **delta_metricas_seccion_28})
    mlflow.log_table(comparacion_ols_seccion_28, artifact_file="comparacion/comparacion_ols_seccion_28.json")
    mlflow.log_table(coeficientes_ols_seccion_28, artifact_file="modelo/coeficientes_ols_seccion_28.json")
    mlflow.log_table(residuos_test_seccion_28, artifact_file="diagnostico/residuos_test_seccion_28.json")
    mlflow.log_text(resultados_ols_seccion_28.summary().as_text(), "modelo/resumen_ols_seccion_28.txt")
print(f"Dataset de la sección 28 guardado en: {RUTA_SALIDA_SECCION_28}")
print("OLS de la sección 28 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 27,37073.461485,45123.234408,0.859886,23.165613
1,OLS sección 28,37073.955260,45123.774721,0.859883,23.165970


🏃 View run ols_temporal_seccion_28 at: http://127.0.0.1:5000/#/experiments/4/runs/6a06fac38c97464cac61871e1fc0809e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 28 guardado en: data_sets_transformados\secondary_market_modelado_seccion_28.csv
OLS de la sección 28 registrado en MLflow para comparación


### Interpretación

La eliminación de `rooms` se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 27.

## 29. OLS sin tres estaciones desde la sección 27

Partimos del dataset acumulativo de la sección 27 y eliminamos `metro_station_Teatralnaya`, `metro_station_Smolenskaya (AP)` y `metro_station_Mayakovskaya`.

In [29]:
VARIABLES_ELIMINADAS_SECCION_29 = [
    "metro_station_Teatralnaya",
    "metro_station_Smolenskaya (AP)",
    "metro_station_Mayakovskaya"
]
RUTA_ORIGEN_SECCION_29 = Path("data_sets_transformados/secondary_market_modelado_seccion_27.csv")
RUTA_SALIDA_SECCION_29 = Path("data_sets_transformados/secondary_market_modelado_seccion_29.csv")
df_ols_seccion_29_origen = pd.read_csv(RUTA_ORIGEN_SECCION_29)
columnas_faltantes_seccion_29 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_29 if columna not in df_ols_seccion_29_origen.columns]
if columnas_faltantes_seccion_29:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_29}")
df_ols_seccion_29 = df_ols_seccion_29_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_29).copy()
df_ols_seccion_29.to_csv(RUTA_SALIDA_SECCION_29, index=False)
df_train_seccion_29 = df_ols_seccion_29[df_ols_seccion_29["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_29 = df_ols_seccion_29[df_ols_seccion_29["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_29 = df_train_seccion_29.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_29 = df_test_seccion_29.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_29 = df_train_seccion_29["price_per_sqm"]
y_test_seccion_29 = df_test_seccion_29["price_per_sqm"]
X_train_seccion_29_sm = sm.add_constant(X_train_seccion_29, has_constant="add")
X_test_seccion_29_sm = sm.add_constant(X_test_seccion_29, has_constant="add")
resultados_ols_seccion_29 = sm.OLS(y_train_seccion_29, X_train_seccion_29_sm).fit()
predicciones_train_seccion_29 = resultados_ols_seccion_29.predict(X_train_seccion_29_sm)
predicciones_test_seccion_29 = resultados_ols_seccion_29.predict(X_test_seccion_29_sm)
metricas_ols_seccion_29 = {
    **calcular_metricas_regresion(y_train_seccion_29, predicciones_train_seccion_29, "train"),
    **calcular_metricas_regresion(y_test_seccion_29, predicciones_test_seccion_29, "test"),
    "r2_ols": float(resultados_ols_seccion_29.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_29.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_29.aic),
    "bic_ols": float(resultados_ols_seccion_29.bic),
    "f_statistic_ols": float(resultados_ols_seccion_29.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_29.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_29.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_29.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_29.pvalues > 0.05).sum())
}
omnibus_seccion_29, prob_omnibus_seccion_29 = omni_normtest(resultados_ols_seccion_29.resid)
jarque_bera_seccion_29, prob_jb_seccion_29, skew_seccion_29, kurtosis_seccion_29 = jarque_bera(resultados_ols_seccion_29.resid)
metricas_ols_seccion_29.update({
    "omnibus_ols": float(omnibus_seccion_29),
    "prob_omnibus_ols": float(prob_omnibus_seccion_29),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_29.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_29),
    "prob_jb_ols": float(prob_jb_seccion_29),
    "skew_ols": float(skew_seccion_29),
    "kurtosis_ols": float(kurtosis_seccion_29)
})
coeficientes_ols_seccion_29 = pd.DataFrame({"variable": resultados_ols_seccion_29.params.index, "coeficiente": resultados_ols_seccion_29.params.to_numpy(), "p_value": resultados_ols_seccion_29.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_29.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_29 = pd.DataFrame({"periodo": df_test_seccion_29["periodo"].to_numpy(), "real": y_test_seccion_29.to_numpy(), "prediccion": predicciones_test_seccion_29.to_numpy(), "residuo": y_test_seccion_29.to_numpy() - predicciones_test_seccion_29.to_numpy()})
comparacion_ols_seccion_29 = pd.DataFrame([{"modelo": "OLS sección 27", **metricas_ols_seccion_27}, {"modelo": "OLS sección 29", **metricas_ols_seccion_29}])
delta_metricas_seccion_29 = {f"delta_{metrica}": metricas_ols_seccion_29[metrica] - metricas_ols_seccion_27[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_29[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_29"):
    mlflow.set_tag("run_anterior_id", "d41a0b6a1c4d4de6ac75794edec5e4ce")
    mlflow.log_params({"seccion": 29, "dataset": str(RUTA_SALIDA_SECCION_29), "dataset_origen": str(RUTA_ORIGEN_SECCION_29), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_29), "n_predictores": X_train_seccion_29.shape[1], "filas_entrenamiento": len(df_train_seccion_29), "filas_prueba": len(df_test_seccion_29), "tipo_modelo": "statsmodels_OLS_seccion_29_desde_seccion_27"})
    mlflow.log_metrics({**metricas_ols_seccion_29, **delta_metricas_seccion_29})
    mlflow.log_table(comparacion_ols_seccion_29, artifact_file="comparacion/comparacion_ols_seccion_29.json")
    mlflow.log_table(coeficientes_ols_seccion_29, artifact_file="modelo/coeficientes_ols_seccion_29.json")
    mlflow.log_table(residuos_test_seccion_29, artifact_file="diagnostico/residuos_test_seccion_29.json")
    mlflow.log_text(resultados_ols_seccion_29.summary().as_text(), "modelo/resumen_ols_seccion_29.txt")
print(f"Dataset de la sección 29 guardado en: {RUTA_SALIDA_SECCION_29}")
print("OLS de la sección 29 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 27,37073.461485,45123.234408,0.859886,23.165613
1,OLS sección 29,37072.766055,45125.261894,0.859874,23.165723


🏃 View run ols_temporal_seccion_29 at: http://127.0.0.1:5000/#/experiments/4/runs/50301279727e4940b163ddcb052f319c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 29 guardado en: data_sets_transformados\secondary_market_modelado_seccion_29.csv
OLS de la sección 29 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las tres estaciones se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad o el número de condición frente a la sección 27.

## 30. OLS sin variables `metro_station_` desde la sección 27

Partimos del dataset acumulativo de la sección 27 y eliminamos todas las columnas cuyo nombre comienza con `metro_station_`, sin considerar su p-value individual.

In [30]:
RUTA_ORIGEN_SECCION_30 = Path("data_sets_transformados/secondary_market_modelado_seccion_27.csv")
RUTA_SALIDA_SECCION_30 = Path("data_sets_transformados/secondary_market_modelado_seccion_30.csv")
df_ols_seccion_30_origen = pd.read_csv(RUTA_ORIGEN_SECCION_30)
VARIABLES_ELIMINADAS_SECCION_30 = [columna for columna in df_ols_seccion_30_origen.columns if columna.startswith("metro_station_")]
if not VARIABLES_ELIMINADAS_SECCION_30:
    raise ValueError("No se encontraron variables metro_station_ para eliminar")
df_ols_seccion_30 = df_ols_seccion_30_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_30).copy()
df_ols_seccion_30.to_csv(RUTA_SALIDA_SECCION_30, index=False)
df_train_seccion_30 = df_ols_seccion_30[df_ols_seccion_30["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_30 = df_ols_seccion_30[df_ols_seccion_30["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_30 = df_train_seccion_30.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_30 = df_test_seccion_30.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_30 = df_train_seccion_30["price_per_sqm"]
y_test_seccion_30 = df_test_seccion_30["price_per_sqm"]
X_train_seccion_30_sm = sm.add_constant(X_train_seccion_30, has_constant="add")
X_test_seccion_30_sm = sm.add_constant(X_test_seccion_30, has_constant="add")
resultados_ols_seccion_30 = sm.OLS(y_train_seccion_30, X_train_seccion_30_sm).fit()
predicciones_train_seccion_30 = resultados_ols_seccion_30.predict(X_train_seccion_30_sm)
predicciones_test_seccion_30 = resultados_ols_seccion_30.predict(X_test_seccion_30_sm)
metricas_ols_seccion_30 = {
    **calcular_metricas_regresion(y_train_seccion_30, predicciones_train_seccion_30, "train"),
    **calcular_metricas_regresion(y_test_seccion_30, predicciones_test_seccion_30, "test"),
    "r2_ols": float(resultados_ols_seccion_30.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_30.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_30.aic),
    "bic_ols": float(resultados_ols_seccion_30.bic),
    "f_statistic_ols": float(resultados_ols_seccion_30.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_30.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_30.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_30.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_30.pvalues > 0.05).sum())
}
omnibus_seccion_30, prob_omnibus_seccion_30 = omni_normtest(resultados_ols_seccion_30.resid)
jarque_bera_seccion_30, prob_jb_seccion_30, skew_seccion_30, kurtosis_seccion_30 = jarque_bera(resultados_ols_seccion_30.resid)
metricas_ols_seccion_30.update({
    "omnibus_ols": float(omnibus_seccion_30),
    "prob_omnibus_ols": float(prob_omnibus_seccion_30),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_30.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_30),
    "prob_jb_ols": float(prob_jb_seccion_30),
    "skew_ols": float(skew_seccion_30),
    "kurtosis_ols": float(kurtosis_seccion_30)
})
coeficientes_ols_seccion_30 = pd.DataFrame({"variable": resultados_ols_seccion_30.params.index, "coeficiente": resultados_ols_seccion_30.params.to_numpy(), "p_value": resultados_ols_seccion_30.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_30.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_30 = pd.DataFrame({"periodo": df_test_seccion_30["periodo"].to_numpy(), "real": y_test_seccion_30.to_numpy(), "prediccion": predicciones_test_seccion_30.to_numpy(), "residuo": y_test_seccion_30.to_numpy() - predicciones_test_seccion_30.to_numpy()})
comparacion_ols_seccion_30 = pd.DataFrame([{"modelo": "OLS sección 27", **metricas_ols_seccion_27}, {"modelo": "OLS sección 30", **metricas_ols_seccion_30}])
delta_metricas_seccion_30 = {f"delta_{metrica}": metricas_ols_seccion_30[metrica] - metricas_ols_seccion_27[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_30[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_30"):
    mlflow.set_tag("run_anterior_id", "d41a0b6a1c4d4de6ac75794edec5e4ce")
    mlflow.log_params({"seccion": 30, "dataset": str(RUTA_SALIDA_SECCION_30), "dataset_origen": str(RUTA_ORIGEN_SECCION_30), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_30), "n_predictores": X_train_seccion_30.shape[1], "filas_entrenamiento": len(df_train_seccion_30), "filas_prueba": len(df_test_seccion_30), "tipo_modelo": "statsmodels_OLS_seccion_30_sin_metro_station"})
    mlflow.log_metrics({**metricas_ols_seccion_30, **delta_metricas_seccion_30})
    mlflow.log_table(comparacion_ols_seccion_30, artifact_file="comparacion/comparacion_ols_seccion_30.json")
    mlflow.log_table(coeficientes_ols_seccion_30, artifact_file="modelo/coeficientes_ols_seccion_30.json")
    mlflow.log_table(residuos_test_seccion_30, artifact_file="diagnostico/residuos_test_seccion_30.json")
    mlflow.log_text(resultados_ols_seccion_30.summary().as_text(), "modelo/resumen_ols_seccion_30.txt")
print(f"Dataset de la sección 30 guardado en: {RUTA_SALIDA_SECCION_30}")
print(f"Variables metro_station_ eliminadas: {len(VARIABLES_ELIMINADAS_SECCION_30)}")
print("OLS de la sección 30 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 27,37073.461485,45123.234408,0.859886,23.165613
1,OLS sección 30,37146.928901,45256.240708,0.859059,23.144575


🏃 View run ols_temporal_seccion_30 at: http://127.0.0.1:5000/#/experiments/4/runs/4c6ec29964ed4fca9cf90c94f975d952
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 30 guardado en: data_sets_transformados\secondary_market_modelado_seccion_30.csv
Variables metro_station_ eliminadas: 84
OLS de la sección 30 registrado en MLflow para comparación


### Interpretación

La eliminación del grupo completo `metro_station_` se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad frente a la sección 27.

## 31. OLS sin cinco variables de distrito desde la sección 30

Partimos del dataset acumulativo de la sección 30 y eliminamos `district_Konkovo`, `district_Mozhaysky`, `district_Alekseevsky`, `district_Rogovskoye` y `district_Ostankinsky`.

In [2]:
VARIABLES_ELIMINADAS_SECCION_31 = [
    "district_Konkovo",
    "district_Mozhaysky",
    "district_Alekseevsky",
    "district_Rogovskoye",
    "district_Ostankinsky"
]
RUTA_ORIGEN_SECCION_31 = Path("data_sets_transformados/secondary_market_modelado_seccion_30.csv")
RUTA_SALIDA_SECCION_31 = Path("data_sets_transformados/secondary_market_modelado_seccion_31.csv")
df_ols_seccion_31_origen = pd.read_csv(RUTA_ORIGEN_SECCION_31)
columnas_faltantes_seccion_31 = [columna for columna in VARIABLES_ELIMINADAS_SECCION_31 if columna not in df_ols_seccion_31_origen.columns]
if columnas_faltantes_seccion_31:
    raise KeyError(f"Columnas no disponibles: {columnas_faltantes_seccion_31}")
df_ols_seccion_31 = df_ols_seccion_31_origen.drop(columns=VARIABLES_ELIMINADAS_SECCION_31).copy()
df_ols_seccion_31.to_csv(RUTA_SALIDA_SECCION_31, index=False)
df_train_seccion_31 = df_ols_seccion_31[df_ols_seccion_31["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_31 = df_ols_seccion_31[df_ols_seccion_31["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_31 = df_train_seccion_31.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_31 = df_test_seccion_31.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_31 = df_train_seccion_31["price_per_sqm"]
y_test_seccion_31 = df_test_seccion_31["price_per_sqm"]
X_train_seccion_31_sm = sm.add_constant(X_train_seccion_31, has_constant="add")
X_test_seccion_31_sm = sm.add_constant(X_test_seccion_31, has_constant="add")
resultados_ols_seccion_31 = sm.OLS(y_train_seccion_31, X_train_seccion_31_sm).fit()
predicciones_train_seccion_31 = resultados_ols_seccion_31.predict(X_train_seccion_31_sm)
predicciones_test_seccion_31 = resultados_ols_seccion_31.predict(X_test_seccion_31_sm)
metricas_ols_seccion_31 = {
    **calcular_metricas_regresion(y_train_seccion_31, predicciones_train_seccion_31, "train"),
    **calcular_metricas_regresion(y_test_seccion_31, predicciones_test_seccion_31, "test"),
    "r2_ols": float(resultados_ols_seccion_31.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_31.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_31.aic),
    "bic_ols": float(resultados_ols_seccion_31.bic),
    "f_statistic_ols": float(resultados_ols_seccion_31.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_31.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_31.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_31.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_31.pvalues > 0.05).sum())
}
omnibus_seccion_31, prob_omnibus_seccion_31 = omni_normtest(resultados_ols_seccion_31.resid)
jarque_bera_seccion_31, prob_jb_seccion_31, skew_seccion_31, kurtosis_seccion_31 = jarque_bera(resultados_ols_seccion_31.resid)
metricas_ols_seccion_31.update({
    "omnibus_ols": float(omnibus_seccion_31),
    "prob_omnibus_ols": float(prob_omnibus_seccion_31),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_31.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_31),
    "prob_jb_ols": float(prob_jb_seccion_31),
    "skew_ols": float(skew_seccion_31),
    "kurtosis_ols": float(kurtosis_seccion_31)
})
coeficientes_ols_seccion_31 = pd.DataFrame({"variable": resultados_ols_seccion_31.params.index, "coeficiente": resultados_ols_seccion_31.params.to_numpy(), "p_value": resultados_ols_seccion_31.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_31.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_31 = pd.DataFrame({"periodo": df_test_seccion_31["periodo"].to_numpy(), "real": y_test_seccion_31.to_numpy(), "prediccion": predicciones_test_seccion_31.to_numpy(), "residuo": y_test_seccion_31.to_numpy() - predicciones_test_seccion_31.to_numpy()})
comparacion_ols_seccion_31 = pd.DataFrame([{"modelo": "OLS sección 30", **metricas_ols_seccion_30}, {"modelo": "OLS sección 31", **metricas_ols_seccion_31}])
delta_metricas_seccion_31 = {f"delta_{metrica}": metricas_ols_seccion_31[metrica] - metricas_ols_seccion_30[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_31[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_31"):
    mlflow.set_tag("run_anterior_id", "4c6ec29964ed4fca9cf90c94f975d952")
    mlflow.log_params({"seccion": 31, "dataset": str(RUTA_SALIDA_SECCION_31), "dataset_origen": str(RUTA_ORIGEN_SECCION_31), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "variables_eliminadas": ",".join(VARIABLES_ELIMINADAS_SECCION_31), "n_predictores": X_train_seccion_31.shape[1], "filas_entrenamiento": len(df_train_seccion_31), "filas_prueba": len(df_test_seccion_31), "tipo_modelo": "statsmodels_OLS_seccion_31_desde_seccion_30"})
    mlflow.log_metrics({**metricas_ols_seccion_31, **delta_metricas_seccion_31})
    mlflow.log_table(comparacion_ols_seccion_31, artifact_file="comparacion/comparacion_ols_seccion_31.json")
    mlflow.log_table(coeficientes_ols_seccion_31, artifact_file="modelo/coeficientes_ols_seccion_31.json")
    mlflow.log_table(residuos_test_seccion_31, artifact_file="diagnostico/residuos_test_seccion_31.json")
    mlflow.log_text(resultados_ols_seccion_31.summary().as_text(), "modelo/resumen_ols_seccion_31.txt")
print(f"Dataset de la sección 31 guardado en: {RUTA_SALIDA_SECCION_31}")
print("OLS de la sección 31 registrado en MLflow para comparación")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS sección 30,37146.928901,45256.240708,0.859059,23.144575
1,OLS sección 31,37149.437615,45256.355993,0.859058,23.147238


🏃 View run ols_temporal_seccion_31 at: http://127.0.0.1:5000/#/experiments/4/runs/c23f82b46c20461798877412394fe438
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset de la sección 31 guardado en: data_sets_transformados\secondary_market_modelado_seccion_31.csv
OLS de la sección 31 registrado en MLflow para comparación


### Interpretación

La eliminación acumulativa de las cinco variables de distrito se acepta solo si conserva o mejora las métricas temporales y reduce la complejidad frente a la sección 30.

## 32. Mercado estandar con corte de 402600 RUB/m2

Partimos del dataset completo de la seccion 27. El corte de 402600 RUB/m2 fue calculado sobre el conjunto de entrenamiento con la regla Q3 + 1,5 IQR. Los registros superiores se conservan en un dataset premium separado.

In [1]:
CORTE_PRECIO_ESTANDAR_SECCION_32 = 402600
RUTA_ORIGEN_SECCION_32 = Path("data_sets_transformados/secondary_market_modelado_seccion_27.csv")
RUTA_SALIDA_SECCION_32 = Path("data_sets_transformados/secondary_market_modelado_seccion_32.csv")
RUTA_SALIDA_PREMIUM_SECCION_32 = Path("data_sets_transformados/secondary_market_modelado_seccion_32_premium.csv")
df_ols_seccion_32_origen = pd.read_csv(RUTA_ORIGEN_SECCION_32)
if df_ols_seccion_32_origen["price_per_sqm"].isna().any() or (df_ols_seccion_32_origen["price_per_sqm"] <= 0).any():
    raise ValueError("price_per_sqm contiene valores faltantes, cero o negativos")
df_ols_seccion_32 = df_ols_seccion_32_origen[df_ols_seccion_32_origen["price_per_sqm"] <= CORTE_PRECIO_ESTANDAR_SECCION_32].copy()
df_premium_seccion_32 = df_ols_seccion_32_origen[df_ols_seccion_32_origen["price_per_sqm"] > CORTE_PRECIO_ESTANDAR_SECCION_32].copy()
df_ols_seccion_32.to_csv(RUTA_SALIDA_SECCION_32, index=False)
df_premium_seccion_32.to_csv(RUTA_SALIDA_PREMIUM_SECCION_32, index=False)
df_train_seccion_32 = df_ols_seccion_32[df_ols_seccion_32["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_32 = df_ols_seccion_32[df_ols_seccion_32["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_32 = df_train_seccion_32.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_32 = df_test_seccion_32.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_32 = df_train_seccion_32["price_per_sqm"]
y_test_seccion_32 = df_test_seccion_32["price_per_sqm"]
if df_train_seccion_32.empty or df_test_seccion_32.empty:
    raise ValueError("El corte temporal no produjo train y test para el mercado estandar")
X_train_seccion_32_sm = sm.add_constant(X_train_seccion_32, has_constant="add")
X_test_seccion_32_sm = sm.add_constant(X_test_seccion_32, has_constant="add")
resultados_ols_seccion_32 = sm.OLS(y_train_seccion_32, X_train_seccion_32_sm).fit()
predicciones_train_seccion_32 = resultados_ols_seccion_32.predict(X_train_seccion_32_sm)
predicciones_test_seccion_32 = resultados_ols_seccion_32.predict(X_test_seccion_32_sm)
metricas_ols_seccion_32 = {
    **calcular_metricas_regresion(y_train_seccion_32, predicciones_train_seccion_32, "train"),
    **calcular_metricas_regresion(y_test_seccion_32, predicciones_test_seccion_32, "test"),
    "r2_ols": float(resultados_ols_seccion_32.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_32.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_32.aic),
    "bic_ols": float(resultados_ols_seccion_32.bic),
    "f_statistic_ols": float(resultados_ols_seccion_32.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_32.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_32.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_32.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_32.pvalues > 0.05).sum()),
    "filas_mercado_estandar": float(len(df_ols_seccion_32)),
    "filas_mercado_premium": float(len(df_premium_seccion_32)),
    "proporcion_mercado_estandar": float(len(df_ols_seccion_32) / len(df_ols_seccion_32_origen)),
    "proporcion_mercado_premium": float(len(df_premium_seccion_32) / len(df_ols_seccion_32_origen))
}
omnibus_seccion_32, prob_omnibus_seccion_32 = omni_normtest(resultados_ols_seccion_32.resid)
jarque_bera_seccion_32, prob_jb_seccion_32, skew_seccion_32, kurtosis_seccion_32 = jarque_bera(resultados_ols_seccion_32.resid)
metricas_ols_seccion_32.update({
    "omnibus_ols": float(omnibus_seccion_32),
    "prob_omnibus_ols": float(prob_omnibus_seccion_32),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_32.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_32),
    "prob_jb_ols": float(prob_jb_seccion_32),
    "skew_ols": float(skew_seccion_32),
    "kurtosis_ols": float(kurtosis_seccion_32)
})
coeficientes_ols_seccion_32 = pd.DataFrame({"variable": resultados_ols_seccion_32.params.index, "coeficiente": resultados_ols_seccion_32.params.to_numpy(), "p_value": resultados_ols_seccion_32.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_32.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_32 = pd.DataFrame({"periodo": df_test_seccion_32["periodo"].to_numpy(), "real": y_test_seccion_32.to_numpy(), "prediccion": predicciones_test_seccion_32.to_numpy(), "residuo": y_test_seccion_32.to_numpy() - predicciones_test_seccion_32.to_numpy()})
comparacion_ols_seccion_32 = pd.DataFrame([{"modelo": "OLS seccion 27 - mercado completo", **metricas_ols_seccion_27}, {"modelo": "OLS seccion 32 - mercado estandar", **metricas_ols_seccion_32}])
delta_metricas_seccion_32 = {f"delta_{metrica}": metricas_ols_seccion_32[metrica] - metricas_ols_seccion_27[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_32[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_32_mercado_estandar"):
    mlflow.set_tag("run_anterior_id", "d41a0b6a1c4d4de6ac75794edec5e4ce")
    mlflow.log_params({"seccion": 32, "dataset": str(RUTA_SALIDA_SECCION_32), "dataset_premium": str(RUTA_SALIDA_PREMIUM_SECCION_32), "dataset_origen": str(RUTA_ORIGEN_SECCION_32), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "corte_precio_estandar_rub_m2": CORTE_PRECIO_ESTANDAR_SECCION_32, "metodo_corte": "Q3_mas_1_5_IQR_entrenamiento", "n_predictores": X_train_seccion_32.shape[1], "filas_entrenamiento": len(df_train_seccion_32), "filas_prueba": len(df_test_seccion_32), "filas_mercado_estandar": len(df_ols_seccion_32), "filas_mercado_premium": len(df_premium_seccion_32), "tipo_modelo": "statsmodels_OLS_seccion_32_mercado_estandar"})
    mlflow.log_metrics({**metricas_ols_seccion_32, **delta_metricas_seccion_32})
    mlflow.log_table(comparacion_ols_seccion_32, artifact_file="comparacion/comparacion_ols_seccion_32.json")
    mlflow.log_table(coeficientes_ols_seccion_32, artifact_file="modelo/coeficientes_ols_seccion_32.json")
    mlflow.log_table(residuos_test_seccion_32, artifact_file="diagnostico/residuos_test_seccion_32.json")
    mlflow.log_text(resultados_ols_seccion_32.summary().as_text(), "modelo/resumen_ols_seccion_32.txt")
print(f"Dataset estandar de la seccion 32 guardado en: {RUTA_SALIDA_SECCION_32}")
print(f"Dataset premium de la seccion 32 guardado en: {RUTA_SALIDA_PREMIUM_SECCION_32}")
print(f"Filas estandar: {len(df_ols_seccion_32)} | Filas premium: {len(df_premium_seccion_32)}")
print("OLS de la seccion 32 registrado en MLflow para comparacion")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS seccion 27 - mercado completo,37073.461485,45123.234408,0.859886,23.165613
1,OLS seccion 32 - mercado estandar,29086.281498,34129.077912,0.818523,20.276878


?? View run ols_temporal_seccion_32_mercado_estandar at: http://127.0.0.1:5000/#/experiments/4/runs/29ba44bd36e74b409e4c62400200402c
?? View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset estandar de la seccion 32 guardado en: data_sets_transformados\secondary_market_modelado_seccion_32.csv
Dataset premium de la seccion 32 guardado en: data_sets_transformados\secondary_market_modelado_seccion_32_premium.csv
Filas estandar: 47052 | Filas premium: 2948
OLS de la seccion 32 registrado en MLflow para comparacion


### Interpretacion

La seccion 32 representa el mercado estandar definido por `price_per_sqm <= 402600 RUB/m2`. Los registros superiores no se eliminan del proyecto: se conservan en el dataset premium para analizarlos por separado.

## 33. Mercado estandar con corte desde el dataset de la seccion 3

Partimos del dataset one-hot completo generado en la seccion 3 y aplicamos el corte de `402600 RUB/m2`. Los registros superiores se conservan en un dataset premium separado. La comparacion predictiva principal usa el mismo subconjunto estandar del periodo de prueba.

In [1]:
CORTE_PRECIO_ESTANDAR_SECCION_33 = 402600
RUTA_ORIGEN_SECCION_33 = Path("data_sets_transformados/secondary_market_modelado_one_hot.csv")
RUTA_SALIDA_SECCION_33 = Path("data_sets_transformados/secondary_market_modelado_seccion_33.csv")
RUTA_SALIDA_PREMIUM_SECCION_33 = Path("data_sets_transformados/secondary_market_modelado_seccion_33_premium.csv")
df_ols_seccion_33_origen = pd.read_csv(RUTA_ORIGEN_SECCION_33)
if df_ols_seccion_33_origen["price_per_sqm"].isna().any() or (df_ols_seccion_33_origen["price_per_sqm"] <= 0).any():
    raise ValueError("price_per_sqm contiene valores faltantes, cero o negativos")
df_ols_seccion_33 = df_ols_seccion_33_origen[df_ols_seccion_33_origen["price_per_sqm"] <= CORTE_PRECIO_ESTANDAR_SECCION_33].copy()
df_premium_seccion_33 = df_ols_seccion_33_origen[df_ols_seccion_33_origen["price_per_sqm"] > CORTE_PRECIO_ESTANDAR_SECCION_33].copy()
df_ols_seccion_33.to_csv(RUTA_SALIDA_SECCION_33, index=False)
df_premium_seccion_33.to_csv(RUTA_SALIDA_PREMIUM_SECCION_33, index=False)
df_train_seccion_33 = df_ols_seccion_33[df_ols_seccion_33["periodo"] < CORTE_TEMPORAL].copy()
df_test_seccion_33 = df_ols_seccion_33[df_ols_seccion_33["periodo"] >= CORTE_TEMPORAL].copy()
X_train_seccion_33 = df_train_seccion_33.drop(columns=["price_per_sqm", "periodo"])
X_test_seccion_33 = df_test_seccion_33.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_33 = df_train_seccion_33["price_per_sqm"]
y_test_seccion_33 = df_test_seccion_33["price_per_sqm"]
if df_train_seccion_33.empty or df_test_seccion_33.empty:
    raise ValueError("El corte temporal no produjo train y test para el mercado estandar")
X_train_seccion_33_sm = sm.add_constant(X_train_seccion_33, has_constant="add")
X_test_seccion_33_sm = sm.add_constant(X_test_seccion_33, has_constant="add")
resultados_ols_seccion_33 = sm.OLS(y_train_seccion_33, X_train_seccion_33_sm).fit()
predicciones_train_seccion_33 = resultados_ols_seccion_33.predict(X_train_seccion_33_sm)
predicciones_test_seccion_33 = resultados_ols_seccion_33.predict(X_test_seccion_33_sm)
metricas_ols_seccion_33 = {
    **calcular_metricas_regresion(y_train_seccion_33, predicciones_train_seccion_33, "train"),
    **calcular_metricas_regresion(y_test_seccion_33, predicciones_test_seccion_33, "test"),
    "r2_ols": float(resultados_ols_seccion_33.rsquared),
    "r2_ajustado_ols": float(resultados_ols_seccion_33.rsquared_adj),
    "aic_ols": float(resultados_ols_seccion_33.aic),
    "bic_ols": float(resultados_ols_seccion_33.bic),
    "f_statistic_ols": float(resultados_ols_seccion_33.fvalue),
    "prob_f_statistic_ols": float(resultados_ols_seccion_33.f_pvalue),
    "condition_number_ols": float(resultados_ols_seccion_33.condition_number),
    "coeficientes_significativos_005": float((resultados_ols_seccion_33.pvalues <= 0.05).sum()),
    "coeficientes_no_significativos_005": float((resultados_ols_seccion_33.pvalues > 0.05).sum()),
    "filas_mercado_estandar": float(len(df_ols_seccion_33)),
    "filas_mercado_premium": float(len(df_premium_seccion_33)),
    "proporcion_mercado_estandar": float(len(df_ols_seccion_33) / len(df_ols_seccion_33_origen)),
    "proporcion_mercado_premium": float(len(df_premium_seccion_33) / len(df_ols_seccion_33_origen))
}
omnibus_seccion_33, prob_omnibus_seccion_33 = omni_normtest(resultados_ols_seccion_33.resid)
jarque_bera_seccion_33, prob_jb_seccion_33, skew_seccion_33, kurtosis_seccion_33 = jarque_bera(resultados_ols_seccion_33.resid)
metricas_ols_seccion_33.update({
    "omnibus_ols": float(omnibus_seccion_33),
    "prob_omnibus_ols": float(prob_omnibus_seccion_33),
    "durbin_watson_ols": float(durbin_watson(resultados_ols_seccion_33.resid)),
    "jarque_bera_ols": float(jarque_bera_seccion_33),
    "prob_jb_ols": float(prob_jb_seccion_33),
    "skew_ols": float(skew_seccion_33),
    "kurtosis_ols": float(kurtosis_seccion_33)
})
coeficientes_ols_seccion_33 = pd.DataFrame({"variable": resultados_ols_seccion_33.params.index, "coeficiente": resultados_ols_seccion_33.params.to_numpy(), "p_value": resultados_ols_seccion_33.pvalues.to_numpy(), "significativa_0_05": resultados_ols_seccion_33.pvalues.to_numpy() <= 0.05})
residuos_test_seccion_33 = pd.DataFrame({"periodo": df_test_seccion_33["periodo"].to_numpy(), "real": y_test_seccion_33.to_numpy(), "prediccion": predicciones_test_seccion_33.to_numpy(), "residuo": y_test_seccion_33.to_numpy() - predicciones_test_seccion_33.to_numpy()})
# Baseline section 3 evaluated on the same standard test subset for a fair comparison.
df_train_seccion_3 = df_ols_seccion_33_origen[df_ols_seccion_33_origen["periodo"] < CORTE_TEMPORAL].copy()
X_train_seccion_3 = df_train_seccion_3.drop(columns=["price_per_sqm", "periodo"])
y_train_seccion_3 = df_train_seccion_3["price_per_sqm"]
resultados_ols_seccion_3_reajustado = sm.OLS(y_train_seccion_3, sm.add_constant(X_train_seccion_3, has_constant="add")).fit()
predicciones_test_seccion_3_sobre_estandar = resultados_ols_seccion_3_reajustado.predict(X_test_seccion_33_sm)
metricas_ols_seccion_3_sobre_estandar = calcular_metricas_regresion(y_test_seccion_33, predicciones_test_seccion_3_sobre_estandar, "test")
metricas_ols_seccion_3_sobre_estandar_mlflow = {f"seccion_3_{clave}": valor for clave, valor in metricas_ols_seccion_3_sobre_estandar.items()}
comparacion_ols_seccion_33 = pd.DataFrame([{"modelo": "OLS seccion 3 - test estandar", **metricas_ols_seccion_3_sobre_estandar}, {"modelo": "OLS seccion 33 - mercado estandar", **metricas_ols_seccion_33}])
delta_metricas_seccion_33 = {f"delta_{metrica}": metricas_ols_seccion_33[metrica] - metricas_ols_seccion_3_sobre_estandar[metrica] for metrica in ["mae_test", "rmse_test", "r2_test", "mape_test"]}
display(comparacion_ols_seccion_33[["modelo", "mae_test", "rmse_test", "r2_test", "mape_test"]])
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Inmobiliario - OLS Temporal")
with mlflow.start_run(run_name="ols_temporal_seccion_33_mercado_estandar_desde_seccion_3"):
    mlflow.set_tag("run_anterior_id", "8632cd73b7b84347bd6cd3ee0b2cfe4e")
    mlflow.log_params({"seccion": 33, "dataset": str(RUTA_SALIDA_SECCION_33), "dataset_premium": str(RUTA_SALIDA_PREMIUM_SECCION_33), "dataset_origen": str(RUTA_ORIGEN_SECCION_33), "objetivo": "price_per_sqm", "corte_temporal": CORTE_TEMPORAL, "corte_precio_estandar_rub_m2": CORTE_PRECIO_ESTANDAR_SECCION_33, "n_predictores": X_train_seccion_33.shape[1], "filas_entrenamiento": len(df_train_seccion_33), "filas_prueba": len(df_test_seccion_33), "filas_mercado_estandar": len(df_ols_seccion_33), "filas_mercado_premium": len(df_premium_seccion_33), "tipo_modelo": "statsmodels_OLS_seccion_33_mercado_estandar_desde_seccion_3"})
    mlflow.log_metrics({**metricas_ols_seccion_33, **metricas_ols_seccion_3_sobre_estandar_mlflow, **delta_metricas_seccion_33})
    mlflow.log_table(comparacion_ols_seccion_33, artifact_file="comparacion/comparacion_ols_seccion_33.json")
    mlflow.log_table(coeficientes_ols_seccion_33, artifact_file="modelo/coeficientes_ols_seccion_33.json")
    mlflow.log_table(residuos_test_seccion_33, artifact_file="diagnostico/residuos_test_seccion_33.json")
    mlflow.log_text(resultados_ols_seccion_33.summary().as_text(), "modelo/resumen_ols_seccion_33.txt")
print(f"Dataset estandar de la seccion 33 guardado en: {RUTA_SALIDA_SECCION_33}")
print(f"Dataset premium de la seccion 33 guardado en: {RUTA_SALIDA_PREMIUM_SECCION_33}")
print(f"Filas estandar: {len(df_ols_seccion_33)} | Filas premium: {len(df_premium_seccion_33)}")
print("OLS de la seccion 33 registrado en MLflow para comparacion")

,modelo,mae_test,rmse_test,r2_test,mape_test
0,OLS seccion 3 - test estandar,35093.255771,41121.289063,0.736545,24.164340
1,OLS seccion 33 - mercado estandar,29090.763965,34121.403972,0.818604,20.273794


                              modelo      mae_test  ...   r2_test  mape_test
0      OLS seccion 3 - test estandar  35093.255771  ...  0.736545  24.164340
1  OLS seccion 33 - mercado estandar  29090.763965  ...  0.818604  20.273794

[2 rows x 5 columns]
🏃 View run ols_temporal_seccion_33_mercado_estandar_desde_seccion_3 at: http://127.0.0.1:5000/#/experiments/4/runs/36fb53d6a8024b7b9701c9f55e28e514
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Dataset estandar de la seccion 33 guardado en: data_sets_transformados\secondary_market_modelado_seccion_33.csv
Dataset premium de la seccion 33 guardado en: data_sets_transformados\secondary_market_modelado_seccion_33_premium.csv
Filas estandar: 47052 | Filas premium: 2948
OLS de la seccion 33 registrado en MLflow para comparacion
                                     variable  p_value
0             metro_station_Elektrozavodskaya  0.99867
1                 metro_station_Profsoyuznaya  0.97709
2                 metro_station_Molodyoz

### Interpretacion

La seccion 33 aplica el mismo corte de `402600 RUB/m2` sobre el dataset one-hot completo de la seccion 3. Su comparacion principal contra la seccion 3 usa exclusivamente el subconjunto estandar del periodo de prueba.